<a href="https://colab.research.google.com/github/ThanhNgo1007/DRMD_LAMDA_DATASET/blob/main/notebook_drmd_lamda_v13_bhr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DRMD trên LAMDA — V13 đồng bộ Static/IRAL/IRAAL/FN/BHR/FFCR/CFJR

Notebook này tạo đủ bảy phương pháp trên cùng mười hạt giống: **MLP tĩnh**,
**DRMD (IRAL)**, **DRMD (IRAAL)**, **DRMD-FN**, **BHR-only**,
**DRMD-FN–BHR** và **FFCR**. IRAL dùng phản hồi nội sinh \(Q_t=R_t\); bốn
nhánh ngân sách thấp dùng cùng split thời gian, B=15 và chi phí từ chối −0,1.
Static và FFCR là hai mốc đồng bộ, nhưng không được xem là đối chứng cùng
ngân sách với các nhánh B=15; IRAL cũng không có ngân sách cố định.

Nhãn chu kỳ t được mở sau khi dự đoán đã khóa và chỉ được fit từ t+1. Đây là
giao thức V13 độc lập; không dùng checkpoint hoặc số liệu V12 để thay cho bất
kỳ lượt V13 nào. Notebook tự nhúng và kiểm SHA-256 của mô-đun BHR,
không yêu cầu Dataset Kaggle đã có package ``experiments``. Pha chính có
**70 lượt**. Hai pha độ nhạy bổ sung 100 lượt ngân sách và 40 lượt chi phí;
cổng báo cáo chính chỉ mở khi đủ **210/210 lượt**, mỗi lượt có đúng **110
chu kỳ** và **885.947 mẫu kiểm thử** theo nguồn LAMDA đã khóa.

Pha CFJR là đối chứng bổ sung độc lập. **MLP-CFJR** khởi tạo lại đúng kiến
trúc MLP đã khóa ở từng tháng, huấn luyện trên tập ban đầu cộng toàn bộ nhãn
đã mở đến tháng trước, rồi mới dự đoán tháng hiện tại. Mười hạt giống và 110
tháng tạo thành 1.100 lần tái huấn luyện; vì vậy pha này có checkpoint theo
chu kỳ, không chặn cổng 210 lượt chính và không được gọi là cận trên lý thuyết.

### Chạy trên Google Colab với dữ liệu Kaggle

Ô cấu hình đầu tiên tự nhận dạng Colab, đọc secret ``KAGGLE_API_TOKEN`` và
tải hai Kaggle Dataset bắt buộc vào cấu trúc tương thích ``/kaggle/input``.
Không ghi token trực tiếp vào notebook. Nếu tiếp tục từ checkpoint đã xuất
thành Kaggle Dataset, đặt ``COLAB_CHECKPOINT_DATASET`` thành ``owner/slug``.
Trên Kaggle, toàn bộ bootstrap Colab được bỏ qua và các input đã gắn vẫn được
dùng như trước. Nếu Google Drive đã được mount, ô đóng gói cuối tự sao lưu các
ZIP với tên có pha và thời điểm UTC để không ghi đè bản trước.


## Ô 1 — Cấu hình giao thức V13 và ma trận BHR khóa trước


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

# ── Bootstrap Google Colab, không chạy trên Kaggle/local ────────────────────
# Tạo secret KAGGLE_API_TOKEN trong Colab (biểu tượng chìa khóa), không dán
# token vào mã nguồn. Dataset checkpoint là tùy chọn và phải cùng giao thức V13.
COLAB_REQUIRED_KAGGLE_DATASETS = [
    {
        "handle": "thanhngo1007/drmd-lamda-dataset",
        "target": "drmd-lamda-dataset",
        "required_directory": "DRMD_LAMDA_DATASET",
    },
    {
        "handle": "thanhngo1007/lamda-full-processed",
        "target": "lamda-full-processed",
        "required_directory": "Baseline",
    },
]
COLAB_CHECKPOINT_DATASET = ""  # Tùy chọn: "owner/checkpoint-dataset-slug"
COLAB_BACKUP_TO_MOUNTED_DRIVE = True
COLAB_DRIVE_BACKUP_DIR = "/content/drive/MyDrive/DRMD_checkpoints"
COLAB_INPUT_ROOT = Path("/kaggle/input")
COLAB_WORK_ROOT = Path("/kaggle/working")


def _detect_google_colab():
    try:
        from google.colab import userdata as colab_userdata
    except ModuleNotFoundError:
        return False, None
    return True, colab_userdata


IS_GOOGLE_COLAB, _COLAB_USERDATA = _detect_google_colab()


def _read_colab_secret(name):
    if not IS_GOOGLE_COLAB or _COLAB_USERDATA is None:
        return None
    try:
        value = _COLAB_USERDATA.get(name)
    except Exception:
        return None
    value = str(value).strip() if value is not None else ""
    return value or None


def _import_kagglehub_for_colab():
    try:
        import kagglehub
    except ModuleNotFoundError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--quiet", "kagglehub",
        ])
        import kagglehub
    return kagglehub


def _find_required_directory(root, directory_name):
    if not root.is_dir():
        return None
    if root.name == directory_name:
        return root
    return next((path for path in root.rglob(directory_name) if path.is_dir()), None)


def _download_colab_dataset(kagglehub, handle, target_name,
                            required_directory=None):
    target = COLAB_INPUT_ROOT / target_name
    existing = (_find_required_directory(target, required_directory)
                if required_directory else None)
    if existing is not None:
        print({"colab_dataset": handle, "status": "already_present",
               "path": str(existing)})
        return target

    try:
        downloaded = kagglehub.dataset_download(handle, output_dir=str(target))
    except Exception as exc:
        raise RuntimeError(
            f"Không tải được Kaggle Dataset {handle!r}. Hãy tạo Colab secret "
            "KAGGLE_API_TOKEN, bật quyền truy cập notebook và kiểm tra quyền "
            "truy cập Dataset."
        ) from exc

    if required_directory:
        marker = _find_required_directory(target, required_directory)
        if marker is None:
            raise RuntimeError(
                f"Dataset {handle!r} đã tải nhưng thiếu thư mục "
                f"{required_directory!r} trong {target}."
            )
    else:
        marker = Path(downloaded)
    print({"colab_dataset": handle, "status": "ready", "path": str(marker)})
    return target


def _prepare_colab_kaggle_inputs():
    if not IS_GOOGLE_COLAB:
        return
    COLAB_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
    COLAB_WORK_ROOT.mkdir(parents=True, exist_ok=True)

    token = os.environ.get("KAGGLE_API_TOKEN") or _read_colab_secret(
        "KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token

    kagglehub = _import_kagglehub_for_colab()
    for dataset in COLAB_REQUIRED_KAGGLE_DATASETS:
        _download_colab_dataset(
            kagglehub,
            dataset["handle"],
            dataset["target"],
            dataset["required_directory"],
        )

    checkpoint_handle = (
        str(COLAB_CHECKPOINT_DATASET).strip()
        or (_read_colab_secret("KAGGLE_CHECKPOINT_DATASET") or "")
    )
    if checkpoint_handle:
        _download_colab_dataset(
            kagglehub,
            checkpoint_handle,
            "v13-checkpoint",
            required_directory=None,
        )
    print({"runtime": "google_colab", "kaggle_input": str(COLAB_INPUT_ROOT),
           "optional_checkpoint_dataset": bool(checkpoint_handle)})


_prepare_colab_kaggle_inputs()

# ── Nhận dạng giao thức V13 ──────────────────────────────────────────────────
BASE_EXPERIMENT_NAME = "lamda_2013_2025_drmd_iral_iraal_fn_bhr_sensitivity_v13"
PROTOCOL_VERSION = "drmd_lamda_iral_iraal_fn_bhr_sensitivity_v13_2026-08-11"
STRATEGY_RUN_TAG = "aligned_drmd_strategies_v13"
DATASET_YEARS = [2013, 2014, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
LAMDA_HF_REPO_ID = "IQSeC-Lab/LAMDA"
LAMDA_HF_REVISION = "ad9614bdd5556767f97ced2fce797c2f06408ebf"
EXPECTED_LAMDA_FILE_SHA256 = {
    "2013/2013_test.parquet": "208528e70c2c71c3765c4749d0293cd2816ca316dce60ef6eef42899d5aca323",
    "2013/2013_train.parquet": "0b83f9a317fa1388406f21837de422cb587cd5ab0ec401dec63f78888b847989",
    "2014/2014_test.parquet": "b5967aa77a1d25e9a3671fa38e266cdddd34e5436755cbaf48463ede5b4bb5bc",
    "2014/2014_train.parquet": "1f65b6053a174eb34a050c26162b294822d3b30e1cf83deae594f26d0b9d9a6f",
    "2016/2016_test.parquet": "daca37c4c706d60497b2cb25bf4cec0d35a5c73dabdfbe747d6f2bf36aa3fc3f",
    "2016/2016_train.parquet": "8f0871a41515e1cbb45cc7cc10d6d489629efe80fc6499c8187c0c148a6155cd",
    "2017/2017_test.parquet": "56a35dccadc8b51eacba89b279ada9be3e7620477d159da4068c0db3efeafa38",
    "2017/2017_train.parquet": "435e5bd1be3be3f36b332d3eb89ecfc5250c6c618f1fb1085f2f0842242b3c4c",
    "2018/2018_test.parquet": "70618444fa642388cdbb7a0747e5b871100c73ec3289a9e78329aa2041a9bcb2",
    "2018/2018_train.parquet": "ad01e1224ef3a459a29309c8ad5eceaabd6a7af8b262df0a438f9aeeae1671dd",
    "2019/2019_test.parquet": "400a46fe12a3d8110ab4a1073545f023ba148495bfe9c64220fe3dceee71653c",
    "2019/2019_train.parquet": "d7ceb62a6730f02652982549a2ed4eac8160351a19b628fba1302fe97ed7d5eb",
    "2020/2020_test.parquet": "6d9fe6fe5232c281b08aa34d25d23eb44fbf2a14597ba6eb7d28376db8aaae7b",
    "2020/2020_train.parquet": "19d5a25c2304d8af929167724f8785632fe8fe7249266a099603146ca11a7b80",
    "2021/2021_test.parquet": "10b2bf2a96be746a80a22d59a7a95c9e4112079d29c92c67ce3c62f556458a7a",
    "2021/2021_train.parquet": "b1642ef38f7669951403367bf56c745c535a9250aba4bb25a2502e123c1c30c1",
    "2022/2022_test.parquet": "4f74131f710c0dff2daf622183df0ef72b5be12ee01418897b3e4a4729313691",
    "2022/2022_train.parquet": "f41b5c4f051dd5dced9adc322b221ac4fab5fa16202b07c168a80ca3d3a1662b",
    "2023/2023_test.parquet": "58f4f6356d1f88fcaeb5fa04c67a1260b4cbe736c18961dc3e51b6cd121598c2",
    "2023/2023_train.parquet": "b1dfb0962484943537d1a02445f69454408c6e8a82910e9ef17f418f6dfc6af0",
    "2024/2024_test.parquet": "e360ff7a9a0b3f1f8c61b17c0aa5de25ce7cd3411e5bb3e2016fdf724c18f2b9",
    "2024/2024_train.parquet": "8cdae48a4765c9af4aae760e92f0d14dc4c2fa89a4164297a110ad0f450f78ff",
    "2025/2025_test.parquet": "702e8131a328dee30c7c1ef4373cefafc39ccc0ba8c5282257836876c783a3c6",
    "2025/2025_train.parquet": "cf65587318bd2820081fba89f4d1f648a33b44519dbbb88b392ba448ff498846",
}
EXPECTED_LAMDA_SAMPLES = 1_008_381
EXPECTED_LAMDA_FEATURES = 4_561
EXPECTED_INITIAL_TRAIN_SAMPLES = 122_434
EXPECTED_TEST_PERIODS = 110
EXPECTED_TEST_SAMPLES = 885_947
TRAINING_WINDOW_MODE = "calendar_12"
TRAINING_WINDOW = 12 if TRAINING_WINDOW_MODE == "calendar_12" else 14
TESTING_WINDOW, GRANULARITY = 1, "month"
EXPERIMENT_NAME = (BASE_EXPERIMENT_NAME if TRAINING_WINDOW_MODE == "calendar_12"
                   else BASE_EXPERIMENT_NAME + "_datawin12")

WORK = COLAB_WORK_ROOT
OUT = WORK / "results" / EXPERIMENT_NAME
DATA_OUT = WORK / "DRMD_datasets" / EXPERIMENT_NAME
RAW_OUT, TABLE_OUT, FIG_OUT = OUT / "raw", OUT / "tables", OUT / "figures"
STRATEGY_RAW_OUT = RAW_OUT / STRATEGY_RUN_TAG
STAGING_OUT, QUARANTINE_OUT = OUT / ".staging", OUT / "quarantine"
for path in (OUT, DATA_OUT, RAW_OUT, TABLE_OUT, FIG_OUT, STRATEGY_RAW_OUT,
             STAGING_OUT, QUARANTINE_OUT):
    path.mkdir(parents=True, exist_ok=True)

# ── Điểm vận hành và BHR khóa trước khi chạy ─────────────────────────────────
SEEDS = [0, 1, 7, 13, 26, 42, 73, 2026, 314159, 281083886]
MP_FIXED = 1.0
DRMD_TRAINING_EPOCHS = 5
AL_SELECTION_MODE = "Probs"
STRATEGY_FINETUNING_SIZE = 5000
STRATEGY_REJECT_COST = -0.1
RECENCY_POLICY = "recent_calendar_months_seeded_boundary_v2"
FEEDBACK_UPDATE_BUDGET = 15
SENSITIVITY_BUDGETS = [5, 15, 50, 100, 200, 400]
SENSITIVITY_REJECT_COSTS = [0.0, -0.1, -1.0]
CFJR_METHOD = "MLP-CFJR"
CFJR_PROTOCOL_VERSION = "mlp_cfjr_full_history_causal_v1_2026-08-11"
CFJR_REFIT_LAUNCH_CUTOFF_HOURS = 9.0
BHR_FN_FRACTION = 0.35
BHR_FP_FRACTION = 0.25
BHR_BACKGROUND_FRACTION = 0.40
BHR_FN_MAX_REPEAT = 4
BHR_FP_MAX_REPEAT = 2

IRAL_POLICY = {"name": "IRAL", "scope": "rejected_feedback_endogenous"}
IRAAL_POLICY = {"name": "IRAAL", "scope": "budgeted_update",
                "budget": FEEDBACK_UPDATE_BUDGET}
FFCR_POLICY = {"name": "FFCR", "scope": "full_feedback_reference"}

def strategy_variant(name, audit_sampling=False, adaptive_fn_penalty=False,
                     balanced_hard_replay=False, full_feedback=False,
                     feedback_budget=FEEDBACK_UPDATE_BUDGET,
                     selector_mode="uncertainty", policy=None,
                     reject_cost=None):
    if policy is None:
        policy = {"name": "IRAAL", "scope": "budgeted_update",
                  "budget": int(feedback_budget)}
    return {
        "name": name, "model": "PPO_Bandit",
        "minority_priority": MP_FIXED, "majority_priority": 1.0,
        "temporal_rewards": True, "temporal_scaling": 6.0,
        "training_epochs": DRMD_TRAINING_EPOCHS, "fn_penalty": 1.0,
        "selector_mode": str(selector_mode),
        "feedback_budget": int(feedback_budget),
        "candidate_multiplier": 1,
        "full_feedback": bool(full_feedback),
        "audit_sampling": bool(audit_sampling),
        "adaptive_fn_penalty": bool(adaptive_fn_penalty),
        "balanced_hard_replay": bool(balanced_hard_replay),
        "reject_cost": float(
            STRATEGY_REJECT_COST if reject_cost is None else reject_cost),
        "policy": dict(policy),
    }

PRIMARY_STRATEGY_VARIANTS = [
    strategy_variant("DRMD-IRAL", feedback_budget=0,
                     selector_mode="iral", policy=IRAL_POLICY),
    strategy_variant("DRMD-IRAAL"),
    strategy_variant("DRMD-FN", audit_sampling=True,
                     adaptive_fn_penalty=True),
    strategy_variant("DRMD-BHR", balanced_hard_replay=True),
    strategy_variant("DRMD-FN-BHR", audit_sampling=True,
                     adaptive_fn_penalty=True, balanced_hard_replay=True),
    strategy_variant("DRMD-FFCR", full_feedback=True, feedback_budget=0,
                     selector_mode="full_feedback", policy=FFCR_POLICY),
]
PRIMARY_STRATEGY_NAMES = [variant["name"] for variant in PRIMARY_STRATEGY_VARIANTS]
assert len(set(PRIMARY_STRATEGY_NAMES)) == 6

def _cost_suffix(value):
    return "C0" if float(value) == 0.0 else "Cm1"

BUDGET_SENSITIVITY_VARIANTS = []
for budget in SENSITIVITY_BUDGETS:
    if budget == FEEDBACK_UPDATE_BUDGET:
        continue
    policy = {"name": "IRAAL", "scope": "budgeted_update", "budget": budget}
    BUDGET_SENSITIVITY_VARIANTS.extend([
        strategy_variant(
            f"DRMD-IRAAL-B{budget}", feedback_budget=budget, policy=policy),
        strategy_variant(
            f"DRMD-FN-BHR-B{budget}", audit_sampling=True,
            adaptive_fn_penalty=True, balanced_hard_replay=True,
            feedback_budget=budget, policy=policy),
    ])

REJECT_COST_SENSITIVITY_VARIANTS = []
for reject_cost in SENSITIVITY_REJECT_COSTS:
    if abs(float(reject_cost) - float(STRATEGY_REJECT_COST)) < 1e-12:
        continue
    suffix = _cost_suffix(reject_cost)
    REJECT_COST_SENSITIVITY_VARIANTS.extend([
        strategy_variant(
            f"DRMD-IRAL-{suffix}", feedback_budget=0,
            selector_mode="iral", policy=IRAL_POLICY,
            reject_cost=reject_cost),
        strategy_variant(
            f"DRMD-IRAAL-{suffix}", reject_cost=reject_cost),
    ])

ALL_STRATEGY_VARIANTS = (
    list(PRIMARY_STRATEGY_VARIANTS)
    + BUDGET_SENSITIVITY_VARIANTS
    + REJECT_COST_SENSITIVITY_VARIANTS)
ALL_STRATEGY_NAMES = [variant["name"] for variant in ALL_STRATEGY_VARIANTS]
assert len(ALL_STRATEGY_VARIANTS) == 20
assert len(set(ALL_STRATEGY_NAMES)) == len(ALL_STRATEGY_NAMES)

VALID_RUN_PHASES = {
    "phase_a_primary", "phase_b_budget_sensitivity",
    "phase_c_reject_cost_sensitivity", "phase_d_cfjr_reference",
    "all", "aggregate_only"}
KAGGLE_RUN_PHASE = 'phase_a_primary'
assert KAGGLE_RUN_PHASE in VALID_RUN_PHASES
if KAGGLE_RUN_PHASE == "phase_a_primary":
    STRATEGY_VARIANTS = list(PRIMARY_STRATEGY_VARIANTS)
elif KAGGLE_RUN_PHASE == "phase_b_budget_sensitivity":
    STRATEGY_VARIANTS = list(BUDGET_SENSITIVITY_VARIANTS)
elif KAGGLE_RUN_PHASE == "phase_c_reject_cost_sensitivity":
    STRATEGY_VARIANTS = list(REJECT_COST_SENSITIVITY_VARIANTS)
else:
    STRATEGY_VARIANTS = list(ALL_STRATEGY_VARIANTS)
RUN_STRATEGY_EXPERIMENT = KAGGLE_RUN_PHASE in {
    "phase_a_primary", "phase_b_budget_sensitivity",
    "phase_c_reject_cost_sensitivity", "all"}
RUN_CFJR_REFERENCE = KAGGLE_RUN_PHASE in {"phase_d_cfjr_reference", "all"}

# ── Điều khiển Kaggle ────────────────────────────────────────────────────────
SESSION_LAUNCH_CUTOFF_HOURS = 10.5
RUNTIME_PRIOR = {"source": "v12_observed_runtime",
                 "median_minutes_per_run": 4.788,
                 "note": "chỉ dùng lập kế hoạch; V13 có checkpoint riêng"}
STOP_AFTER_DRMD_RUNS = None
STOP_AFTER_STATIC_RUNS = None
STOP_AFTER_CFJR_REFITS = None
MAX_CONSECUTIVE_FAILURES = 3
SESSION_STARTED_AT = time.monotonic()
FINALIZE_REPORT = KAGGLE_RUN_PHASE == "aggregate_only"
STRICT_INVARIANTS = FINALIZE_REPORT
REQUIRE_CUDA = True
BATCH_PREDICT_ROWS = 4096
LAMDA_ROOT_OVERRIDE = None

STATIC_RUN_COUNT = (len(SEEDS)
                    if KAGGLE_RUN_PHASE in {"phase_a_primary", "all"} else 0)
STRATEGY_RUN_COUNT = (len(STRATEGY_VARIANTS) * len(SEEDS)
                      if RUN_STRATEGY_EXPERIMENT else 0)
TOTAL_PLANNED_RUN_COUNT = STATIC_RUN_COUNT + STRATEGY_RUN_COUNT
CFJR_PLANNED_SEQUENCES = len(SEEDS)
CFJR_PLANNED_REFITS = len(SEEDS) * EXPECTED_TEST_PERIODS

PARAMETER_PROVENANCE = {
    "training_window_months": {
        "value": TRAINING_WINDOW, "category": "temporal_protocol",
        "basis": "12 calendar months; missing months are not replaced"},
    "dataset_revision": {
        "value": LAMDA_HF_REVISION, "category": "dataset_protocol",
        "validation": "all 24 Baseline parquet files match locked SHA-256"},
    "testing_window_months": {
        "value": TESTING_WINDOW, "category": "dataset_protocol"},
    "minority_priority": {
        "value": MP_FIXED, "category": "controlled_factor",
        "basis": "neutral reward weight shared by all twenty DRMD configurations"},
    "reject_cost": {
        "value": STRATEGY_REJECT_COST, "category": "locked_operating_point",
        "sensitivity": SENSITIVITY_REJECT_COSTS,
        "basis": "separate IRAL/IRAAL attribution; no B-by-cost cross-product"},
    "training_epochs": {
        "value": DRMD_TRAINING_EPOCHS, "category": "inherited_from_drmd"},
    "feedback_update_budget": {
        "value": FEEDBACK_UPDATE_BUDGET, "category": "locked_operating_point",
        "sensitivity": SENSITIVITY_BUDGETS,
        "validation": "IRAAL and FN-BHR are paired at every budget"},
    "iral_feedback": {
        "value": "Q_t_equals_R_t", "category": "inherited_from_drmd",
        "basis": "every integrated reject action is labeled after prediction",
        "validation": "selected indexes equal rejected indexes in every period"},
    "finetuning_size": {
        "value": STRATEGY_FINETUNING_SIZE, "category": "controlled_capacity"},
    "bhr_replay": {
        "value": {"fn_fraction": BHR_FN_FRACTION,
                  "fp_fraction": BHR_FP_FRACTION,
                  "background_fraction": BHR_BACKGROUND_FRACTION,
                  "fn_max_repeat": BHR_FN_MAX_REPEAT,
                  "fp_max_repeat": BHR_FP_MAX_REPEAT},
        "category": "proposed_method",
        "basis": "retain confirmed misses while anchoring false alarms",
        "validation": "labels from t are fit only at t+1; no extra label query"},
    "aligned_references": {
        "value": {"static_model": "Static-MLP", "full_feedback": "DRMD-FFCR",
                  "causal_full_history": CFJR_METHOD},
        "category": "synchronized_reference",
        "basis": ("same temporal split and all ten seeds; CFJR uses the locked "
                  "MLP architecture and all labels available through t-1"),
        "validation": "210/210 method-seed artifacts before full aggregation"},
    "cfjr_reference": {
        "value": {"method": CFJR_METHOD,
                  "protocol_version": CFJR_PROTOCOL_VERSION,
                  "sequences": CFJR_PLANNED_SEQUENCES,
                  "monthly_refits": CFJR_PLANNED_REFITS},
        "category": "strong_causal_empirical_reference",
        "basis": ("restart the locked MLP at every month and fit the initial "
                  "window plus all labels released through t-1"),
        "validation": ("period-level atomic artifacts; no current-month label "
                       "enters fit; separate completeness gate")},
    "experiment_matrix": {
        "value": {
            "primary": {"runs": 70, "methods": 7},
            "budget_sensitivity": {"runs": 100, "methods": 10},
            "reject_cost_sensitivity": {"runs": 40, "methods": 4},
            "total": 210},
        "category": "predeclared_analysis_plan",
        "basis": "targeted one-factor sensitivity without an unnecessary cross-grid"},
    "seeds": {
        "value": SEEDS, "category": "locked_replication",
        "basis": ("retain five V12 seeds and add five; with five paired seeds "
                  "the minimum two-sided exact sign-flip p is 2/2^5 = 0.0625, "
                  "whereas with ten it is 2/2^10 = 0.001953125"),
        "validation": "all comparisons are paired over all ten seeds"},
}

PROTOCOL_DEVIATIONS = {
    "new_strategy": {
        "name": "Balanced Hard Replay",
        "scope": "training-memory composition only",
        "label_budget_change": 0,
        "checkpoint_compatibility": "same_210_run_protocol_only"},
}

assert abs(BHR_FN_FRACTION + BHR_FP_FRACTION + BHR_BACKGROUND_FRACTION - 1.0) < 1e-12
print({"experiment": EXPERIMENT_NAME, "protocol_version": PROTOCOL_VERSION,
       "phase": KAGGLE_RUN_PHASE, "seeds": SEEDS,
       "static_runs": STATIC_RUN_COUNT,
       "strategy_runs": STRATEGY_RUN_COUNT,
       "strategy_variants": PRIMARY_STRATEGY_NAMES,
       "phase_strategy_variants": [item["name"] for item in STRATEGY_VARIANTS],
       "all_strategy_variants": ALL_STRATEGY_NAMES,
       "total_planned_runs": TOTAL_PLANNED_RUN_COUNT,
       "cfjr_planned_sequences": CFJR_PLANNED_SEQUENCES,
       "cfjr_planned_refits": CFJR_PLANNED_REFITS,
       "run_cfjr_reference": RUN_CFJR_REFERENCE,
       "bhr": PARAMETER_PROVENANCE["bhr_replay"]["value"]})


## Ô 2 — Khôi phục checkpoint riêng của V13


In [ ]:
# Chỉ khôi phục checkpoint mang đúng tên thí nghiệm V13.
RESTORE_EXISTING_RESULTS = True
if RESTORE_EXISTING_RESULTS:
    import shutil
    import zipfile

    input_root = Path("/kaggle/input")
    allowed_roots = {STRATEGY_RUN_TAG, "static_baselines", "cfjr_reference"}
    expected_zip_name = f"{EXPERIMENT_NAME}_checkpoint_raw.zip"
    checkpoint_zips = sorted(input_root.rglob(expected_zip_name))
    checkpoint_inputs = sorted({
        path for path in input_root.rglob("raw")
        if path.is_dir() and any((path / root).exists() for root in allowed_roots)
    })
    restored = 0
    already_present = 0

    for checkpoint_root in checkpoint_inputs:
        for src in list(checkpoint_root.rglob("*.p")) + list(checkpoint_root.rglob("*.p.meta.json")):
            relative = src.relative_to(checkpoint_root)
            if not relative.parts or relative.parts[0] not in allowed_roots:
                continue
            dst = RAW_OUT / relative
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                already_present += 1
            else:
                shutil.copy2(src, dst); restored += 1

    for zip_path in checkpoint_zips:
        if not zipfile.is_zipfile(zip_path):
            continue
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.namelist():
                parts = Path(member).parts
                if (len(parts) < 2 or parts[0] != "raw"
                        or parts[1] not in allowed_roots
                        or Path(member).is_absolute()
                        or ".." in parts
                        or not (member.endswith(".p") or member.endswith(".p.meta.json"))):
                    continue
                dst = OUT / member
                dst.parent.mkdir(parents=True, exist_ok=True)
                if dst.exists():
                    already_present += 1
                else:
                    with archive.open(member) as src, open(dst, "wb") as out:
                        shutil.copyfileobj(src, out)
                    restored += 1

    print({"checkpoint_protocol": PROTOCOL_VERSION,
           "checkpoint_archives_found": len(checkpoint_zips),
           "checkpoint_raw_roots_found": len(checkpoint_inputs),
           "restored_artifact_files": restored,
           "already_present_artifact_files": already_present})
    if not checkpoint_zips and not checkpoint_inputs:
        print("ℹ️ Không có checkpoint V13; bắt đầu ma trận mới.")


## Ô 3 — Kiểm kê sơ bộ artifact đã khôi phục

Ô này chỉ liệt kê tệp và khả năng đọc pickle. Artifact chỉ được công nhận sau khi vượt cổng đối chiếu dữ liệu, nhân quả, chữ ký và SHA-256 ở các ô huấn luyện/tổng hợp.


In [ ]:
# Kiểm kê sơ bộ sau khôi phục; chưa dùng để công nhận kết quả.
import collections
import json
import pickle
from pathlib import Path

def kiem_ke_artifact_da_khoi_phuc(raw_out: Path) -> dict:
    period_distribution = collections.Counter()
    files_by_variant = collections.Counter()
    unreadable = []
    missing_sidecar = []
    for artifact_path in sorted(Path(raw_out).rglob("*.p")):
        if ("static_baselines" in artifact_path.parts
                or "cfjr_reference" in artifact_path.parts):
            continue
        try:
            with open(artifact_path, "rb") as handle:
                result = pickle.load(handle)
            period_distribution[len(result.get("f1", []))] += 1
            files_by_variant[artifact_path.parent.name] += 1
            if not Path(str(artifact_path) + ".meta.json").exists():
                missing_sidecar.append(str(artifact_path))
        except Exception as exc:
            unreadable.append({
                "path": str(artifact_path),
                "error": f"{type(exc).__name__}: {exc}",
            })
    return {
        "period_distribution": dict(period_distribution),
        "files_by_variant": dict(files_by_variant),
        "unreadable": unreadable,
        "missing_sidecar": missing_sidecar,
        "status": "inventory_only_not_scientific_validation",
    }

RESTORED_ARTIFACT_INVENTORY = kiem_ke_artifact_da_khoi_phuc(RAW_OUT)
print(json.dumps(RESTORED_ARTIFACT_INVENTORY, indent=2, ensure_ascii=False))
(TABLE_OUT / "restored_artifact_inventory.json").write_text(
    json.dumps(RESTORED_ARTIFACT_INVENTORY, indent=2, ensure_ascii=False),
    encoding="utf-8")


## Ô 4 — Bắt `UndefinedMetricWarning` thay vì để nó ngập log

Các chu kỳ không có đủ hỗ trợ lớp được ghi thành artifact kiểm toán thay vì làm ngập
nhật ký. Thanh tiến trình dùng trực tiếp `len(X_tests)`, không ghi cứng số chu kỳ.

In [ ]:
import warnings, csv as _csv
from sklearn.exceptions import UndefinedMetricWarning

warnings.simplefilter("always", UndefinedMetricWarning)
UNDEFINED_LOG: list[dict] = []
CURRENT_RUN_TAG, CURRENT_PERIOD_IDX = "", -1

_orig_showwarning = warnings.showwarning
def _ghi_nhat_ky(message, category, filename, lineno, file=None, line=None):
    if category is UndefinedMetricWarning:
        UNDEFINED_LOG.append({"run_tag": CURRENT_RUN_TAG, "period": CURRENT_PERIOD_IDX,
                              "message": str(message)})
        return                                   # không in, tránh ngập log Kaggle
    _orig_showwarning(message, category, filename, lineno, file, line)
warnings.showwarning = _ghi_nhat_ky

def xuat_nhat_ky_suy_bien():
    if not UNDEFINED_LOG:
        print("✅ Không có độ đo suy biến nào bị ghi nhận."); return
    with open(TABLE_OUT / "undefined_metric_periods.csv", "w", newline="", encoding="utf-8") as fh:
        w = _csv.DictWriter(fh, fieldnames=["run_tag", "period", "message"])
        w.writeheader(); w.writerows(UNDEFINED_LOG)
    print(f"⚠️  {len(UNDEFINED_LOG)} lần độ đo suy biến — xem undefined_metric_periods.csv")

## Ô 5 — Cố định các nguồn ngẫu nhiên chính và ghi dấu môi trường

Python `random`, NumPy, PyTorch, CUDA và thuật toán cuDNN được khóa ngay trước mỗi lượt.
`PYTHONHASHSEED` chỉ có hiệu lực đầy đủ khi được đặt trước lúc kernel khởi động, nên
notebook không dựa vào thứ tự hash cho bất kỳ phép chia hay chọn mẫu nào.

In [ ]:
import os, random, platform, subprocess
# Phải đặt trước lần khởi tạo ngữ cảnh CUDA đầu tiên.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np, torch

def co_dinh_hat_giong(seed: int) -> None:
    """Gọi NGAY TRƯỚC mỗi lần khởi tạo mô hình."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ["PYTHONHASHSEED"] = str(seed)  # áp dụng cho tiến trình con tạo sau đây

def git_commit_for(path):
    try:
        return subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "không có"

def dau_moi_truong() -> dict:
    import sklearn, scipy
    return {"python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "scipy": scipy.__version__,
            "scikit_learn": sklearn.__version__,
            "cuda": torch.version.cuda if torch.cuda.is_available() else None,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "cudnn_deterministic_at_stamp": bool(
                torch.backends.cudnn.deterministic),
            "deterministic_algorithms_at_stamp": bool(
                torch.are_deterministic_algorithms_enabled()),
            "determinism_policy": (
                "co_dinh_hat_giong(seed) được gọi ngay trước mỗi lần khởi tạo mô hình"),
            "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
            "python_hash_seed_note": "kernel hiện tại phải được khởi động với biến này; không dùng hash-order trong giao thức",
            "working_directory_git_commit": git_commit_for(Path.cwd()),
            "protocol_version": PROTOCOL_VERSION,
            "protocol_deviations": PROTOCOL_DEVIATIONS,
            "training_window_mode": TRAINING_WINDOW_MODE}

ENVIRONMENT_STAMP = dau_moi_truong()
print(json.dumps(ENVIRONMENT_STAMP, indent=2, ensure_ascii=False))

## Ô 6 — Nạp thư viện và vá DRMD

Hàm phần thưởng được đối chiếu với `environment.py` của mã gốc: trước khi nhân trọng
số thời gian, `TN = +1`, `FP = −1`, `TP = +m_p`, `FN = −m_p · λ_FN`.

In [ ]:
# Kaggle already provides NumPy/SciPy/scikit-learn/PyArrow.  Install only the
# DRMD dependencies missing from the base image; avoiding broad upgrades keeps
# CUDA packages consistent with the Kaggle runtime.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "StrEnum==0.4.15", "hydra-core==1.3.2", "backpack-for-pytorch==1.7.1"])
from importlib import metadata as _package_metadata

_required_runtime_distributions = {
    "StrEnum": "0.4.15",
    "hydra-core": "1.3.2",
    "backpack-for-pytorch": "1.7.1",
}
_installed_runtime_distributions = {
    name: _package_metadata.version(name)
    for name in _required_runtime_distributions
}
if _installed_runtime_distributions != _required_runtime_distributions:
    raise RuntimeError(
        "Phiên bản dependency DRMD lệch cấu hình khóa: "
        f"{_installed_runtime_distributions}")
_torch_distribution_version = _package_metadata.version("torch")
if not str(torch.__version__).startswith(_torch_distribution_version):
    raise RuntimeError(
        "pip đã thay torch trên đĩa nhưng kernel còn giữ module cũ; "
        "hãy khởi động lại phiên Kaggle trước khi chạy.")
ENVIRONMENT_STAMP["installed_runtime_distributions"] = (
    _installed_runtime_distributions)


In [ ]:
import os, sys, re, csv, json, pickle, zipfile
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import scipy.sparse as sp

if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

INPUT_ROOT = Path("/kaggle/input")
INPUT_PROJECT_CANDIDATES = [
    INPUT_ROOT / "datasets/thanhngo1007/drmd-lamda-dataset/DRMD_LAMDA_DATASET",
    INPUT_ROOT / "drmd-lamda-dataset/DRMD_LAMDA_DATASET",
    INPUT_ROOT / "drmd-lamda-dataset",
]

def is_project(path):
    return path.is_dir() and (path / "references/DRMD").is_dir()

PROJECT = next((p for p in INPUT_PROJECT_CANDIDATES if is_project(p)), None)
if PROJECT is None and INPUT_ROOT.exists():
    # Kaggle mounts datasets under a user-controlled slug.  Search only for
    # directories that contain the DRMD source to avoid selecting a wrong input.
    PROJECT = next((p for p in INPUT_ROOT.rglob("DRMD_LAMDA_DATASET") if is_project(p)), None)
if PROJECT is None and INPUT_ROOT.exists():
    PROJECT = next((p for p in INPUT_ROOT.iterdir() if is_project(p)), None)
if PROJECT is None:
    mounted = [p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "Khong tim thay DRMD_LAMDA_DATASET. Trong Kaggle, chon Add Input va gan dataset "
        "chua thu muc references/DRMD va du lieu LAMDA. Mounted inputs hien co: " + repr(mounted))

DRMD_ROOT = PROJECT / "references/DRMD"
EXP_DIR = DRMD_ROOT / "Experiments"
TESSERACT_ROOT = PROJECT / "references/tesseract-ml-release"
RPAL_ROOT = PROJECT / "references/RPAL"

for p in [PROJECT, DRMD_ROOT, EXP_DIR, TESSERACT_ROOT, RPAL_ROOT]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("PROJECT:", PROJECT)
print("DRMD source:", DRMD_ROOT)


ENVIRONMENT_STAMP.update({
    "project_git_commit": git_commit_for(PROJECT),
    "drmd_git_commit": git_commit_for(DRMD_ROOT),
})
print("Dấu vết mã nguồn:", {
    "project_git_commit": ENVIRONMENT_STAMP["project_git_commit"],
    "drmd_git_commit": ENVIRONMENT_STAMP["drmd_git_commit"],
})
# V13 tự mang theo đúng mã BHR; không phụ thuộc Dataset Kaggle đã có thư mục
# experiments hay chưa. Tệp được bung vào /kaggle/working và đưa lên đầu sys.path.
import hashlib as _bhr_hashlib
import importlib as _bhr_importlib

BHR_EMBEDDED_SOURCE = r'''"""Balanced Hard Replay cho thí nghiệm DRMD-FN trên dòng dữ liệu LAMDA.

Mô-đun này chỉ thay cách cấu tạo bộ nhớ huấn luyện từ các nhãn đã được mở.
Nhãn của chu kỳ ``t`` phải được đưa vào bằng :meth:`observe` sau dự đoán và
chỉ được dùng khi gọi :meth:`build` cho chu kỳ ``t + 1`` hoặc muộn hơn.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any

import numpy as np

try:  # Kaggle có SciPy; nhánh NumPy giữ cho kiểm thử logic chạy tối giản.
    import scipy.sparse as sp
except ModuleNotFoundError:  # pragma: no cover - chỉ dùng ở môi trường kiểm thử nhẹ
    sp = None


@dataclass(frozen=True)
class ReplayTelemetry:
    """Số liệu kiểm toán của một bộ nhớ được dựng cho lần fit kế tiếp."""

    fit_period: int
    memory_cap: int
    memory_rows: int
    unique_rows_used: int
    fn_unique_buffer: int
    fp_unique_buffer: int
    background_unique_buffer: int
    fn_rows: int
    fp_rows: int
    background_rows: int
    fn_replay_rows: int
    fp_replay_rows: int
    replay_fraction: float
    mean_labeled_age_periods: float
    max_labeled_age_periods: float
    newest_source_period: int
    causal_lag_ok: bool

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class ReplayBatch:
    """Ma trận huấn luyện và dấu vết nguồn của BHR."""

    X: Any
    y: np.ndarray
    t: np.ndarray
    source_period: np.ndarray
    source_bucket: np.ndarray
    telemetry: ReplayTelemetry


class _Bucket:
    """Bộ đệm thưa, độc quyền theo loại lỗi tại thời điểm nhãn được mở."""

    def __init__(self, name: str, unique_cap: int, seed: int) -> None:
        self.name = str(name)
        self.unique_cap = int(unique_cap)
        self.seed = int(seed)
        self.X: Any | None = None
        self.y = np.asarray([], dtype=np.int8)
        self.t = np.asarray([], dtype="datetime64[ns]")
        self.source_period = np.asarray([], dtype=np.int64)

    def __len__(self) -> int:
        return int(self.y.size)

    @staticmethod
    def _as_matrix(X: Any) -> Any:
        if sp is None:
            matrix = np.asarray(X)
            if matrix.ndim != 2:
                raise ValueError("X phải là ma trận hai chiều.")
            return matrix
        matrix = X if sp.issparse(X) else sp.csr_matrix(np.asarray(X))
        return matrix.tocsr()

    @staticmethod
    def _vstack(matrices: list[Any]) -> Any:
        if sp is None:
            return np.vstack(matrices)
        return sp.vstack(matrices, format="csr")

    def append(
        self,
        X: Any,
        y: np.ndarray,
        t: np.ndarray,
        source_period: int,
    ) -> None:
        y_array = np.asarray(y, dtype=np.int8).reshape(-1)
        if y_array.size == 0:
            return
        t_array = np.asarray(t).reshape(-1).astype("datetime64[ns]")
        X_matrix = self._as_matrix(X)
        if not (X_matrix.shape[0] == y_array.size == t_array.size):
            raise ValueError("X, y và t phải có cùng số hàng.")
        periods = np.full(y_array.size, int(source_period), dtype=np.int64)
        self.X = (
            X_matrix
            if self.X is None
            else self._vstack([self.X, X_matrix])
        )
        self.y = np.concatenate([self.y, y_array])
        self.t = np.concatenate([self.t, t_array])
        self.source_period = np.concatenate([self.source_period, periods])
        self._cap_to_recent()

    def _cap_to_recent(self) -> None:
        if len(self) <= self.unique_cap:
            return
        month_ids = self.t.astype("datetime64[M]").astype(np.int64)
        chosen: list[np.ndarray] = []
        remaining = self.unique_cap
        for month_id in np.sort(np.unique(month_ids))[::-1]:
            candidates = np.flatnonzero(month_ids == month_id)
            if candidates.size <= remaining:
                chosen.append(candidates)
                remaining -= int(candidates.size)
            else:
                local_seed = (
                    self.seed + 2654435761 * int(month_id)
                ) % (2**32)
                rng = np.random.default_rng(local_seed)
                chosen.append(
                    np.sort(rng.choice(candidates, remaining, replace=False))
                )
                remaining = 0
            if remaining == 0:
                break
        indexes = np.sort(np.concatenate(chosen).astype(np.int64, copy=False))
        if indexes.size != self.unique_cap:
            raise RuntimeError("Không dựng được bộ đệm gần nhất đúng kích thước.")
        assert self.X is not None
        self.X = self.X[indexes]
        self.y = self.y[indexes]
        self.t = self.t[indexes]
        self.source_period = self.source_period[indexes]


class BalancedHardReplayBuffer:
    """Bộ nhớ tái phát cân bằng giữa FN, FP khó và mẫu nền gần đây.

    Các ngăn là độc quyền theo dự đoán đã khóa tại lúc nhãn được mở:

    * ``fn``: nhãn thật 1, dự đoán nhị phân 0;
    * ``fp``: nhãn thật 0, dự đoán nhị phân 1;
    * ``background``: dữ liệu huấn luyện khởi tạo cùng TP/TN mới.

    Quota là quota hàng huấn luyện, không phải quota nhãn mới. Nếu ngăn FN/FP
    chưa đủ, các mẫu nền gần nhất bù phần trống. Mỗi mẫu FN/FP không vượt quá
    giới hạn lặp đã khóa.
    """

    def __init__(
        self,
        memory_cap: int = 5000,
        fn_fraction: float = 0.35,
        fp_fraction: float = 0.25,
        fn_max_repeat: int = 4,
        fp_max_repeat: int = 2,
        seed: int = 0,
    ) -> None:
        if int(memory_cap) <= 0:
            raise ValueError("memory_cap phải dương.")
        if not (0.0 <= fn_fraction <= 1.0 and 0.0 <= fp_fraction <= 1.0):
            raise ValueError("Các tỷ lệ phải thuộc [0, 1].")
        if fn_fraction + fp_fraction > 1.0:
            raise ValueError("Tổng tỷ lệ FN và FP không được vượt 1.")
        if int(fn_max_repeat) < 1 or int(fp_max_repeat) < 1:
            raise ValueError("Giới hạn lặp phải ít nhất bằng 1.")

        self.memory_cap = int(memory_cap)
        self.fn_fraction = float(fn_fraction)
        self.fp_fraction = float(fp_fraction)
        self.fn_max_repeat = int(fn_max_repeat)
        self.fp_max_repeat = int(fp_max_repeat)
        self.seed = int(seed)
        self._buckets = {
            "fn": _Bucket("fn", self.memory_cap, self.seed + 11),
            "fp": _Bucket("fp", self.memory_cap, self.seed + 23),
            "background": _Bucket(
                "background", self.memory_cap, self.seed + 37
            ),
        }

    def initialize(self, X: Any, y: np.ndarray, t: np.ndarray) -> None:
        """Đưa tập huấn luyện ban đầu vào ngăn nền, với nguồn trước test."""
        if any(len(bucket) for bucket in self._buckets.values()):
            raise RuntimeError("BHR chỉ được khởi tạo một lần.")
        y_array = np.asarray(y, dtype=np.int8).reshape(-1)
        if np.setdiff1d(np.unique(y_array), [0, 1]).size:
            raise ValueError("BHR yêu cầu nhãn nhị phân 0/1.")
        self._buckets["background"].append(X, y_array, t, source_period=-1)

    def observe(
        self,
        X: Any,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        t: np.ndarray,
        source_period: int,
    ) -> None:
        """Ghi nhãn đã mở sau dự đoán; chưa dựng bộ nhớ cho cùng chu kỳ."""
        if int(source_period) < 0:
            raise ValueError("source_period của phản hồi phải không âm.")
        y_array = np.asarray(y_true, dtype=np.int8).reshape(-1)
        pred_array = np.asarray(y_pred, dtype=np.int8).reshape(-1)
        t_array = np.asarray(t).reshape(-1)
        X_matrix = _Bucket._as_matrix(X)
        if not (
            X_matrix.shape[0] == y_array.size == pred_array.size == t_array.size
        ):
            raise ValueError("X, y_true, y_pred và t phải có cùng số hàng.")
        if np.setdiff1d(np.unique(y_array), [0, 1]).size:
            raise ValueError("y_true phải là nhãn nhị phân 0/1.")
        if np.setdiff1d(np.unique(pred_array), [0, 1]).size:
            raise ValueError("y_pred phải là dự đoán nhị phân 0/1.")

        masks = {
            "fn": (y_array == 1) & (pred_array == 0),
            "fp": (y_array == 0) & (pred_array == 1),
        }
        masks["background"] = ~(masks["fn"] | masks["fp"])
        for name, mask in masks.items():
            indexes = np.flatnonzero(mask)
            self._buckets[name].append(
                X_matrix[indexes], y_array[indexes], t_array[indexes], source_period
            )

    @staticmethod
    def _allocate(
        bucket_size: int,
        target: int,
        max_repeat: int,
        existing_counts: np.ndarray | None = None,
    ) -> tuple[list[int], np.ndarray]:
        counts = (
            np.zeros(bucket_size, dtype=np.int64)
            if existing_counts is None
            else np.asarray(existing_counts, dtype=np.int64).copy()
        )
        selected: list[int] = []
        # Chỉ số lớn hơn là mẫu mới hơn vì mọi lần append đi theo thời gian.
        recent_first = np.arange(bucket_size - 1, -1, -1, dtype=np.int64)
        while len(selected) < int(target):
            eligible = recent_first[counts[recent_first] < int(max_repeat)]
            if eligible.size == 0:
                break
            remaining = int(target) - len(selected)
            take = eligible[:remaining]
            selected.extend(int(index) for index in take)
            counts[take] += 1
        return selected, counts

    def build(self, fit_period: int) -> ReplayBatch:
        """Dựng bộ nhớ cho ``fit_period`` và cưỡng chế trễ phản hồi một chu kỳ."""
        fit_period = int(fit_period)
        newest = max(
            (
                int(bucket.source_period.max())
                for bucket in self._buckets.values()
                if len(bucket)
            ),
            default=-1,
        )
        if newest >= fit_period:
            raise ValueError(
                "Vi phạm nhân quả: phản hồi của chu kỳ hiện tại không được fit ngay."
            )

        fn_target = int(np.floor(self.memory_cap * self.fn_fraction))
        fp_target = int(np.floor(self.memory_cap * self.fp_fraction))
        background_target = self.memory_cap - fn_target - fp_target

        allocations: dict[str, list[int]] = {}
        counts: dict[str, np.ndarray] = {}
        allocations["fn"], counts["fn"] = self._allocate(
            len(self._buckets["fn"]), fn_target, self.fn_max_repeat
        )
        allocations["fp"], counts["fp"] = self._allocate(
            len(self._buckets["fp"]), fp_target, self.fp_max_repeat
        )
        allocations["background"], counts["background"] = self._allocate(
            len(self._buckets["background"]), background_target, 1
        )

        remaining = self.memory_cap - sum(len(items) for items in allocations.values())
        # Quota hard còn trống được bù bởi nền gần đây; sau đó mới dùng phần
        # sức chứa lặp hard còn lại. Không mẫu nào vượt giới hạn lặp đã khóa.
        for name, repeat_limit in (
            ("background", 1),
            ("fn", self.fn_max_repeat),
            ("fp", self.fp_max_repeat),
        ):
            if remaining <= 0:
                break
            extra, updated = self._allocate(
                len(self._buckets[name]), remaining, repeat_limit, counts[name]
            )
            allocations[name].extend(extra)
            counts[name] = updated
            remaining -= len(extra)

        matrices: list[Any] = []
        labels: list[np.ndarray] = []
        times: list[np.ndarray] = []
        source_periods: list[np.ndarray] = []
        bucket_names: list[np.ndarray] = []
        for name in ("background", "fn", "fp"):
            indexes = np.asarray(allocations[name], dtype=np.int64)
            if indexes.size == 0:
                continue
            bucket = self._buckets[name]
            assert bucket.X is not None
            matrices.append(bucket.X[indexes])
            labels.append(bucket.y[indexes])
            times.append(bucket.t[indexes])
            source_periods.append(bucket.source_period[indexes])
            bucket_names.append(np.full(indexes.size, name, dtype="U10"))

        if not matrices:
            raise RuntimeError("BHR chưa có dữ liệu để dựng bộ nhớ.")
        X_out = _Bucket._vstack(matrices)
        y_out = np.concatenate(labels)
        t_out = np.concatenate(times)
        periods_out = np.concatenate(source_periods)
        names_out = np.concatenate(bucket_names)

        rng = np.random.default_rng(
            (self.seed + 1000003 * fit_period) % (2**32)
        )
        order = rng.permutation(y_out.size)
        X_out = X_out[order]
        y_out = y_out[order]
        t_out = t_out[order]
        periods_out = periods_out[order]
        names_out = names_out[order]

        fn_rows = int(len(allocations["fn"]))
        fp_rows = int(len(allocations["fp"]))
        background_rows = int(len(allocations["background"]))
        fn_unique_used = int(np.count_nonzero(counts["fn"]))
        fp_unique_used = int(np.count_nonzero(counts["fp"]))
        background_unique_used = int(np.count_nonzero(counts["background"]))
        unique_used = fn_unique_used + fp_unique_used + background_unique_used
        replay_rows = (fn_rows - fn_unique_used) + (fp_rows - fp_unique_used)
        labeled_ages = fit_period - periods_out[periods_out >= 0]
        telemetry = ReplayTelemetry(
            fit_period=fit_period,
            memory_cap=self.memory_cap,
            memory_rows=int(y_out.size),
            unique_rows_used=unique_used,
            fn_unique_buffer=len(self._buckets["fn"]),
            fp_unique_buffer=len(self._buckets["fp"]),
            background_unique_buffer=len(self._buckets["background"]),
            fn_rows=fn_rows,
            fp_rows=fp_rows,
            background_rows=background_rows,
            fn_replay_rows=max(0, fn_rows - fn_unique_used),
            fp_replay_rows=max(0, fp_rows - fp_unique_used),
            replay_fraction=(
                float(replay_rows / y_out.size) if y_out.size else 0.0
            ),
            mean_labeled_age_periods=(
                float(np.mean(labeled_ages)) if labeled_ages.size else float("nan")
            ),
            max_labeled_age_periods=(
                float(np.max(labeled_ages)) if labeled_ages.size else float("nan")
            ),
            newest_source_period=newest,
            causal_lag_ok=bool(newest < fit_period),
        )
        return ReplayBatch(
            X=X_out,
            y=y_out,
            t=t_out,
            source_period=periods_out,
            source_bucket=names_out,
            telemetry=telemetry,
        )

    def configuration(self) -> dict[str, Any]:
        return {
            "memory_cap": self.memory_cap,
            "fn_fraction": self.fn_fraction,
            "fp_fraction": self.fp_fraction,
            "background_fraction": 1.0 - self.fn_fraction - self.fp_fraction,
            "fn_max_repeat": self.fn_max_repeat,
            "fp_max_repeat": self.fp_max_repeat,
            "seed": self.seed,
            "feedback_timing": "observe_after_predict_t_then_fit_t_plus_1",
        }
'''
BHR_EMBEDDED_SHA256 = '4bf5ac205ee4d3a183005a27894893a28d41833232a478751518edfea354b64a'
BHR_RUNTIME_ROOT = WORK / "_v13_embedded_source"
BHR_MODULE_PATH = BHR_RUNTIME_ROOT / "experiments/strategies/balanced_hard_replay.py"
for _package_dir in (BHR_RUNTIME_ROOT / "experiments",
                     BHR_RUNTIME_ROOT / "experiments/strategies"):
    _package_dir.mkdir(parents=True, exist_ok=True)
    (_package_dir / "__init__.py").write_text("", encoding="utf-8")
_bhr_bytes = BHR_EMBEDDED_SOURCE.encode("utf-8")
if _bhr_hashlib.sha256(_bhr_bytes).hexdigest() != BHR_EMBEDDED_SHA256:
    raise RuntimeError("Mã BHR nhúng trong notebook không khớp SHA-256.")
BHR_MODULE_PATH.write_bytes(_bhr_bytes)
if str(BHR_RUNTIME_ROOT) in sys.path:
    sys.path.remove(str(BHR_RUNTIME_ROOT))
sys.path.insert(0, str(BHR_RUNTIME_ROOT))
_bhr_importlib.invalidate_caches()
print("BHR embedded source:", BHR_MODULE_PATH,
      "sha256=", BHR_EMBEDDED_SHA256)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHẦN 1: MONKEY-PATCH __init__ & __reward_function
# ═══════════════════════════════════════════════════════════════
import sys
import importlib
import torch

if REQUIRE_CUDA and KAGGLE_RUN_PHASE != "aggregate_only" and not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA is required. In Kaggle Settings, enable a GPU accelerator before Run All.")
import DRMD.environment as drmd_env

importlib.reload(drmd_env)
print("🔄 Reloaded DRMD.environment")

# ─── 1A: Patch __init__ để bổ sung hệ số phạt FN ───
_original_init = drmd_env.DRMD.__init__

def _patched_init(self, settings, *args, **kwargs):
    _original_init(self, settings, *args, **kwargs)
    self.fn_penalty = float(getattr(settings, '_fn_penalty', 1.0))
    if int(self.minority_label) != 1 or int(self.majority_label) != 0:
        raise AssertionError(
            "DRMD-FN yêu cầu nhãn mã độc=1 và lành tính=0; không dùng tần suất lớp để đổi nhãn.")
    self.adaptive_fn_penalty = bool(getattr(settings, '_adaptive_fn_penalty', False))
    self.afnp_audit_sampling = bool(getattr(settings, '_afnp_audit_sampling', False))
    self.afnp_budget = int(getattr(settings, '_afnp_budget', 0))
    self.afnp_seed = int(getattr(settings, '_afnp_seed', getattr(settings, 'seed', 0)))
    self.afnp_window = int(getattr(settings, '_afnp_window', 12))
    self.afnp_dual = 0.0
    self.afnp_target_fnr = None
    self.afnp_controller_step = 0
    self.afnp_audit_history = []
    # Thuộc tính chiến lược gắn động, không sửa API kho DRMD tham chiếu.
    self._feedback_selector_mode = str(getattr(settings, '_feedback_selector_mode', 'iral'))
    self._feedback_budget = int(getattr(settings, '_feedback_budget', 0))
    self._candidate_multiplier = int(getattr(settings, '_candidate_multiplier', 10))
    self._full_feedback = bool(getattr(settings, '_full_feedback', False))
    self._training_history_cap = getattr(settings, '_training_history_cap', None)
    self._balanced_hard_replay = bool(getattr(settings, '_balanced_hard_replay', False))
    self._bhr_fn_fraction = float(getattr(settings, '_bhr_fn_fraction', 0.35))
    self._bhr_fp_fraction = float(getattr(settings, '_bhr_fp_fraction', 0.25))
    self._bhr_fn_max_repeat = int(getattr(settings, '_bhr_fn_max_repeat', 4))
    self._bhr_fp_max_repeat = int(getattr(settings, '_bhr_fp_max_repeat', 2))
    self._strategy_seed = int(settings.seed)
    self._fit_calls = 0
    self._selection_history = []
    self._temporal_reference_policy = FIXED_TEMPORAL_REFERENCE_POLICY
    self._temporal_reference_start = None
    self._temporal_reference_end = None
    self._temporal_reference_period = None
    self._current_memory_start = None
    self._current_memory_end = None
    self._current_memory_period = None
    self._temporal_reference_history = []
    if self.fn_penalty != 1.0:
        print(f"  ⚙️ FN Penalty = {self.fn_penalty} (Bất đối xứng)")

drmd_env.DRMD.__init__ = _patched_init
print("✅ Patched: DRMD.__init__ (FN penalty)")

# ─── 1B: Patch __reward_function (FN Penalty bất đối xứng) ───
_original_reward = drmd_env.DRMD._DRMD__reward_function

def _patched_reward_function(self, action, label, time=None, obs=None):
    """
    Phiên bản hỗ trợ FN Penalty bất đối xứng.
    Khi self.fn_penalty > 1.0, phạt FN nặng hơn bình thường.
    Tương thích 100% khi fn_penalty = 1.0 (hành vi giống hàm gốc).
    """
    reward = torch.zeros_like(action, dtype=torch.float32, device=self.device)

    # Classification Reward
    reward += torch.where(action == label, self.correct_reward, self.incorrect_cost)

    # Phạt bất đối xứng cho FN
    fn_penalty = getattr(self, 'fn_penalty', 1.0)
    if fn_penalty != 1.0:
        # FN mask: model dự đoán benign (action=0) nhưng thực tế là malware (label=1)
        fn_mask = (action != label) & (label == self.minority_label)
        reward[fn_mask] *= fn_penalty

    # Apply minority/majority scaling
    reward *= torch.where(label == self.minority_label, self.minority_priority, self.majority_priority)

    # Temporal Scaling
    if time is not None and self.temporal_rewards:
        temporal_position = (time + self.first_date - self.start_date + 1) / self.training_period
        reward *= temporal_position * self.temporal_scaling

    # Reject action handling
    if self.is_reject_action:
        rejection_mask = (action == self.reject_action) & self.is_reject_action
        reward[rejection_mask] = self.reject_cost

        if (obs is not None and self.reward_rejected_outcome
                and bool(torch.any(rejection_mask).item())):
            next_action = self.model.agent.get_next_likely_action(obs[rejection_mask], action[rejection_mask])
            rejected_outcome_reward = -_patched_reward_function(self, next_action, label[rejection_mask])
            scaling_factor = torch.where(rejected_outcome_reward > 0, self.reject_positive_scale, self.reject_negative_scale)
            reward[rejection_mask] += rejected_outcome_reward * scaling_factor

    return reward

drmd_env.DRMD._DRMD__reward_function = _patched_reward_function
print("✅ Patched: DRMD.__reward_function (FN Penalty bất đối xứng)")

if 'DRMD.base' in sys.modules:
    importlib.reload(sys.modules['DRMD.base'])
    print("🔄 Reloaded DRMD.base")

# ═══════════════════════════════════════════════════════════════
# PHẦN 2: MONKEY-PATCH fit_predict_reject_sample_update
# ═══════════════════════════════════════════════════════════════
import DRMD.utils.classifier_utils as clf_utils
from copy import deepcopy
from tqdm import tqdm
import numpy as np
import scipy
from sklearn import metrics as skmetrics
from tesseract import utils as tess_utils, metrics as tess_metrics
from DRMD.utils.classifier_utils import UncertaintyRejector, UncertaintySelector
from experiments.strategies.balanced_hard_replay import BalancedHardReplayBuffer

def _deterministic_recency_cap(X, y, t, cap, seed):
    """Giữ tháng gần nhất; lấy mẫu đều có seed tại tháng biên.

    Hàm không đọc nhãn để chọn hàng. Quy tắc tránh thiên lệch hệ thống
    do luôn giữ các hàng cuối khi một tháng vượt giới hạn bộ đệm; dấu
    vân tay nguồn vẫn là điều kiện để tái lập đúng tập hàng.
    """
    cap = int(cap)
    if cap <= 0:
        raise ValueError('Giới hạn lịch sử phải dương.')
    n_rows = int(len(y))
    if n_rows <= cap:
        return X, np.asarray(y), np.asarray(t)
    times = np.asarray(t)
    month_ids = times.astype('datetime64[M]').astype(np.int64)
    chosen, remaining = [], cap
    for month_id in np.sort(np.unique(month_ids))[::-1]:
        candidates = np.flatnonzero(month_ids == month_id)
        if candidates.size <= remaining:
            chosen.append(candidates)
            remaining -= int(candidates.size)
        else:
            local_seed = (
                int(seed) + 2654435761 * int(month_id)
            ) % (2 ** 32)
            rng = np.random.default_rng(local_seed)
            chosen.append(np.sort(rng.choice(
                candidates, size=remaining, replace=False)))
            remaining = 0
        if remaining == 0:
            break
    indexes = np.sort(np.concatenate(chosen).astype(int, copy=False))
    if indexes.size != cap:
        raise RuntimeError(
            f'Bộ đệm lịch sử có {indexes.size} hàng, kỳ vọng {cap}.')
    return X[indexes], np.asarray(y)[indexes], times[indexes]

def _apply_training_memory(clf, X, y, t):
    """Giữ đúng giới hạn 5.000 mẫu gần nhất bằng quy tắc đã khóa trong giao thức V13."""
    cap = getattr(clf, '_training_history_cap', None)
    if cap is None or X.shape[0] <= int(cap):
        return X, np.asarray(y), np.asarray(t)
    return _deterministic_recency_cap(
        X, y, t, int(cap), int(getattr(clf, '_strategy_seed', 0)))

def _actor_uncertainty(clf, X_batch, t_batch):
    """Tính bất định từ policy; không đọc nhãn của chu kỳ đang chọn."""
    actor = clf.model.agent.actor
    was_training = actor.training
    actor.eval()
    chunks = []
    try:
        with torch.no_grad():
            for start in range(0, X_batch.shape[0], max(1, int(BATCH_PREDICT_ROWS))):
                stop = min(start + max(1, int(BATCH_PREDICT_ROWS)), X_batch.shape[0])
                obs = _observation_chunk(clf, X_batch[start:stop], t_batch[start:stop])
                probabilities = torch.softmax(actor(obs), dim=-1)
                chunks.append((1.0 - probabilities.max(dim=1).values).cpu().numpy())
                del obs, probabilities
    finally:
        actor.train(was_training)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.concatenate(chunks).astype(np.float64, copy=False)

def _select_feedback_indexes(clf, X_test, t_test, rejected_indexes,
                             budget_override=None, excluded_indexes=()):
    """Dựng tập phản hồi sau khi khóa dự đoán, không đọc nhãn hiện tại."""
    selection_started = time.perf_counter()
    rejected = np.unique(np.asarray(rejected_indexes, dtype=int))
    excluded = np.unique(np.asarray(excluded_indexes, dtype=int))
    rejected = np.setdiff1d(rejected, excluded, assume_unique=False)
    mode = str(getattr(clf, '_feedback_selector_mode', 'iral'))
    budget = (int(getattr(clf, '_feedback_budget', 0))
              if budget_override is None else int(budget_override))
    available = np.setdiff1d(
        np.arange(int(X_test.shape[0]), dtype=int), excluded, assume_unique=False)
    base = {
        "selector_mode": mode, "budget": budget,
        "n_rejected": int(rejected.size), "n_rows": int(X_test.shape[0]),
        "selection_seconds": time.perf_counter() - selection_started,
    }
    if mode == 'iral':
        # IRAL nguyên gốc không có B cố định: mọi hành động từ chối tạo phản
        # hồi cho chu kỳ kế tiếp, tức Q_t = R_t.
        selected = rejected.copy()
        return selected, {
            **base, "budget_semantics": "endogenous_rejected_set",
            "candidate_pool_size": int(rejected.size),
            "n_selected": int(selected.size), "mean_uncertainty": float('nan')}
    if mode == 'full_feedback':
        # FFCR được ghi đè thành toàn bộ chỉ mục ở cuối pha quyết định. Trả về
        # ngay tại đây để không suy luận bất định thừa trên 885.947 mẫu.
        selected = available.copy()
        return selected, {
            **base, "budget_semantics": "full_feedback_reference",
            "candidate_pool_size": int(available.size),
            "n_selected": int(selected.size), "mean_uncertainty": float('nan')}
    if mode != 'uncertainty':
        raise ValueError(f'Bộ chọn phản hồi V13 không hợp lệ: {mode}')
    target = min(max(0, budget), int(available.size))
    uncertainty = _actor_uncertainty(clf, X_test, t_test)
    rejected_order = rejected[np.argsort(-uncertainty[rejected], kind='stable')]
    selected = rejected_order[:target].tolist()
    if len(selected) < target:
        remainder = np.setdiff1d(available, np.asarray(selected, dtype=int), assume_unique=False)
        remainder = remainder[np.argsort(-uncertainty[remainder], kind='stable')]
        selected.extend(remainder[:target - len(selected)].tolist())
    selected = np.asarray(selected, dtype=int)
    if selected.size != target or np.unique(selected).size != selected.size:
        raise RuntimeError(f'IRAAL chọn {selected.size} mẫu, kỳ vọng {target}.')
    return selected, {
        **base, "budget_semantics": "fixed_label_budget",
        "candidate_pool_size": int(available.size),
        "n_selected": int(selected.size),
        "selection_seconds": time.perf_counter() - selection_started,
        "mean_uncertainty": (float(np.mean(uncertainty[selected]))
                             if selected.size else float('nan')),
    }

def _patched_fprsu(clf, X_train, X_tests,
                   y_train, y_tests, t_train, t_tests,
                   fit_function=None, predict_function=None,
                   rebalancers=(), rejectors=(), selectors=(),
                   reject_label=None, select_reject_labeled=False, clf_uses_t=True,
                   posthoc_augmented_AL=0):

    fit_function = clf.fit if fit_function is None else fit_function
    predict_function = (tess_utils.select_prediction_function(clf, labels_only=True)
                        if predict_function is None else predict_function)

    for stage in tuple(rebalancers) + tuple(rejectors) + tuple(selectors):
        stage.resolve_schedule(len(X_tests))

    results = {}
    selected_indexes = None
    bhr_buffer = None
    if getattr(clf, '_balanced_hard_replay', False):
        bhr_buffer = BalancedHardReplayBuffer(
            memory_cap=int(clf._training_history_cap),
            fn_fraction=clf._bhr_fn_fraction,
            fp_fraction=clf._bhr_fp_fraction,
            fn_max_repeat=clf._bhr_fn_max_repeat,
            fp_max_repeat=clf._bhr_fp_max_repeat,
            seed=clf._strategy_seed)
        bhr_buffer.initialize(X_train, y_train, t_train)

    results['rejected_indexes_all'] = []
    results['selected_indexes_all'] = []
    results['audit_indexes_all'] = []
    results['fn_penalty_used'] = []
    results['fn_penalty_next'] = []
    results['afnp_target_fnr'] = []
    results['afnp_estimated_fnr'] = []
    results['audit_positive_count'] = []
    results['audit_fn_count'] = []
    results['audit_count'] = []
    results['bhr_telemetry'] = []

    results['rejected_y_true'] = []
    results['rejected_malware_count'] = []
    results['rejected_benign_count'] = []
    results['selected_y_true'] = []
    results['selected_malware_count'] = []
    results['selected_benign_count'] = []
    results['policy_action_all'] = []
    results['binary_y_true_all'] = []
    results['binary_y_preds_all'] = []
    results['binary_t_all'] = []
    results['selection_diagnostics'] = []

    global CURRENT_PERIOD_IDX
    for i, (X_test, y_test, t_test) in tqdm(
            enumerate(zip(X_tests, y_tests, t_tests)), total=len(X_tests),
            desc="Observed test periods"):
        CURRENT_PERIOD_IDX = i

        for rebalancer in rebalancers:
            if not rebalancer.schedule[i]:
                continue
            X_train, y_train, t_train = rebalancer.alter(
                clf, X_train, y_train, t_train, X_test, y_test, t_test)

        results = tess_metrics.get_train_info(
            X_train, y_train, t_train, existing=results)

        if ((selected_indexes is not None and len(selected_indexes) > 0)
                or i == 0):
            fit_function(X_train, y_train, t_train) if clf_uses_t else fit_function(X_train, y_train)

        kept_indexes, rejected_indexes, selected_indexes = None, None, None
        y_pred = predict_function(X_test, t_test) if clf_uses_t else predict_function(X_test)
        policy_action = np.asarray(y_pred, dtype=int).copy()
        results['policy_action_all'].append(policy_action.tolist())
        binary_pred = clf.predict_binary_all(X_test, t_test)
        results['binary_y_true_all'].append(np.asarray(y_test, dtype=int).tolist())
        results['binary_y_preds_all'].append(np.asarray(binary_pred, dtype=int).tolist())
        results['binary_t_all'].append(np.asarray(t_test).astype(str).tolist())


        # Mẫu kiểm toán được lấy đều mà không đọc nhãn. Phần còn lại của B mới
        # được giao cho bộ chọn bất định , do đó tổng số nhãn không vượt B.
        audit_indexes = np.array([], dtype=int)
        total_feedback_budget = min(
            int(getattr(clf, '_feedback_budget', 0)), int(len(y_test)))
        active_feedback_budget = total_feedback_budget
        if (getattr(clf, 'afnp_audit_sampling', False)
                and total_feedback_budget > 0
                and not getattr(clf, '_full_feedback', False)):
            audit_count = min(
                total_feedback_budget,
                max(1, int(np.ceil(np.sqrt(total_feedback_budget)))))
            audit_rng = np.random.default_rng(
                int(clf.afnp_seed) + 1000003 * int(i))
            audit_indexes = np.sort(audit_rng.choice(
                len(y_test), size=audit_count, replace=False).astype(int))
            active_feedback_budget = total_feedback_budget - audit_count

        selection_diagnostic = {
            'selector_mode': str(getattr(clf, '_feedback_selector_mode', 'none')),
            'budget': int(getattr(clf, '_feedback_budget', 0)),
            'n_rows': int(len(y_test)), 'n_rejected': 0,
            'candidate_pool_size': 0, 'n_selected': 0}
        if reject_label is not None:
            y_pred_array = np.asarray(y_pred)
            rejected_indexes = np.flatnonzero(y_pred_array == reject_label).astype(int)
            kept_indexes = np.flatnonzero(y_pred_array != reject_label).astype(int)
            if rejected_indexes.size:
                y_pred_array[rejected_indexes] = np.asarray(
                    clf.predict_reject_alt(
                        X_test[rejected_indexes], t_test[rejected_indexes]),
                    dtype=int)
                y_pred = y_pred_array.tolist()
            if select_reject_labeled:
                active_indexes, selection_diagnostic = _select_feedback_indexes(
                    clf, X_test, t_test, rejected_indexes,
                    budget_override=active_feedback_budget,
                    excluded_indexes=audit_indexes)
                selected_indexes = np.concatenate(
                    (audit_indexes, np.asarray(active_indexes, dtype=int)))
                if np.unique(selected_indexes).size != selected_indexes.size:
                    raise RuntimeError('Mẫu kiểm toán và mẫu chủ động bị trùng.')
                selection_diagnostic.update({
                    'audit_count': int(audit_indexes.size),
                    'active_budget': int(active_feedback_budget),
                    'total_feedback_budget': int(total_feedback_budget)})

        for rejector in rejectors:
            if not rejector.schedule[i]:
                continue
            kept_indexes, rejected_indexes = rejector.reject_wrapper(
                clf, X_train, y_train, t_train,
                X_test, y_test, t_test,
                kept_indexes, rejected_indexes)

        for selector in selectors:
            if not selector.schedule[i]:
                continue
            selected_indexes = selector.query_wrapper(
                clf, X_train, y_train, t_train,
                X_test, y_test, t_test, selected_indexes)

        # Dùng dự đoán nhị phân đã khóa để ước lượng FNR trên mẫu kiểm toán.
        # Nhãn chu kỳ t chỉ cập nhật hệ số dùng từ lần fit ở chu kỳ t+1.
        lambda_used = float(getattr(clf, 'fn_penalty', 1.0))
        audit_positive = int(np.sum(
            np.asarray(y_test)[audit_indexes] == 1)) if audit_indexes.size else 0
        audit_fn = int(np.sum(
            (np.asarray(y_test)[audit_indexes] == 1)
            & (np.asarray(binary_pred)[audit_indexes] == 0))) if audit_indexes.size else 0
        estimated_fnr = float('nan')
        if getattr(clf, 'adaptive_fn_penalty', False):
            clf.afnp_audit_history.append((audit_fn, audit_positive))
            total_fn = int(sum(item[0] for item in clf.afnp_audit_history))
            total_positive = int(sum(item[1] for item in clf.afnp_audit_history))
            minimum_support = max(
                2, int(np.ceil(np.sqrt(max(1, clf.afnp_budget)))))
            if clf.afnp_target_fnr is None and total_positive >= minimum_support:
                clf.afnp_target_fnr = (total_fn + 0.5) / (total_positive + 1.0)
            elif clf.afnp_target_fnr is not None:
                recent = clf.afnp_audit_history[-max(1, int(clf.afnp_window)):]
                recent_fn = int(sum(item[0] for item in recent))
                recent_positive = int(sum(item[1] for item in recent))
                if recent_positive > 0:
                    estimated_fnr = (recent_fn + 0.5) / (recent_positive + 1.0)
                    clf.afnp_controller_step += 1
                    step_size = 1.0 / np.sqrt(float(clf.afnp_controller_step))
                    clf.afnp_dual = float(max(
                        0.0,
                        clf.afnp_dual + step_size * (
                            estimated_fnr - clf.afnp_target_fnr)))
                    clf.fn_penalty = 1.0 + clf.afnp_dual

        results['audit_indexes_all'].append(audit_indexes.tolist())
        results['fn_penalty_used'].append(lambda_used)
        results['fn_penalty_next'].append(float(getattr(clf, 'fn_penalty', 1.0)))
        results['afnp_target_fnr'].append(
            float(clf.afnp_target_fnr)
            if clf.afnp_target_fnr is not None else float('nan'))
        results['afnp_estimated_fnr'].append(float(estimated_fnr))
        results['audit_positive_count'].append(audit_positive)
        results['audit_fn_count'].append(audit_fn)
        results['audit_count'].append(int(audit_indexes.size))

        # Dự đoán tháng t hoàn tất trước khi nhãn tháng t được bổ sung.
        # Nhãn này chỉ tác động từ lần fit ở tháng t+1.
        if getattr(clf, '_full_feedback', False):
            selected_indexes = np.arange(len(y_test), dtype=int)
            selection_diagnostic.update({
                'selector_mode': 'full_feedback',
                'budget': int(len(y_test)),
                'candidate_pool_size': int(len(y_test)),
                'n_selected': int(len(y_test))})

        if rejected_indexes is not None and len(rejected_indexes) > 0:
            rej_y_true = y_test[rejected_indexes]
            results['rejected_y_true'].append(rej_y_true.tolist())
            results['rejected_malware_count'].append(int(np.sum(rej_y_true == 1)))
            results['rejected_benign_count'].append(int(np.sum(rej_y_true == 0)))
        else:
            results['rejected_y_true'].append([])
            results['rejected_malware_count'].append(0)
            results['rejected_benign_count'].append(0)

        if selected_indexes is not None and selected_indexes.shape[0] > 0:
            sel_y_true = y_test[selected_indexes]
            results['selected_y_true'].append(sel_y_true.tolist())
            results['selected_malware_count'].append(int(np.sum(sel_y_true == 1)))
            results['selected_benign_count'].append(int(np.sum(sel_y_true == 0)))

            X_selected = X_test[selected_indexes]
            y_selected = y_test[selected_indexes]
            t_selected = t_test[selected_indexes]
            if bhr_buffer is not None:
                bhr_buffer.observe(
                    X_selected, y_selected,
                    np.asarray(binary_pred)[selected_indexes], t_selected,
                    source_period=i)
            else:
                X_train = scipy.sparse.vstack((X_train, X_selected))
                y_train = np.hstack((y_train, y_selected))
                t_train = np.hstack((t_train, t_selected))
                X_train, y_train, t_train = _apply_training_memory(
                    clf, X_train, y_train, t_train)
            results['selected'].append(selected_indexes.size)
        else:
            results['selected_y_true'].append([])
            results['selected_malware_count'].append(0)
            results['selected_benign_count'].append(0)
            results['selected'].append(0)

        if bhr_buffer is not None:
            bhr_batch = bhr_buffer.build(fit_period=i + 1)
            X_train, y_train, t_train = bhr_batch.X, bhr_batch.y, bhr_batch.t
            bhr_telemetry = bhr_batch.telemetry.to_dict()
            bhr_telemetry['enabled'] = True
        else:
            # Sau dự đoán đầu tiên, chuẩn hóa bộ nhớ cho mọi nhánh không BHR,
            # kể cả IRAL ở tháng không phát sinh hành động từ chối. Lần fit
            # khởi tạo phía trên vẫn dùng toàn bộ cửa sổ huấn luyện ban đầu.
            X_train, y_train, t_train = _apply_training_memory(
                clf, X_train, y_train, t_train)
            bhr_telemetry = {
                'enabled': False, 'fit_period': int(i + 1),
                'memory_rows': int(X_train.shape[0]),
                'causal_lag_ok': True,
            }
        results['bhr_telemetry'].append(bhr_telemetry)

        selection_diagnostic.update({
            'n_selected': int(0 if selected_indexes is None else len(selected_indexes)),
            'n_rejected_total': int(
                0 if rejected_indexes is None else len(rejected_indexes)),
            'selected_malware': int(results['selected_malware_count'][-1]),
            'selected_benign': int(results['selected_benign_count'][-1]),
            'rejected_malware': int(results['rejected_malware_count'][-1]),
            'rejected_benign': int(results['rejected_benign_count'][-1]),
            'selection_used_labels': False,
        })
        results['selection_diagnostics'].append(selection_diagnostic)
        clf._selection_history.append({
            'period_index': int(i), **selection_diagnostic})

        results['rejected_indexes_all'].append(
            rejected_indexes.tolist() if rejected_indexes is not None else [])
        results['selected_indexes_all'].append(
            selected_indexes.tolist() if selected_indexes is not None else [])

        if rejected_indexes is not None and len(rejected_indexes) > 0:
            y_pred_array = np.asarray(y_pred)
            rejected_true = y_test[rejected_indexes]
            rejected_pred = y_pred_array[rejected_indexes]
            _, rejected_fp, rejected_fn, rejected_tp = skmetrics.confusion_matrix(
                rejected_true, rejected_pred, labels=[0, 1]).ravel()
            rejected_f1_denominator = 2 * rejected_tp + rejected_fp + rejected_fn
            results['f1_r'].append(
                2 * rejected_tp / rejected_f1_denominator
                if rejected_f1_denominator else float('nan'))
            results['rejected'].append(len(rejected_indexes))
            if kept_indexes is not None and kept_indexes.shape[0] > 0:
                y_test = y_test[kept_indexes]
                y_pred = y_pred_array[kept_indexes]
                t_test = t_test[kept_indexes]
                results = tess_metrics.calculate_metrics(
                    y_test, y_pred, existing=results)
            else:
                y_test = y_test[:0]
                y_pred = y_pred_array[:0]
                t_test = t_test[:0]
                for key in ('tp', 'fp', 'tn', 'fn', 'p', 'n', 'tot'):
                    results.setdefault(key, []).append(0)
                for cumulative, base in (
                    ('tp_cumu', 'tp'), ('fp_cumu', 'fp'), ('tn_cumu', 'tn'),
                    ('fn_cumu', 'fn'), ('p_cumu', 'p'), ('n_cumu', 'n'),
                    ('tot_cumu', 'tot')):
                    results.setdefault(cumulative, []).append(sum(results[base]))
                for key in (
                    'tpr', 'fnr', 'fpr', 'tnr', 'precision', 'recall',
                    'f1', 'precision_n', 'recall_n', 'f1_n'):
                    results.setdefault(key, []).append(float('nan'))
        else:
            results['rejected'].append(0)
            results['f1_r'].append(float('nan'))
            results = tess_metrics.calculate_metrics(
                y_test, y_pred, existing=results)

        if 'y_preds' not in results:
            results['y_tests'] = [y_test]
            results['y_preds'] = [y_pred]
            results['t_tests'] = [t_test]
        else:
            results['y_tests'].append(y_test)
            results['y_preds'].append(y_pred)
            results['t_tests'].append(t_test)

    return results

clf_utils.fit_predict_reject_sample_update = _patched_fprsu
print("✅ Runtime V13: IRAL/IRAAL, DRMD-FN và Balanced Hard Replay nhân quả")


# ══════════════════════════════════════════════════════════════════════════════
# V13 — suy luận theo lô cho actor ba hành động
# ══════════════════════════════════════════════════════════════════════════════
# Gán trực tiếp vào namespace của DRMD.base để lần chạy lại cell trong cùng
# kernel không giữ tham chiếu tới hàm cũ đã import bằng `from ... import ...`.
import DRMD.base as _drmd_base_runtime
_drmd_base_runtime.fit_predict_reject_sample_update = _patched_fprsu

# Nhánh hai hành động giữ nguyên đường suy luận gốc; ma trận V13 chỉ dùng
# actor ba hành động và khóa kích thước lô trong chữ ký triển khai.
_original_drmd_test = drmd_env.DRMD._DRMD__test

def _observation_chunk(model_env, X_chunk, t_chunk):
    dense = X_chunk.toarray() if sp.issparse(X_chunk) else np.asarray(X_chunk)
    obs = torch.from_numpy(np.ascontiguousarray(dense, dtype=np.float32)).to(model_env.device)
    if model_env.temporal_feature:
        time_column = drmd_env.convert_time(t_chunk, model_env.device) - model_env.first_date
        obs = torch.cat((time_column.unsqueeze(1), obs), dim=1)
    return obs

def _batched_drmd_test(self, X, t):
    if not self.is_reject_action:
        return _original_drmd_test(self, X, t)
    if self.validation_episodes > 0:
        self._DRMD__load(opt=self.val_extension)
    self.model.agent.eval()
    actions, logprobs, entropies, values = [], [], [], []
    chunk_rows = max(1, int(BATCH_PREDICT_ROWS))
    for start in range(0, X.shape[0], chunk_rows):
        stop = min(start + chunk_rows, X.shape[0])
        obs = _observation_chunk(self, X[start:stop], t[start:stop])
        action, logprob, entropy, value = self.model.step(obs=obs)
        actions.append(action.detach().cpu())
        logprobs.append(logprob.detach().cpu())
        entropies.append(entropy.detach().cpu())
        values.append(value.detach().cpu().reshape(-1))
        del obs, action, logprob, entropy, value
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    action = torch.cat(actions)
    logprob = torch.cat(logprobs)
    entropy = torch.cat(entropies)
    value = torch.cat(values)
    # Giữ đúng ngữ nghĩa mã gốc: max entropy được lấy trên TOÀN chu kỳ.
    return (action.tolist(), torch.exp(logprob).tolist(),
            (entropy.max() - entropy).tolist(), value.tolist())

def _batched_predict_reject_alt(self, X_test, t_test):
    self.model.agent.eval()
    predictions = []
    chunk_rows = max(1, int(BATCH_PREDICT_ROWS))
    with torch.no_grad():
        for start in range(0, X_test.shape[0], chunk_rows):
            stop = min(start + chunk_rows, X_test.shape[0])
            obs = _observation_chunk(self, X_test[start:stop], t_test[start:stop])
            time_column = drmd_env.convert_time(t_test[start:stop], self.device)
            reject_action = torch.ones_like(time_column) * self.reject_action
            action = self.model.agent.get_next_likely_action(obs=obs, action=reject_action)
            predictions.extend(action.detach().cpu().reshape(-1).tolist())
            del obs, time_column, reject_action, action
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return predictions

drmd_env.DRMD._DRMD__test = _batched_drmd_test
drmd_env.DRMD.predict_reject_alt = _batched_predict_reject_alt


def _batched_predict_binary_all(self, X_test, t_test):
    self.model.agent.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, X_test.shape[0], max(1, int(BATCH_PREDICT_ROWS))):
            stop = min(start + max(1, int(BATCH_PREDICT_ROWS)), X_test.shape[0])
            obs = _observation_chunk(self, X_test[start:stop], t_test[start:stop])
            logits = self.model.agent.actor(obs)
            predictions.extend(torch.argmax(logits[:, :2], dim=1).cpu().tolist())
            del obs, logits
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return predictions

drmd_env.DRMD.predict_binary_all = _batched_predict_binary_all

# ══════════════════════════════════════════════════════════════════════════════
# V13 — ngân sách phản hồi và bộ nhớ Balanced Hard Replay
# ══════════════════════════════════════════════════════════════════════════════
import hashlib as _artifact_hashlib
import torch.nn.functional as _torch_F

FIXED_TEMPORAL_REFERENCE_POLICY = "fixed_initial_training_window_v1"

# DRMD gốc ghép toàn bộ siêu tham số vào ``clf.fpath``. Khi thư mục đích đã có
# chữ ký, tên sidecar ``.p.meta.json`` có thể vượt giới hạn 255 byte của ext4.
# Tên ngắn dưới đây chỉ là khóa lưu trữ; cấu hình đầy đủ vẫn nằm trong sidecar.
ARTIFACT_STEM_PREFIX = "drmd-r"
ARTIFACT_FILENAME_MAX_BYTES = 120

def compact_artifact_stem(legacy_stem):
    digest = _artifact_hashlib.sha256(
        str(legacy_stem).encode("utf-8")).hexdigest()[:20]
    stem = f"{ARTIFACT_STEM_PREFIX}-{digest}"
    if len((stem + ".p.meta.json").encode("utf-8")) > ARTIFACT_FILENAME_MAX_BYTES:
        raise OSError("Tên artifact rút gọn vẫn vượt ngưỡng an toàn.")
    return stem

_init_before_compact_artifact_name = drmd_env.DRMD.__init__

def _init_with_compact_artifact_name(self, settings, *args, **kwargs):
    _init_before_compact_artifact_name(self, settings, *args, **kwargs)
    self.legacy_fpath = self.fpath
    self.fpath = compact_artifact_stem(self.legacy_fpath)

drmd_env.DRMD.__init__ = _init_with_compact_artifact_name
_original_training_data_setup = drmd_env.DRMD._DRMD__training_data_setup
_original_drmd_fit = drmd_env.DRMD.fit

def _training_data_setup_with_common_temporal_axis(self, X_train, y_train, t_train):
    """Khóa chuẩn phần thưởng thời gian theo cửa sổ huấn luyện ban đầu.

    Mã DRMD gốc tính lại ``start_date`` từ mẫu nhỏ tuổi nhất còn trong bộ nhớ.
    Khi thành phần bộ nhớ thay đổi theo phản hồi, cách đó làm thay đổi đồng thời
    dữ liệu và chuẩn hóa phần thưởng. Giao thức V13 cố định gốc và mẫu số theo cửa sổ
    huấn luyện ban đầu, đúng ý đồ trọng số thời gian tuyệt đối tăng tuyến tính
    của DRMD. Quy tắc không đọc tháng kiểm thử tương lai và dùng chung cho mọi
    biến thể.
    """
    # DRMD gốc đọc trực tiếp thuộc tính .year/.month của từng mốc.
    # BHR lưu thời gian chuẩn numpy.datetime64 để kiểm toán và vì vậy
    # phải đổi giao diện sang datetime.datetime trước khi gọi mã gốc.
    # Phép đổi chỉ thay kiểu biểu diễn, không đổi tháng hay thứ tự mẫu.
    _time_array = np.asarray(t_train).reshape(-1)
    if np.issubdtype(_time_array.dtype, np.datetime64):
        t_train = _time_array.astype('datetime64[us]').astype(object)
    elif not all(hasattr(value, 'year') and hasattr(value, 'month')
                 for value in _time_array):
        raise TypeError('t_train không cung cấp thuộc tính year/month.')
    X_fit, y_fit, t_fit = _original_training_data_setup(
        self, X_train, y_train, t_train)
    current_start = int(self.start_date)
    current_end = int(self.end_date)
    current_period = int(self.training_period)
    if current_period <= 0:
        raise ValueError("Khoảng thời gian của bộ nhớ hiện hành không dương.")
    self._current_memory_start = current_start
    self._current_memory_end = current_end
    self._current_memory_period = current_period

    if self._temporal_reference_start is None:
        self._temporal_reference_start = current_start
        self._temporal_reference_end = current_end
        self._temporal_reference_period = current_period
        if int(self.first_date) != current_start:
            raise ValueError("first_date không khớp cửa sổ huấn luyện ban đầu.")
        if current_period != TRAINING_WINDOW:
            raise ValueError(
                f"Chuẩn thời gian ban đầu {current_period}, kỳ vọng {TRAINING_WINDOW} tháng.")

    self.start_date = int(self._temporal_reference_start)
    self.end_date = int(self._temporal_reference_end)
    self.training_period = int(self._temporal_reference_period)
    times = np.asarray(t_fit)
    month_ids = times.astype('datetime64[M]').astype(np.int64)
    years = times.astype('datetime64[Y]').astype(np.int64) + 1970
    months = month_ids - times.astype('datetime64[Y]').astype('datetime64[M]').astype(np.int64) + 1
    absolute_month = years * 12 + months
    factors = (absolute_month - self.start_date + 1) / self.training_period * self.temporal_scaling
    if not np.isfinite(factors).all():
        raise FloatingPointError("Hệ số phần thưởng thời gian không hữu hạn.")
    self._temporal_reference_history.append({
        "fit_index": int(self._fit_calls),
        "reference_start": int(self.start_date),
        "reference_end": int(self.end_date),
        "reference_period": int(self.training_period),
        "memory_start": current_start, "memory_end": current_end,
        "memory_period": current_period,
        "factor_min": float(np.min(factors)),
        "factor_max": float(np.max(factors)),
        "factor_mean": float(np.mean(factors)),
    })
    return X_fit, y_fit, t_fit

drmd_env.DRMD._DRMD__training_data_setup = (
    _training_data_setup_with_common_temporal_axis)


def _patched_drmd_fit(self, X_train, y_train, t_train):
    _original_drmd_fit(self, X_train, y_train, t_train)
    for parameter_name, parameter in self.model.agent.named_parameters():
        if not torch.isfinite(parameter).all():
            raise FloatingPointError(
                f"Tham số không hữu hạn sau fit {self._fit_calls}: {parameter_name}")
    self._fit_calls += 1

drmd_env.DRMD.fit = _patched_drmd_fit

# `references/DRMD/DRMD/base.py` trong workspace có một cache-check cục bộ làm
# khởi tạo DRMD hai lần. Staging của notebook đã xử lý cache; hàm chuẩn dưới đây
# khớp luồng gốc nhưng chỉ tạo classifier đúng một lần.
def _canonical_run(settings, feature_indexes=None):
    X_all, y_all, t_all = _drmd_base_runtime.load_data(settings)
    if feature_indexes is not None:
        X_all = X_all[:, feature_indexes]
        settings = deepcopy(settings)
        settings.fs_size = int(feature_indexes.sum().item())

    splits = _drmd_base_runtime.temporal.time_aware_train_test_split(
        X=X_all, y=y_all, t=t_all,
        train_size=settings.training_window,
        test_size=settings.testing_window,
        granularity=settings.granularity)
    if settings.virtual_months:
        splits = _drmd_base_runtime.convert_virtual_months(
            splits, X_all, y_all, t_all)

    clf = drmd_env.DRMD(settings)
    print(f"\n{clf.fpath}\n")
    if settings.is_reject_action:
        if (settings.is_active and settings.al_rate > 0
                and not settings.select_reject_labeled):
            selector = UncertaintySelector(settings.al_rate, clf_uses_t=True)
            results = _patched_fprsu(
                clf, *splits, reject_label=settings.reject_action_id,
                select_reject_labeled=False, selectors=[selector])
        else:
            results = _patched_fprsu(
                clf, *splits, reject_label=settings.reject_action_id,
                select_reject_labeled=settings.select_reject_labeled,
                posthoc_augmented_AL=settings.posthoc_augmented_AL)
    elif (settings.is_reject and settings.reject_rate > 0
          and settings.is_active and settings.al_rate > 0):
        rejector = UncertaintyRejector(settings.reject_rate, clf_uses_t=True)
        selector = UncertaintySelector(settings.al_rate, clf_uses_t=True)
        results = _patched_fprsu(
            clf, *splits, selectors=[selector], rejectors=[rejector])
    elif settings.is_reject and settings.reject_rate > 0:
        rejector = UncertaintyRejector(settings.reject_rate, clf_uses_t=True)
        results = _patched_fprsu(clf, *splits, rejectors=[rejector])
    elif settings.is_active and settings.al_rate > 0:
        selector = UncertaintySelector(settings.al_rate, clf_uses_t=True)
        results = _patched_fprsu(clf, *splits, selectors=[selector])
    else:
        results = _patched_fprsu(clf, *splits)

    results["selection_history"] = list(clf._selection_history)
    results["temporal_reference_history"] = list(clf._temporal_reference_history)
    results["strategy_runtime_config"] = {
        "feedback_selector_mode": str(clf._feedback_selector_mode),
        "feedback_budget": int(clf._feedback_budget),
        "candidate_multiplier": int(clf._candidate_multiplier),
        "adaptive_fn_penalty": bool(clf.adaptive_fn_penalty),
        "audit_sampling": bool(clf.afnp_audit_sampling),
        "balanced_hard_replay": bool(clf._balanced_hard_replay),
        "bhr_fn_fraction": float(clf._bhr_fn_fraction),
        "bhr_fp_fraction": float(clf._bhr_fp_fraction),
        "bhr_fn_max_repeat": int(clf._bhr_fn_max_repeat),
        "bhr_fp_max_repeat": int(clf._bhr_fp_max_repeat),
        "full_feedback": bool(clf._full_feedback),
        "bhr_error_partition_prediction": "binary_argmax_head",
        "policy_action_space": [0, 1, 2],
        "reject_action_index": int(clf.reject_action),
        "training_history_cap": clf._training_history_cap,
        "temporal_reference_policy": clf._temporal_reference_policy,
        "temporal_reference_start": int(clf._temporal_reference_start),
        "temporal_reference_end": int(clf._temporal_reference_end),
        "temporal_reference_period": int(clf._temporal_reference_period),
        "feedback_timing": "predict_t_then_fit_t_plus_1",
        "single_classifier_initialization": True,
    }
    save_path = Path(settings.results_save_location) / f"{clf.fpath}.p"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "wb") as handle:
        pickle.dump(results, handle)
    try:
        _drmd_base_runtime.metrics.print_metrics(results)
        print("AUT theo F1 (%):", 100.0 * _drmd_base_runtime.metrics.aut(results, "f1"))
    except Exception as exc:
        print("Artifact đã lưu; bỏ qua lỗi hiển thị độ đo:", repr(exc))
    return results

_drmd_base_runtime.run = _canonical_run

# Chữ ký sidecar phải thay đổi khi chính mã monkey-patch trong notebook thay
# đổi, ngay cả khi checkout DRMD/Tesseract và tên giao thức chưa đổi. Không băm
# trực tiếp code object bằng marshal vì co_filename của IPython đổi theo phiên.
import types as _types
import hashlib as _hashlib

def _stable_code_descriptor(code):
    def normalize(value):
        if isinstance(value, _types.CodeType):
            return _stable_code_descriptor(value)
        if isinstance(value, bytes):
            return {"bytes_hex": value.hex()}
        if isinstance(value, tuple):
            return [normalize(item) for item in value]
        if isinstance(value, list):
            return {"list": [normalize(item) for item in value]}
        if isinstance(value, (set, frozenset)):
            # ``repr(frozenset)`` phụ thuộc PYTHONHASHSEED và đã làm chữ ký
            # Schema cũ thay đổi giữa hai phiên Kaggle dù mã không đổi.
            items = [normalize(item) for item in value]
            items.sort(key=lambda item: json.dumps(
                item, ensure_ascii=False, sort_keys=True,
                separators=(",", ":"), default=str))
            return {"set_type": type(value).__name__, "items": items}
        if isinstance(value, dict):
            items = [(normalize(key), normalize(item))
                     for key, item in value.items()]
            items.sort(key=lambda pair: json.dumps(
                pair[0], ensure_ascii=False, sort_keys=True,
                separators=(",", ":"), default=str))
            return {"dict_items": items}
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return {"type": type(value).__name__, "repr": repr(value)}

    return {
        "argcount": code.co_argcount,
        "posonlyargcount": code.co_posonlyargcount,
        "kwonlyargcount": code.co_kwonlyargcount,
        "code_hex": code.co_code.hex(),
        "consts": normalize(code.co_consts),
        "names": list(code.co_names),
        "varnames": list(code.co_varnames),
        "freevars": list(code.co_freevars),
        "cellvars": list(code.co_cellvars),
    }

_runtime_patch_components = (
    _patched_init, _patched_reward_function, _patched_fprsu,
    _deterministic_recency_cap, _apply_training_memory,
    _observation_chunk, _batched_drmd_test, _batched_predict_reject_alt,
    _actor_uncertainty, _select_feedback_indexes, _patched_drmd_fit,
    _training_data_setup_with_common_temporal_axis,
    _batched_predict_binary_all, _canonical_run,
)
PATCH_IMPLEMENTATION_FINGERPRINT = _hashlib.sha256(
    json.dumps(
        [_stable_code_descriptor(function.__code__)
         for function in _runtime_patch_components],
        ensure_ascii=False, sort_keys=True, separators=(",", ":"),
    ).encode()
).hexdigest()
print("✅ Runtime patch V13: IRAL/IRAAL, DRMD-FN và Balanced Hard Replay")
print("Runtime patch fingerprint:", PATCH_IMPLEMENTATION_FINGERPRINT)

## Ô 7 — Nạp đúng bộ dữ liệu LAMDA đã khóa

Notebook chỉ chấp nhận bộ dữ liệu LAMDA khớp định danh kho, revision, lược đồ và dấu vân tay nguồn. Không quét rồi ghép tùy ý mọi tệp Parquet trong thư mục đầu vào Kaggle.


In [ ]:
import hashlib
import pyarrow.parquet as pq

EXPECTED_LAMDA_RELATIVE_FILES = {
    f"{year}/{year}_{split}.parquet"
    for year in DATASET_YEARS for split in ("train", "test")
}

def baseline_candidates(search_roots):
    candidates = set()
    for root in search_roots:
        if not root.exists():
            continue
        if root.is_dir() and root.name.lower() == "baseline":
            candidates.add(root.resolve())
        for pattern in ("Baseline", "baseline"):
            candidates.update(path.resolve() for path in root.rglob(pattern) if path.is_dir())
    return sorted(candidates, key=str)

def matching_lamda_files(root):
    files = []
    for path in root.rglob("*.parquet"):
        try:
            relative = path.relative_to(root).as_posix()
        except ValueError:
            continue
        if relative in EXPECTED_LAMDA_RELATIVE_FILES:
            files.append(path)
    return sorted(files, key=lambda path: path.relative_to(root).as_posix())

search_roots = [Path("/kaggle/input"), Path("/kaggle/working/LAMDA_FULL")]
if LAMDA_ROOT_OVERRIDE is not None:
    valid_roots = [Path(LAMDA_ROOT_OVERRIDE)]
else:
    valid_roots = [
        root for root in baseline_candidates(search_roots)
        if {path.relative_to(root).as_posix() for path in matching_lamda_files(root)}
           == EXPECTED_LAMDA_RELATIVE_FILES
    ]

if len(valid_roots) != 1:
    details = {
        str(root): len(matching_lamda_files(root))
        for root in baseline_candidates(search_roots)
    }
    raise RuntimeError(
        "Phải xác định đúng một nguồn LAMDA/Baseline có đủ "
        f"{len(EXPECTED_LAMDA_RELATIVE_FILES)} tệp. "
        f"Tìm thấy {len(valid_roots)} nguồn hợp lệ: {details}. "
        "Đặt LAMDA_ROOT_OVERRIDE tới thư mục Baseline cần dùng."
    )

LAMDA_BASELINE_ROOT = valid_roots[0]
lamda_parquets = matching_lamda_files(LAMDA_BASELINE_ROOT)
assert len(lamda_parquets) == 2 * len(DATASET_YEARS)

def source_file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

source_files = []
for path in lamda_parquets:
    parquet_file = pq.ParquetFile(path)
    source_files.append({
        "relative_path": path.relative_to(LAMDA_BASELINE_ROOT).as_posix(),
        "bytes": int(path.stat().st_size),
        "sha256": source_file_sha256(path),
        "rows": int(parquet_file.metadata.num_rows),
        "columns": list(parquet_file.schema_arrow.names),
        "schema": [
            {"name": field.name, "type": str(field.type)}
            for field in parquet_file.schema_arrow
        ],
    })
observed_lamda_file_sha256 = {
    item["relative_path"]: item["sha256"] for item in source_files
}
if observed_lamda_file_sha256 != EXPECTED_LAMDA_FILE_SHA256:
    all_paths = sorted(
        set(observed_lamda_file_sha256) | set(EXPECTED_LAMDA_FILE_SHA256))
    mismatches = {
        relative_path: {
            "observed": observed_lamda_file_sha256.get(relative_path),
            "expected": EXPECTED_LAMDA_FILE_SHA256.get(relative_path),
        }
        for relative_path in all_paths
        if observed_lamda_file_sha256.get(relative_path)
        != EXPECTED_LAMDA_FILE_SHA256.get(relative_path)
    }
    raise ValueError(
        "LAMDA Baseline không khớp revision đã khóa "
        f"{LAMDA_HF_REVISION}: {json.dumps(mismatches, ensure_ascii=False)}")

source_descriptor = {
    "hf_repo_id": LAMDA_HF_REPO_ID,
    "hf_revision": LAMDA_HF_REVISION,
    "files": source_files,
}
fingerprint_payload = json.dumps(
    source_descriptor, ensure_ascii=False, sort_keys=True).encode()
DATASET_SOURCE_FINGERPRINT = hashlib.sha256(fingerprint_payload).hexdigest()
(TABLE_OUT / "lamda_source_manifest.json").write_text(
    json.dumps({"root": str(LAMDA_BASELINE_ROOT),
                "hf_repo_id": LAMDA_HF_REPO_ID,
                "hf_revision": LAMDA_HF_REVISION,
                "revision_verified_by_sha256": True,
                "source_fingerprint_sha256": DATASET_SOURCE_FINGERPRINT,
                "files": source_files}, indent=2, ensure_ascii=False),
    encoding="utf-8")

print("LAMDA Baseline root:", LAMDA_BASELINE_ROOT)
print("Selected parquet files:", len(lamda_parquets))
print("Source fingerprint:", DATASET_SOURCE_FINGERPRINT)
for path in lamda_parquets:
    print(" ", path.relative_to(LAMDA_BASELINE_ROOT))

In [ ]:
META_COLS = {
    "hash", "sha256", "label", "family", "vt_count",
    "year_month", "year", "month"
}

def parse_year_month(value):
    s = str(value)
    if len(s) >= 7:
        return datetime.strptime(s[:7] + "-01", "%Y-%m-%d")
    if len(s) == 4:
        return datetime.strptime(s + "-01-01", "%Y-%m-%d")
    return datetime.fromisoformat(s)

def label_to_int(v):
    if isinstance(v, (int, np.integer, float, np.floating)):
        value = int(v)
        if float(v) == value and value in (0, 1):
            return value
        raise ValueError(f"Unsupported numeric label: {v!r}")
    s = str(v).strip().lower()
    if s in {"1", "true", "malware", "malicious"}:
        return 1
    if s in {"0", "false", "benign", "goodware"}:
        return 0
    raise ValueError(f"Unsupported string label: {v!r}")

def feature_columns_from_schema(names):
    feat_cols = [c for c in names if c.lower().startswith("feat_")]
    if feat_cols:
        return feat_cols
    return [c for c in names if c.lower() not in META_COLS]

def load_all_rows_metadata(parquet_files):
    meta_rows = []
    feature_cols_ref = None

    for path in sorted(parquet_files):
        names = list(pq.ParquetFile(path).schema_arrow.names)
        lower_map = {c.lower(): c for c in names}

        label_col = lower_map.get("label")
        ym_col = lower_map.get("year_month")
        assert label_col is not None, f"Khong co cot label trong {path}"
        assert ym_col is not None, f"Khong co cot year_month trong {path}"

        feature_cols = feature_columns_from_schema(names)
        if feature_cols_ref is None:
            feature_cols_ref = feature_cols
        else:
            assert set(feature_cols) == set(feature_cols_ref), f"Feature columns khong dong nhat: {path}"

        id_col = lower_map.get("sha256") or lower_map.get("hash")
        metadata_columns = [label_col, ym_col] + ([id_col] if id_col else [])
        table = pq.read_table(path, columns=metadata_columns)
        labels = [label_to_int(x.as_py()) for x in table[label_col]]
        times = [parse_year_month(x.as_py()) for x in table[ym_col]]
        sample_ids = ([str(x.as_py()) for x in table[id_col]] if id_col
                      else [f"{path.name}:{idx}" for idx in range(len(labels))])

        for idx, (yv, tv, sample_id) in enumerate(zip(labels, times, sample_ids)):
            meta_rows.append({
                "path": path,
                "row_index": idx,
                "label": yv,
                "time": tv,
                "month": f"{tv.year:04d}-{tv.month:02d}",
                "sample_id": sample_id,
            })

    return meta_rows, feature_cols_ref

meta_rows, feature_cols = load_all_rows_metadata(lamda_parquets)
sample_ids = [row["sample_id"] for row in meta_rows]
duplicate_sample_ids = len(sample_ids) - len(set(sample_ids))
if duplicate_sample_ids:
    raise ValueError(f"LAMDA có {duplicate_sample_ids} mã mẫu trùng; dừng trước khi chia dữ liệu.")

print("Metadata rows:", len(meta_rows))
print("Features:", len(feature_cols))
print("Months:", sorted(set(r["month"] for r in meta_rows)))

In [ ]:
def build_Xyt_from_rows(rows, feature_cols):
    rows_by_path = defaultdict(list)
    for r in rows:
        rows_by_path[r["path"]].append(r)

    X_parts, y_parts, t_parts = [], [], []

    for path, rows in sorted(rows_by_path.items(), key=lambda kv: str(kv[0])):
        row_indices = [r["row_index"] for r in rows]
        table = pq.read_table(path, columns=feature_cols)
        X_table = (table.select(feature_cols)
                   if row_indices == list(range(table.num_rows))
                   else table.select(feature_cols).take(row_indices))
        # Không tạo một ma trận dense n_rows × 4.561. Chuyển từng khối
        # cột sang CSR rồi ghép ngang, giữ đỉnh bộ nhớ ổn định trên Kaggle.
        feature_block_columns = 256
        sparse_blocks = []
        for start in range(0, len(feature_cols), feature_block_columns):
            block_columns = feature_cols[start:start + feature_block_columns]
            dense_block = np.column_stack([
                X_table[column].combine_chunks().to_numpy(zero_copy_only=False)
                for column in block_columns
            ]).astype(np.float32, copy=False)
            sparse_blocks.append(sp.csr_matrix(dense_block))
            del dense_block
        X_parts.append(sp.hstack(sparse_blocks, format="csr"))
        del sparse_blocks, X_table, table
        y_parts.append(np.asarray([r["label"] for r in rows], dtype=np.int64))
        t_parts.append(np.asarray([r["time"] for r in rows], dtype=object))

    X = sp.vstack(X_parts).tocsr()
    y = np.concatenate(y_parts)
    t = np.concatenate(t_parts)

    # Giữ thứ tự nguồn/row_index trong cùng tháng. Quicksort mặc định
    # không ổn định và có thể làm thay đổi mẫu được chọn khi xác suất hòa.
    order = np.argsort(t, kind="stable")
    return X[order], y[order], t[order]

X, y, t = build_Xyt_from_rows(meta_rows, feature_cols)

DATASET_ARTIFACT_TAG = "LAMDA" + "_".join(str(year) for year in DATASET_YEARS)
pickle.dump(X, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-X.p", "wb"))
pickle.dump(y, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-y.p", "wb"))
pickle.dump(t, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-t.p", "wb"))
pickle.dump(feature_cols, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-feature-cols.p", "wb"))

print("X:", X.shape)
print("y:", y.shape, "malware_rate:", round(float(y.mean()), 4))
print("t:", min(t), "->", max(t))
print("Saved dataset tag:", DATASET_ARTIFACT_TAG)

In [ ]:
def month_key(dt):
    return f"{dt.year:04d}-{dt.month:02d}"

def validate_dataset(X, y, t):
    months = sorted(set(month_key(x) for x in t))
    rows = []
    for m in months:
        idx = np.array([i for i, tv in enumerate(t) if month_key(tv) == m])
        yy = y[idx]
        rows.append({
            "month": m,
            "total": int(len(idx)),
            "malware": int(yy.sum()),
            "benign": int((yy == 0).sum()),
            "malware_rate": float(yy.mean()) if len(yy) else 0,
        })

    zero_rows = int((X.getnnz(axis=1) == 0).sum()) if sp.issparse(X) else int((X.sum(axis=1) == 0).sum())
    sparsity = 1.0 - (X.nnz / (X.shape[0] * X.shape[1])) if sp.issparse(X) else float(np.mean(X == 0))
    stored_values = X.data if sp.issparse(X) else np.asarray(X).reshape(-1)
    nonfinite_values = int((~np.isfinite(stored_values)).sum())
    if nonfinite_values:
        raise ValueError(f"Ma trận đặc trưng có {nonfinite_values} giá trị NaN/Inf.")
    if set(np.unique(y)) - {0, 1}:
        raise ValueError(f"Nhãn ngoài miền nhị phân: {np.unique(y)}")
    if X.shape[0] != len(y) or len(y) != len(t):
        raise ValueError("X, y và t không có cùng số mẫu.")
    if X.shape[1] != len(feature_cols):
        raise ValueError("Số cột X không khớp danh sách đặc trưng.")
    if any(t[i] > t[i + 1] for i in range(len(t) - 1)):
        raise ValueError("Dữ liệu chưa được sắp tăng dần theo thời gian.")

    report = {
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_months": int(len(months)),
        "start_month": months[0],
        "end_month": months[-1],
        "malware_rate": float(y.mean()),
        "zero_rows": zero_rows,
        "sparsity": float(sparsity),
        "nonfinite_stored_values": nonfinite_values,
        "labels": sorted(int(value) for value in np.unique(y)),
    }

    return report, rows

validation_report, month_rows = validate_dataset(X, y, t)

expected_dataset_values = {
    "n_samples": EXPECTED_LAMDA_SAMPLES,
    "n_features": EXPECTED_LAMDA_FEATURES,
}
dataset_mismatches = {
    key: {"observed": validation_report.get(key), "expected": expected}
    for key, expected in expected_dataset_values.items()
    if int(validation_report.get(key, -1)) != int(expected)
}
if dataset_mismatches:
    raise ValueError(
        "LAMDA không khớp revision đã khóa: "
        + json.dumps(dataset_mismatches, ensure_ascii=False))

with open(TABLE_OUT / "dataset_validation_report.json", "w") as f:
    json.dump(validation_report, f, indent=2)

with open(TABLE_OUT / "month_distribution.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=month_rows[0].keys())
    writer.writeheader()
    writer.writerows(month_rows)

print(validation_report)
for r in month_rows:
    print(r)

In [ ]:
from tesseract import temporal

# ================= MONKEY PATCH: SỬA HÀM CHIA DỮ LIỆU TRỰC TIẾP TRONG RAM =================
if not hasattr(temporal, "original_time_aware_train_test_split"):
    temporal.original_time_aware_train_test_split = temporal.time_aware_train_test_split

def custom_time_aware_train_test_split(*args, **kwargs):
    # Gọi hàm chia dữ liệu gốc
    splits = temporal.original_time_aware_train_test_split(*args, **kwargs)
    X_train, X_tests, y_train, y_tests, t_train, t_tests = splits

    # Lọc bỏ hoàn toàn các period test bị rỗng (ví dụ: các tháng trống của năm 2015)
    non_empty_idx = [i for i, xt in enumerate(X_tests) if xt.shape[0] > 0]
    X_tests = [X_tests[i] for i in non_empty_idx]
    y_tests = [y_tests[i] for i in non_empty_idx]
    t_tests = [t_tests[i] for i in non_empty_idx]

    return X_train, X_tests, y_train, y_tests, t_train, t_tests

# Ghi đè hàm gốc trong bộ nhớ
temporal.time_aware_train_test_split = custom_time_aware_train_test_split
print("Đã vá (monkey-patched) hàm chia dữ liệu thành công!")
# =========================================================================================

# Thực hiện chia dữ liệu (hàm mới sẽ tự động lọc các period trống của năm 2015)
splits = temporal.time_aware_train_test_split(
    X=X,
    y=y,
    t=t,
    train_size=TRAINING_WINDOW,
    test_size=TESTING_WINDOW,
    granularity=GRANULARITY,
)

X_train, X_tests, y_train, y_tests, t_train, t_tests = splits

split_rows = []
split_rows.append({
    "period": "train",
    "n": int(X_train.shape[0]),
    "start": str(min(t_train)),
    "end": str(max(t_train)),
    "malware_rate": float(y_train.mean()),
})

for i, (xt, yt, tt) in enumerate(zip(X_tests, y_tests, t_tests), start=1):
    split_rows.append({
        "period": f"test_{i}",
        "n": int(xt.shape[0]),
        "start": str(min(tt)) if len(tt) else "",
        "end": str(max(tt)) if len(tt) else "",
        "malware_rate": float(yt.mean()) if len(yt) else 0,
    })

with open(TABLE_OUT / "time_split_check.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=split_rows[0].keys())
    writer.writeheader()
    writer.writerows(split_rows)

for r in split_rows:
    print(r)


# ══════════════════════════════════════════════════════════════════════════════
# V13 — kiểm chứng cửa sổ huấn luyện theo tháng lịch
# ══════════════════════════════════════════════════════════════════════════════
from collections import Counter as _Counter

if set(np.unique(y_train)) != {0, 1}:
    raise ValueError(f"Tập huấn luyện ban đầu không có đủ hai lớp: {np.unique(y_train)}")
if not X_tests or any(len(test_labels) == 0 for test_labels in y_tests):
    raise ValueError("Sau lọc vẫn còn chu kỳ kiểm thử rỗng.")
first_test_time = min(min(period_times) for period_times in t_tests)
if max(t_train) >= first_test_time:
    raise ValueError(
        f"Vi phạm thứ tự thời gian: max(train)={max(t_train)} >= "
        f"min(test)={first_test_time}")

def kiem_chung_cua_so_huan_luyen(t_train, t_tests) -> dict:
    """In từng tháng lịch trong cửa sổ huấn luyện và chỉ rõ tháng nào khuyết.

    Tesseract cắt theo BIÊN LỊCH nên cửa sổ `TRAINING_WINDOW` tháng có thể chứa ít
    tháng thực có dữ liệu hơn. Báo cáo phải phát biểu đúng con số này.
    """
    def mkey(d): return f"{d.year:04d}-{d.month:02d}"
    dem = _Counter(mkey(d) for d in t_train)
    d0 = min(t_train); m0 = d0.year * 12 + d0.month
    lich = []
    for k in range(TRAINING_WINDOW):
        m = m0 + k
        lich.append(f"{(m - 1) // 12:04d}-{((m - 1) % 12) + 1:02d}")
    co = [k for k in lich if dem.get(k, 0) > 0]
    khuyet = [k for k in lich if dem.get(k, 0) == 0]

    print(f"Cửa sổ huấn luyện trải {TRAINING_WINDOW} tháng lịch: {lich[0]} … {lich[-1]}")
    print(f"  Tháng CÓ dữ liệu : {len(co)}")
    print(f"  Tháng KHUYẾT     : {len(khuyet)}  {khuyet if khuyet else ''}")
    print(f"  Tổng số mẫu      : {len(t_train):,}".replace(",", "."))
    for k in lich:
        n = dem.get(k, 0)
        print(f"    {k}  {('khuyết' if n == 0 else format(n, ',').replace(',', '.')):>10}")
    if khuyet:
        print(f"\n  ⚠️  Phát biểu đúng trong báo cáo: 'cửa sổ huấn luyện trải "
              f"{TRAINING_WINDOW} tháng lịch, chứa dữ liệu của {len(co)} tháng'.")
        print("      KHÔNG viết 'huấn luyện trên 12 tháng dữ liệu'.")
        print("      Ràng buộc C1 vẫn thoả mãn: mọi mẫu huấn luyện đều trước mọi mẫu kiểm thử.")
    return {"training_window_calendar_months": TRAINING_WINDOW,
            "months_with_data": len(co), "months_missing": khuyet,
            "n_train_samples": int(len(t_train)), "n_test_periods": len(t_tests)}

TRAIN_WINDOW_AUDIT = kiem_chung_cua_so_huan_luyen(t_train, t_tests)
(TABLE_OUT / "training_window_audit.json").write_text(
    json.dumps(TRAIN_WINDOW_AUDIT, indent=2, ensure_ascii=False), encoding="utf-8")

EXPECTED_PERIODS = len(X_tests)
OBSERVED_TEST_SAMPLES = int(sum(int(labels.size) for labels in y_tests))
if EXPECTED_PERIODS != EXPECTED_TEST_PERIODS:
    raise ValueError(
        f"Số chu kỳ kiểm thử {EXPECTED_PERIODS}, kỳ vọng {EXPECTED_TEST_PERIODS}.")
if OBSERVED_TEST_SAMPLES != EXPECTED_TEST_SAMPLES:
    raise ValueError(
        f"Số mẫu kiểm thử {OBSERVED_TEST_SAMPLES}, kỳ vọng {EXPECTED_TEST_SAMPLES}.")
if int(y_train.size) != EXPECTED_INITIAL_TRAIN_SAMPLES:
    raise ValueError(
        f"Số mẫu huấn luyện ban đầu {y_train.size}, "
        f"kỳ vọng {EXPECTED_INITIAL_TRAIN_SAMPLES}.")

observed_test_months = []
for period, (features, labels, times) in enumerate(
        zip(X_tests, y_tests, t_tests), start=1):
    if not (features.shape[0] == labels.size == times.size):
        raise ValueError(f"Chu kỳ {period}: X/y/t lệch số hàng.")
    months = np.asarray(times).astype("datetime64[M]")
    unique_months = np.unique(months)
    if unique_months.size != 1:
        raise ValueError(f"Chu kỳ {period} không nằm trong đúng một tháng lịch.")
    observed_test_months.append(unique_months[0])
if len(np.unique(observed_test_months)) != EXPECTED_PERIODS:
    raise ValueError("Các chu kỳ kiểm thử có tháng lịch bị trùng.")
if any(left >= right for left, right in zip(
        observed_test_months[:-1], observed_test_months[1:])):
    raise ValueError("Thứ tự tháng kiểm thử không tăng nghiêm ngặt.")

_data_pipeline_components = (
    baseline_candidates, matching_lamda_files, source_file_sha256,
    parse_year_month, label_to_int, feature_columns_from_schema,
    load_all_rows_metadata, build_Xyt_from_rows, validate_dataset,
    custom_time_aware_train_test_split, kiem_chung_cua_so_huan_luyen,
)
DATA_PIPELINE_IMPLEMENTATION_FINGERPRINT = hashlib.sha256(
    json.dumps(
        [_stable_code_descriptor(function.__code__)
         for function in _data_pipeline_components],
        ensure_ascii=False, sort_keys=True, separators=(",", ":"),
    ).encode()
).hexdigest()

split_fingerprint_payload = {
    "protocol_version": PROTOCOL_VERSION,
    "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
    "training_window_mode": TRAINING_WINDOW_MODE,
    "training_window": TRAINING_WINDOW,
    "testing_window": TESTING_WINDOW,
    "granularity": GRANULARITY,
    "feature_dim": int(X.shape[1]),
    "expected_test_periods": EXPECTED_TEST_PERIODS,
    "expected_test_samples": EXPECTED_TEST_SAMPLES,
    "observed_test_samples": OBSERVED_TEST_SAMPLES,
    "data_pipeline_implementation_fingerprint": (
        DATA_PIPELINE_IMPLEMENTATION_FINGERPRINT),
    "train": split_rows[0],
    "tests": split_rows[1:],
}
SPLIT_FINGERPRINT = hashlib.sha256(
    json.dumps(split_fingerprint_payload, ensure_ascii=False, sort_keys=True).encode()
).hexdigest()
(TABLE_OUT / "split_fingerprint.json").write_text(
    json.dumps({**split_fingerprint_payload,
                "split_fingerprint_sha256": SPLIT_FINGERPRINT},
               indent=2, ensure_ascii=False),
    encoding="utf-8")
print(f"\nEXPECTED_PERIODS = {EXPECTED_PERIODS}  (dùng cho ô kiểm tra toàn vẹn)")
print("SPLIT_FINGERPRINT =", SPLIT_FINGERPRINT)

## Ô 8 — MLP tĩnh đồng bộ trên cùng 10 seed


In [ ]:
# MLP tĩnh đồng bộ trên cùng 10 seed và cùng split V13.
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
import csv
import hashlib
import json
import os
import pickle
import time
import uuid
import zipfile

STATIC_METHOD = "Static-MLP"
STATIC_PROTOCOL = "static_initial_window_no_update"
STATIC_CACHE_DIR = RAW_OUT / "static_baselines"
STATIC_CACHE_DIR.mkdir(parents=True, exist_ok=True)
STATIC_MODEL_SPEC = {
    "scaler": "MaxAbsScaler",
    "hidden_layer_sizes": [512, 256, 128], "activation": "relu",
    "solver": "adam", "alpha": 1e-4,
    "learning_rate_init": 1e-3, "max_iter": 20,
    "shuffle": True, "early_stopping": True,
    "validation_fraction": 0.1, "n_iter_no_change": 10,
}
STATIC_ENV_SPEC = {
    key: ENVIRONMENT_STAMP[key]
    for key in ("python", "numpy", "scipy", "scikit_learn")
}

def _static_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _static_path(seed):
    return STATIC_CACHE_DIR / f"Static-MLP-static-Seed{int(seed)}.p"

def _static_sidecar(path):
    return Path(str(path) + ".meta.json")

def _static_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if not np.isin(y_pred, [0, 1]).all():
        raise ValueError("Static-MLP sinh dự đoán ngoài miền 0/1.")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "p": int(tp + fn), "n": int(tn + fp), "tot": int(y_true.size),
        "precision": (float(tp / (tp + fp)) if tp + fp else float("nan")),
        "recall": (float(tp / (tp + fn)) if tp + fn else float("nan")),
        "f1": (float(2 * tp / (2 * tp + fp + fn))
               if 2 * tp + fp + fn else float("nan")),
        "fnr": (float(fn / (fn + tp)) if fn + tp else float("nan")),
        "fpr": (float(fp / (fp + tn)) if fp + tn else float("nan")),
    }

def _make_locked_mlp(seed):
    """Một đặc tả kiến trúc dùng chung cho Static-MLP và MLP-CFJR."""
    return make_pipeline(
        MaxAbsScaler(copy=True),
        MLPClassifier(
            hidden_layer_sizes=(512, 256, 128), activation="relu",
            solver="adam", alpha=1e-4, learning_rate_init=1e-3,
            max_iter=20, shuffle=True, early_stopping=True,
            validation_fraction=0.1, n_iter_no_change=10,
            random_state=int(seed)))

def _evaluate_static_mlp(seed):
    model = _make_locked_mlp(seed)
    model.fit(X_train, y_train)
    rows = []
    period_predictions = []
    for period, (X_test, y_test, t_test) in enumerate(
            zip(X_tests, y_tests, t_tests), start=1):
        prediction = np.asarray(model.predict(X_test), dtype=np.int8)
        period_predictions.append(prediction)
        rows.append({
            "method": STATIC_METHOD, "seed": int(seed), "period": int(period),
            "period_start": str(np.min(t_test)), "period_end": str(np.max(t_test)),
            "protocol": STATIC_PROTOCOL,
            "true_malware_rate": float(np.mean(y_test)),
            **_static_metrics(y_test, prediction),
        })
    return rows, period_predictions

def _validate_static_artifact(path, seed, require_sidecar=False):
    with open(path, "rb") as handle:
        artifact = pickle.load(handle)
    expected = {
        "method": STATIC_METHOD, "seed": int(seed),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "model_spec": STATIC_MODEL_SPEC, "environment_spec": STATIC_ENV_SPEC,
    }
    for key, value in expected.items():
        if artifact.get(key) != value:
            raise ValueError(f"Static cache sai {key}.")
    if int(artifact.get("schema_version", -1)) != 6:
        raise ValueError("Static cache không thuộc schema 6 có dự đoán gốc.")
    if artifact.get("protocol") != STATIC_PROTOCOL:
        raise ValueError("Static cache sai protocol.")
    if artifact.get("protocol_version") != PROTOCOL_VERSION:
        raise ValueError("Static cache thuộc protocol_version khác.")
    rows = artifact.get("period_rows", [])
    period_predictions = artifact.get("period_predictions", [])
    if (len(rows) != EXPECTED_PERIODS
            or len(period_predictions) != EXPECTED_PERIODS):
        raise ValueError("Static cache sai số chu kỳ hoặc thiếu dự đoán gốc.")
    observed_test_samples = 0
    for index, row in enumerate(rows):
        canonical_y = np.asarray(y_tests[index], dtype=int)
        canonical_t = np.asarray(t_tests[index]).astype("datetime64[ns]")
        canonical_total = int(X_tests[index].shape[0])
        prediction = np.asarray(
            period_predictions[index], dtype=np.int64).reshape(-1)
        if (prediction.size != canonical_total
                or not np.isin(prediction, [0, 1]).all()):
            raise ValueError(
                f"Static cache sai dự đoán gốc ở chu kỳ {index + 1}.")
        observed_test_samples += canonical_total
        required = {"tp", "tn", "fp", "fn", "p", "n", "tot",
                    "f1", "fnr", "fpr"}
        if not required.issubset(row):
            raise ValueError(f"Static cache thiếu cột ở chu kỳ {index + 1}.")
        recomputed = _static_metrics(canonical_y, prediction)
        counts = {key: int(row[key]) for key in ("tp", "tn", "fp", "fn")}
        recomputed_counts = {
            key: int(recomputed[key]) for key in ("tp", "tn", "fp", "fn")
        }
        if min(counts.values()) < 0:
            raise ValueError(f"Static cache có số đếm âm ở chu kỳ {index + 1}.")
        if counts != recomputed_counts:
            raise ValueError(
                f"Static cache sai ma trận nhầm lẫn ở chu kỳ {index + 1}.")
        expected_p = int(np.sum(canonical_y == 1))
        expected_n = int(np.sum(canonical_y == 0))
        period_start = np.datetime64(row.get("period_start"), "ns")
        period_end = np.datetime64(row.get("period_end"), "ns")
        if (row.get("method") != STATIC_METHOD
                or int(row.get("seed", -1)) != int(seed)
                or int(row.get("period", -1)) != index + 1
                or int(row.get("tot", -1)) != canonical_total
                or int(row.get("p", -1)) != expected_p
                or int(row.get("n", -1)) != expected_n
                or counts["tp"] + counts["fn"] != expected_p
                or counts["tn"] + counts["fp"] != expected_n
                or period_start != canonical_t.min()
                or period_end != canonical_t.max()):
            raise ValueError(f"Static cache sai hàng chu kỳ {index + 1}.")
        expected_metrics = {
            "f1": (2 * counts["tp"] /
                   (2 * counts["tp"] + counts["fp"] + counts["fn"])
                   if 2 * counts["tp"] + counts["fp"] + counts["fn"]
                   else float("nan")),
            "fnr": (counts["fn"] / expected_p if expected_p else float("nan")),
            "fpr": (counts["fp"] / expected_n if expected_n else float("nan")),
        }
        for metric, expected_value in expected_metrics.items():
            observed_value = float(row[metric])
            if not ((np.isnan(observed_value) and np.isnan(expected_value))
                    or np.isclose(observed_value, expected_value,
                                  rtol=0.0, atol=1e-12)):
                raise ValueError(
                    f"Static cache sai {metric} ở chu kỳ {index + 1}.")
    if observed_test_samples != EXPECTED_TEST_SAMPLES:
        raise ValueError("Static cache sai tổng số mẫu kiểm thử.")
    if (artifact.get("n_test_samples") is not None
            and int(artifact["n_test_samples"]) != EXPECTED_TEST_SAMPLES):
        raise ValueError("Static cache khai báo sai n_test_samples.")
    metadata_path = _static_sidecar(path)
    if require_sidecar and not metadata_path.exists():
        raise FileNotFoundError("Static cache V13 thiếu sidecar.")
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if (metadata.get("status") != "complete"
                or int(metadata.get("schema_version", -1)) != 6
                or metadata.get("protocol_version") != PROTOCOL_VERSION
                or metadata.get("method") != STATIC_METHOD
                or int(metadata.get("seed", -1)) != int(seed)
                or metadata.get("dataset_source_fingerprint")
                != DATASET_SOURCE_FINGERPRINT
                or metadata.get("split_fingerprint") != SPLIT_FINGERPRINT
                or metadata.get("artifact_sha256") != _static_sha256(path)):
            raise ValueError("Static sidecar không khớp artifact.")
    return artifact

def _save_static_artifact(seed, rows, period_predictions, imported_from=None):
    final_path = _static_path(seed)
    temporary = final_path.with_name(final_path.name + f".{uuid.uuid4().hex}.tmp")
    artifact = {
        "schema_version": 6, "protocol_version": PROTOCOL_VERSION,
        "method": STATIC_METHOD, "seed": int(seed), "protocol": STATIC_PROTOCOL,
        "n_periods": len(rows),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "model_spec": STATIC_MODEL_SPEC, "environment_spec": STATIC_ENV_SPEC,
        "n_test_samples": int(sum(int(row["tot"]) for row in rows)),
        "imported_from": imported_from, "period_rows": rows,
        "period_predictions": [
            np.asarray(values, dtype=np.int8) for values in period_predictions
        ],
    }
    with open(temporary, "wb") as handle:
        pickle.dump(artifact, handle)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(temporary, final_path)
    metadata = {
        "status": "complete", "schema_version": 6,
        "protocol_version": PROTOCOL_VERSION,
        "method": STATIC_METHOD, "seed": int(seed),
        "artifact_sha256": _static_sha256(final_path),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "imported_from": imported_from,
    }
    metadata_path = _static_sidecar(final_path)
    metadata_tmp = metadata_path.with_name(metadata_path.name + f".{uuid.uuid4().hex}.tmp")
    metadata_tmp.write_text(json.dumps(metadata, indent=2, ensure_ascii=False),
                            encoding="utf-8")
    os.replace(metadata_tmp, metadata_path)
    return final_path

def _static_candidates(seed):
    filename = _static_path(seed).name
    candidates = [_static_path(seed)]
    if INPUT_ROOT.exists():
        candidates.extend(sorted(INPUT_ROOT.rglob(filename)))
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique

def _write_static_csv(path, rows):
    if not rows:
        return
    keys = []
    for row in rows:
        for key in row:
            if key not in keys:
                keys.append(key)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader(); writer.writerows(rows)

STATIC_RESULTS = {}
STATIC_CACHE_AUDIT = []
STATIC_RUNS_STARTED_THIS_SESSION = 0
STATIC_SESSION_STOP_REASON = "completed_static_plan"
for seed in SEEDS:
    artifact = None
    accepted_path = None
    for candidate in _static_candidates(seed):
        if not candidate.exists():
            continue
        try:
            artifact = _validate_static_artifact(
                candidate, seed, require_sidecar=(candidate == _static_path(seed)))
            accepted_path = candidate
            break
        except Exception as exc:
            STATIC_CACHE_AUDIT.append({
                "seed": int(seed), "candidate": str(candidate), "state": "rejected",
                "detail": f"{type(exc).__name__}: {exc}"})
    if artifact is not None:
        rows = artifact["period_rows"]
        period_predictions = artifact["period_predictions"]
        if (accepted_path != _static_path(seed)
                or artifact.get("protocol_version") != PROTOCOL_VERSION
                or int(artifact.get("schema_version", 0)) != 6):
            accepted_path = _save_static_artifact(
                seed, rows, period_predictions,
                imported_from=str(accepted_path))
            state = "validated_import"
        else:
            state = "validated_v13_cache"
        STATIC_RESULTS[(STATIC_METHOD, int(seed))] = rows
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": str(accepted_path), "state": state,
            "detail": "exact data/split/model/environment match"})
        print({"static": STATIC_METHOD, "seed": int(seed), "state": state})
        continue
    if KAGGLE_RUN_PHASE == "aggregate_only":
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": "", "state": "missing",
            "detail": "aggregate_only does not train"})
        continue
    if (STOP_AFTER_STATIC_RUNS is not None
            and STATIC_RUNS_STARTED_THIS_SESSION >= STOP_AFTER_STATIC_RUNS):
        STATIC_SESSION_STOP_REASON = "static_run_cap"
        break
    if ((time.monotonic() - SESSION_STARTED_AT) / 3600
            >= SESSION_LAUNCH_CUTOFF_HOURS):
        STATIC_SESSION_STOP_REASON = "static_launch_cutoff"
        break
    try:
        STATIC_RUNS_STARTED_THIS_SESSION += 1
        rows, period_predictions = _evaluate_static_mlp(seed)
        path = _save_static_artifact(seed, rows, period_predictions)
        _validate_static_artifact(path, seed, require_sidecar=True)
        STATIC_RESULTS[(STATIC_METHOD, int(seed))] = rows
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": str(path), "state": "trained_v13",
            "detail": "fit once on initial temporal window"})
        print({"static": STATIC_METHOD, "seed": int(seed), "state": "trained_v13"})
    except Exception as exc:
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": "", "state": "failed",
            "detail": f"{type(exc).__name__}: {exc}"})
        print("Static-MLP failed", seed, repr(exc))

_write_static_csv(
    TABLE_OUT / "v13_static_mlp_monthly_raw.csv",
    [row for rows in STATIC_RESULTS.values() for row in rows])
_write_static_csv(TABLE_OUT / "v13_static_cache_audit.csv", STATIC_CACHE_AUDIT)

def _export_static_checkpoint_atomic():
    checkpoint_path = OUT / f"{EXPERIMENT_NAME}_checkpoint_raw.zip"
    temporary = checkpoint_path.with_name(checkpoint_path.name + ".tmp")
    with zipfile.ZipFile(
            temporary, "w", compression=zipfile.ZIP_DEFLATED,
            compresslevel=1) as archive:
        for path in sorted(STATIC_CACHE_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(OUT))
    os.replace(temporary, checkpoint_path)
    return checkpoint_path

STATIC_CHECKPOINT = _export_static_checkpoint_atomic()
print({"static_valid": len(STATIC_RESULTS), "static_expected": len(SEEDS),
       "static_runs_started_this_session": STATIC_RUNS_STARTED_THIS_SESSION,
       "static_stop_reason": STATIC_SESSION_STOP_REASON,
       "static_checkpoint": str(STATIC_CHECKPOINT)})


## Ô 9 — CFJR: tái huấn luyện hợp nhất nhân quả toàn lịch sử

Chỉ pha `phase_d_cfjr_reference` hoặc `all` mới huấn luyện các chu kỳ còn thiếu. Những pha khác chỉ kiểm tra và nạp checkpoint CFJR đã có.


In [ ]:
# MLP-CFJR: tái huấn luyện hợp nhất nhân quả trên toàn lịch sử.
#
# Ở chu kỳ t, mô hình mới chỉ được fit trên tập khởi tạo và các chu kỳ < t.
# Nhãn của t chỉ được dùng để chấm điểm sau khi dự đoán đã khóa, rồi mới trở
# thành dữ liệu lịch sử cho chu kỳ t+1. CFJR dùng đúng kiến trúc Static-MLP;
# đây là cận tham chiếu thực nghiệm mạnh, không phải cận trên lý thuyết.
import scipy.sparse as sp

CFJR_CACHE_DIR = RAW_OUT / "cfjr_reference"
CFJR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CFJR_SCHEMA_VERSION = 1
CFJR_MODEL_SPEC = dict(STATIC_MODEL_SPEC)
CFJR_ENV_SPEC = dict(STATIC_ENV_SPEC)

def _cfjr_path(seed, period):
    return (CFJR_CACHE_DIR / f"Seed{int(seed)}"
            / f"MLP-CFJR-Seed{int(seed)}-Period{int(period):03d}.p")

def _cfjr_sidecar(path):
    return Path(str(path) + ".meta.json")

def _cfjr_expected_training_samples(period):
    return int(X_train.shape[0] + sum(
        X_tests[index].shape[0] for index in range(int(period) - 1)))

def _cfjr_expected_history_end(period):
    if int(period) == 1:
        values = np.asarray(t_train).astype("datetime64[ns]")
    else:
        values = np.asarray(t_tests[int(period) - 2]).astype("datetime64[ns]")
    return values.max()

def _cfjr_candidates(seed, period):
    filename = _cfjr_path(seed, period).name
    candidates = [_cfjr_path(seed, period)]
    if INPUT_ROOT.exists():
        candidates.extend(sorted(INPUT_ROOT.rglob(filename)))
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique

def _validate_cfjr_artifact(path, seed, period, require_sidecar=False):
    with open(path, "rb") as handle:
        artifact = pickle.load(handle)
    expected = {
        "schema_version": CFJR_SCHEMA_VERSION,
        "protocol_version": CFJR_PROTOCOL_VERSION,
        "method": CFJR_METHOD,
        "seed": int(seed), "period": int(period),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "model_spec": CFJR_MODEL_SPEC,
        "environment_spec": CFJR_ENV_SPEC,
    }
    for key, value in expected.items():
        if artifact.get(key) != value:
            raise ValueError(f"CFJR cache sai {key} ở seed={seed}, period={period}.")

    canonical_y = np.asarray(y_tests[int(period) - 1], dtype=int)
    canonical_t = np.asarray(
        t_tests[int(period) - 1]).astype("datetime64[ns]")
    prediction = np.asarray(
        artifact.get("prediction", []), dtype=np.int64).reshape(-1)
    if (prediction.size != canonical_y.size
            or not np.isin(prediction, [0, 1]).all()):
        raise ValueError("CFJR cache sai miền hoặc số lượng dự đoán.")

    row = dict(artifact.get("period_row", {}))
    recomputed = _static_metrics(canonical_y, prediction)
    for key in ("tp", "tn", "fp", "fn", "p", "n", "tot"):
        if int(row.get(key, -1)) != int(recomputed[key]):
            raise ValueError(f"CFJR cache sai {key}.")
    for key in ("f1", "fnr", "fpr"):
        observed, expected_value = float(row.get(key)), float(recomputed[key])
        if not ((np.isnan(observed) and np.isnan(expected_value))
                or np.isclose(observed, expected_value, rtol=0.0, atol=1e-12)):
            raise ValueError(f"CFJR cache sai {key}.")

    expected_training_samples = _cfjr_expected_training_samples(period)
    expected_history_end = _cfjr_expected_history_end(period)
    observed_history_end = np.datetime64(
        artifact.get("history_end"), "ns")
    test_start = canonical_t.min()
    if (int(artifact.get("training_samples", -1))
            != expected_training_samples):
        raise ValueError("CFJR cache sai số mẫu lịch sử dùng để fit.")
    if (observed_history_end != expected_history_end
            or not observed_history_end < test_start):
        raise ValueError("CFJR cache vi phạm chân trời nhân quả.")
    if (np.datetime64(row.get("period_start"), "ns") != test_start
            or np.datetime64(row.get("period_end"), "ns")
            != canonical_t.max()):
        raise ValueError("CFJR cache sai thời gian kiểm thử.")

    metadata_path = _cfjr_sidecar(path)
    if require_sidecar and not metadata_path.exists():
        raise FileNotFoundError("CFJR cache thiếu sidecar.")
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if (metadata.get("status") != "complete"
                or int(metadata.get("schema_version", -1))
                != CFJR_SCHEMA_VERSION
                or metadata.get("protocol_version") != CFJR_PROTOCOL_VERSION
                or metadata.get("method") != CFJR_METHOD
                or int(metadata.get("seed", -1)) != int(seed)
                or int(metadata.get("period", -1)) != int(period)
                or metadata.get("artifact_sha256") != _static_sha256(path)):
            raise ValueError("CFJR sidecar không khớp artifact.")
    return artifact

def _save_cfjr_artifact(seed, period, artifact, imported_from=None):
    final_path = _cfjr_path(seed, period)
    final_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = final_path.with_name(final_path.name + f".{uuid.uuid4().hex}.tmp")
    payload = dict(artifact)
    payload.update({
        "schema_version": CFJR_SCHEMA_VERSION,
        "protocol_version": CFJR_PROTOCOL_VERSION,
        "method": CFJR_METHOD,
        "seed": int(seed), "period": int(period),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "model_spec": CFJR_MODEL_SPEC,
        "environment_spec": CFJR_ENV_SPEC,
        "imported_from": imported_from,
    })
    with open(temporary, "wb") as handle:
        pickle.dump(payload, handle)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(temporary, final_path)
    metadata = {
        "status": "complete", "schema_version": CFJR_SCHEMA_VERSION,
        "protocol_version": CFJR_PROTOCOL_VERSION,
        "method": CFJR_METHOD, "seed": int(seed), "period": int(period),
        "artifact_sha256": _static_sha256(final_path),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "imported_from": imported_from,
    }
    metadata_path = _cfjr_sidecar(final_path)
    metadata_tmp = metadata_path.with_name(
        metadata_path.name + f".{uuid.uuid4().hex}.tmp")
    metadata_tmp.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(metadata_tmp, metadata_path)
    return final_path

def _cfjr_stack_history(X_blocks, y_blocks):
    if all(sp.issparse(block) for block in X_blocks):
        X_history = sp.vstack(X_blocks, format="csr")
    else:
        X_history = np.vstack([
            block.toarray() if sp.issparse(block) else np.asarray(block)
            for block in X_blocks])
    y_history = np.concatenate([
        np.asarray(block, dtype=int).reshape(-1) for block in y_blocks])
    return X_history, y_history

def _fit_cfjr_period(seed, period, X_blocks, y_blocks, history_end):
    X_history, y_history = _cfjr_stack_history(X_blocks, y_blocks)
    expected_training_samples = _cfjr_expected_training_samples(period)
    if X_history.shape[0] != expected_training_samples:
        raise ValueError("CFJR tạo sai kích thước tập lịch sử.")
    if np.unique(y_history).size < 2:
        raise ValueError("CFJR không thể fit khi tập lịch sử chỉ có một lớp.")

    model = _make_locked_mlp(seed)
    fit_started = time.monotonic()
    model.fit(X_history, y_history)
    fit_seconds = float(time.monotonic() - fit_started)
    X_test = X_tests[int(period) - 1]
    y_test = np.asarray(y_tests[int(period) - 1], dtype=int)
    t_test = np.asarray(t_tests[int(period) - 1]).astype("datetime64[ns]")
    prediction = np.asarray(model.predict(X_test), dtype=np.int8)
    metrics = _static_metrics(y_test, prediction)
    mlp = model.named_steps["mlpclassifier"]
    row = {
        "method": CFJR_METHOD, "seed": int(seed), "period": int(period),
        "period_start": str(t_test.min()), "period_end": str(t_test.max()),
        "protocol": "causal_full_history_joint_retraining",
        "training_samples": int(X_history.shape[0]),
        "training_benign": int(np.sum(y_history == 0)),
        "training_malware": int(np.sum(y_history == 1)),
        "history_end": str(np.datetime64(history_end, "ns")),
        "fit_seconds": fit_seconds,
        "n_iter": int(getattr(mlp, "n_iter_", -1)),
        "true_malware_rate": float(np.mean(y_test)),
        **metrics,
    }
    return {
        "training_samples": int(X_history.shape[0]),
        "history_end": str(np.datetime64(history_end, "ns")),
        "fit_seconds": fit_seconds,
        "period_row": row,
        "prediction": prediction,
    }

CFJR_RESULTS = {}
CFJR_CACHE_AUDIT = []
CFJR_REFITS_STARTED_THIS_SESSION = 0
CFJR_STOP_REASON = "completed_or_loaded_cfjr_plan"
CFJR_LAUNCH_ALLOWED = bool(RUN_CFJR_REFERENCE)
CFJR_CONSECUTIVE_FAILURES = 0

for seed in SEEDS:
    X_blocks = [X_train]
    y_blocks = [y_train]
    history_end = np.asarray(t_train).astype("datetime64[ns]").max()
    seed_rows = []
    for period in range(1, EXPECTED_PERIODS + 1):
        artifact = None
        accepted_path = None
        for candidate in _cfjr_candidates(seed, period):
            if not candidate.exists():
                continue
            try:
                artifact = _validate_cfjr_artifact(
                    candidate, seed, period,
                    require_sidecar=(candidate == _cfjr_path(seed, period)))
                accepted_path = candidate
                break
            except Exception as exc:
                CFJR_CACHE_AUDIT.append({
                    "seed": int(seed), "period": int(period),
                    "candidate": str(candidate), "state": "rejected",
                    "detail": f"{type(exc).__name__}: {exc}"})

        if artifact is not None:
            if accepted_path != _cfjr_path(seed, period):
                accepted_path = _save_cfjr_artifact(
                    seed, period, artifact, imported_from=str(accepted_path))
                state = "validated_import"
            else:
                state = "validated_cfjr_cache"
            seed_rows.append(dict(artifact["period_row"]))
            CFJR_CACHE_AUDIT.append({
                "seed": int(seed), "period": int(period),
                "candidate": str(accepted_path), "state": state,
                "detail": "exact data/split/model/causal-history match"})
        elif CFJR_LAUNCH_ALLOWED:
            elapsed_hours = (time.monotonic() - SESSION_STARTED_AT) / 3600
            if elapsed_hours >= CFJR_REFIT_LAUNCH_CUTOFF_HOURS:
                CFJR_LAUNCH_ALLOWED = False
                CFJR_STOP_REASON = "cfjr_launch_cutoff"
            elif (STOP_AFTER_CFJR_REFITS is not None
                  and CFJR_REFITS_STARTED_THIS_SESSION
                  >= STOP_AFTER_CFJR_REFITS):
                CFJR_LAUNCH_ALLOWED = False
                CFJR_STOP_REASON = "cfjr_refit_cap"

            if CFJR_LAUNCH_ALLOWED:
                try:
                    CFJR_REFITS_STARTED_THIS_SESSION += 1
                    artifact = _fit_cfjr_period(
                        seed, period, X_blocks, y_blocks, history_end)
                    path = _save_cfjr_artifact(seed, period, artifact)
                    artifact = _validate_cfjr_artifact(
                        path, seed, period, require_sidecar=True)
                    seed_rows.append(dict(artifact["period_row"]))
                    CFJR_CACHE_AUDIT.append({
                        "seed": int(seed), "period": int(period),
                        "candidate": str(path), "state": "trained_cfjr",
                        "detail": (f"fit_seconds={artifact['fit_seconds']:.3f}; "
                                   f"training_samples={artifact['training_samples']}")})
                    CFJR_CONSECUTIVE_FAILURES = 0
                    print({"cfjr": CFJR_METHOD, "seed": int(seed),
                           "period": int(period), "state": "trained",
                           "training_samples": artifact["training_samples"],
                           "fit_seconds": round(artifact["fit_seconds"], 3)})
                except Exception as exc:
                    CFJR_CONSECUTIVE_FAILURES += 1
                    CFJR_CACHE_AUDIT.append({
                        "seed": int(seed), "period": int(period),
                        "candidate": "", "state": "failed",
                        "detail": f"{type(exc).__name__}: {exc}"})
                    print("MLP-CFJR failed", seed, period, repr(exc))
                    if CFJR_CONSECUTIVE_FAILURES >= MAX_CONSECUTIVE_FAILURES:
                        CFJR_LAUNCH_ALLOWED = False
                        CFJR_STOP_REASON = "cfjr_consecutive_failures"
        else:
            CFJR_CACHE_AUDIT.append({
                "seed": int(seed), "period": int(period),
                "candidate": "", "state": "missing",
                "detail": "current phase does not train CFJR"})

        # Mở nhãn chu kỳ t chỉ sau khi dự đoán/cache của t đã được xử lý.
        X_blocks.append(X_tests[period - 1])
        y_blocks.append(y_tests[period - 1])
        history_end = np.asarray(
            t_tests[period - 1]).astype("datetime64[ns]").max()

    CFJR_RESULTS[(CFJR_METHOD, int(seed))] = sorted(
        seed_rows, key=lambda row: int(row["period"]))

_write_static_csv(
    TABLE_OUT / "v13_cfjr_monthly_raw.csv",
    [row for rows in CFJR_RESULTS.values() for row in rows])
_write_static_csv(TABLE_OUT / "v13_cfjr_cache_audit.csv", CFJR_CACHE_AUDIT)
CFJR_COMPLETE_SEEDS = sorted([
    int(seed) for (method, seed), rows in CFJR_RESULTS.items()
    if method == CFJR_METHOD and len(rows) == EXPECTED_PERIODS])
print({"cfjr_complete_seeds": CFJR_COMPLETE_SEEDS,
       "cfjr_expected_seeds": SEEDS,
       "cfjr_refits_started_this_session": CFJR_REFITS_STARTED_THIS_SESSION,
       "cfjr_stop_reason": CFJR_STOP_REASON,
       "cfjr_cache_dir": str(CFJR_CACHE_DIR)})


## Ô 10 — Chạy DRMD theo pha: chính, độ nhạy ngân sách hoặc chi phí


In [ ]:
from datetime import datetime, timezone
from copy import deepcopy
import csv
import hashlib
import json
import os
import pickle
import shutil
import traceback
import uuid

import DRMD.base as drmd_base
drmd_base.os = os
from DRMD.base import run
from DRMD.config import Settings, generate_experiment_configs

RUN_PROGRESS_LOG = OUT / "run_progress.jsonl"
FAILURE_LOG = OUT / "failed_runs.jsonl"
QUARANTINE_LOG = OUT / "quarantine_log.jsonl"
RUN_STATUS_PATH = OUT / "run_status.json"
DRMD_RUNS_STARTED_THIS_SESSION = 0
DRMD_RUNS_COMPLETED_THIS_SESSION = 0
DRMD_SESSION_STOPPED = False
SESSION_STOP_REASON = "completed_plan"
FAILED_RUNS = []

REQUIRED_RESULT_SERIES = {
    "tp", "tn", "fp", "fn", "p", "n", "tot",
    "selected", "rejected", "selected_indexes_all", "rejected_indexes_all",
    "rejected_malware_count", "rejected_benign_count",
    "y_tests", "y_preds", "t_tests",
    "binary_y_true_all", "binary_y_preds_all", "binary_t_all",
    "selection_diagnostics", "audit_indexes_all",
    "fn_penalty_used", "fn_penalty_next", "audit_count",
    "policy_action_all",
    "afnp_target_fnr", "afnp_estimated_fnr",
    "audit_positive_count", "audit_fn_count",
    "rejected_y_true", "selected_y_true",
    "selected_malware_count", "selected_benign_count",
    "bhr_telemetry",
}

def session_elapsed_hours():
    return (time.monotonic() - SESSION_STARTED_AT) / 3600

def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), default=str)

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def append_jsonl(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False,
                                    default=str), encoding="utf-8")
    os.replace(temporary, path)

def write_progress_event(event, cfg=None, extra=None):
    payload = {"time_utc": datetime.now(timezone.utc).isoformat(),
               "event": event, "elapsed_hours": round(session_elapsed_hours(), 4)}
    if cfg is not None:
        payload.update({"seed": int(cfg.seed), "al": int(cfg.al_rate),
                        "rej": int(cfg.reject_rate), "reject_cost": float(cfg.reject_cost),
                        "is_reject_action": bool(cfg.is_reject_action),
                        "is_fine_tuning": bool(cfg.is_fine_tuning),
                        "al_select_mode": str(cfg.al_select_mode),
                        "select_reject_labeled": bool(cfg.select_reject_labeled),
                        "posthoc_augmented_AL": int(cfg.posthoc_augmented_AL)})
    if extra:
        payload.update(extra)
    append_jsonl(RUN_PROGRESS_LOG, payload)

def legacy_result_stem(cfg):
    """Tái tạo tên dài của DRMD gốc, chỉ dùng cho truy vết và cache cũ."""
    data = str(cfg.dataset).split(",")
    ds = data[0].split("/")[1]
    fs = data[1].split("-")[0]
    stem = f"DRMD-{cfg.model}-DS{ds}-FS{fs}"
    stem += f"-mp{cfg.minority_priority}"
    stem += f"{f'-Ts{cfg.temporal_scaling}' if cfg.temporal_rewards else ''}"
    stem += f"-bs{cfg.minibatch_size}"
    stem += f"-e{cfg.update_epochs}x{cfg.training_epochs}"
    stem += f"-HL{cfg.layer_size}x{cfg.hidden_layers}"
    stem += f"{f'-FT{cfg.finetuning_size}' if cfg.is_fine_tuning else ''}"
    stem += f"-Rej{cfg.reject_rate}-RA{cfg.is_reject_action}-RC{cfg.reject_cost}"
    stem += f"-RO{cfg.reward_rejected_outcome}-RS{cfg.reject_positive_scale}x{cfg.reject_negative_scale}"
    stem += f"-AL{cfg.al_rate}-SelectReject{cfg.select_reject_labeled}"
    stem += f"{f'-PHAAL{cfg.posthoc_augmented_AL}' if cfg.posthoc_augmented_AL > 0 else ''}"
    stem += f"-ALMode{cfg.al_select_mode}-Seed{cfg.seed}"
    return stem

def expected_result_path(cfg):
    return Path(cfg.results_save_location) / (compact_artifact_stem(
        legacy_result_stem(cfg)) + ".p")

def legacy_result_path(cfg):
    return Path(cfg.results_save_location) / (legacy_result_stem(cfg) + ".p")

def compatible_cached_paths(cfg):
    """Nhận cache tên ngắn và cache cũ nếu tên cũ hợp lệ trên hệ tệp."""
    paths = [expected_result_path(cfg)]
    legacy_path = legacy_result_path(cfg)
    # ``.meta.json`` được ghi cùng artifact; giữ biên nhỏ hơn 255 byte.
    if (legacy_path != paths[0]
            and len((legacy_path.name + ".meta.json").encode("utf-8")) <= 255):
        paths.append(legacy_path)
    return paths

def sidecar_path(artifact_path):
    return Path(str(artifact_path) + ".meta.json")

def source_code_fingerprints():
    paths = {
        "drmd_environment": DRMD_ROOT / "DRMD/environment.py",
        "drmd_base": DRMD_ROOT / "DRMD/base.py",
        "drmd_classifier_utils": DRMD_ROOT / "DRMD/utils/classifier_utils.py",
        "drmd_config": DRMD_ROOT / "DRMD/config.py",
        "drmd_agents": DRMD_ROOT / "DRMD/agents.py",
        "tesseract_temporal": TESSERACT_ROOT / "tesseract/temporal.py",
        "tesseract_metrics": TESSERACT_ROOT / "tesseract/metrics.py",
        "tesseract_selection": TESSERACT_ROOT / "tesseract/selection.py",
        "tesseract_evaluation": TESSERACT_ROOT / "tesseract/evaluation.py",
        "balanced_hard_replay": BHR_MODULE_PATH,
    }
    missing_sources = {
        name: str(path) for name, path in paths.items() if not path.is_file()
    }
    if missing_sources:
        raise FileNotFoundError(
            "Thiếu tệp nguồn bắt buộc để lập chữ ký: "
            + json.dumps(missing_sources, ensure_ascii=False))
    fingerprints = {
        name: sha256_file(path) for name, path in paths.items()
    }
    fingerprints["notebook_runtime_patch"] = PATCH_IMPLEMENTATION_FINGERPRINT
    fingerprints["notebook_data_pipeline"] = (
        DATA_PIPELINE_IMPLEMENTATION_FINGERPRINT)
    return fingerprints

SOURCE_CODE_FINGERPRINTS = source_code_fingerprints()

def run_spec(cfg, label, track, variant, policy=None):
    return {
        "label": label,
        "variant_name": variant["name"],
        "track": track,
        "policy": policy["name"] if policy else "external",
        "policy_scope": policy.get("scope") if policy else "primary",
        "fn_penalty": float(variant["fn_penalty"]),
        "audit_sampling": bool(variant.get("audit_sampling", False)),
        "adaptive_fn_penalty": bool(variant.get("adaptive_fn_penalty", False)),
        "balanced_hard_replay": bool(variant.get("balanced_hard_replay", False)),
        "bhr_memory_cap": STRATEGY_FINETUNING_SIZE,
        "bhr_fn_fraction": BHR_FN_FRACTION,
        "bhr_fp_fraction": BHR_FP_FRACTION,
        "bhr_background_fraction": BHR_BACKGROUND_FRACTION,
        "bhr_fn_max_repeat": BHR_FN_MAX_REPEAT,
        "bhr_fp_max_repeat": BHR_FP_MAX_REPEAT,
        "software_environment": {
            key: ENVIRONMENT_STAMP.get(key)
            for key in ("python", "torch", "numpy", "scipy",
                        "scikit_learn", "cuda",
                        "cublas_workspace_config",
                        "determinism_policy",
                        "installed_runtime_distributions")
        },
        "mp": float(cfg.minority_priority),
        "external_al_budget": int(cfg.al_rate),
        "external_reject_budget": int(cfg.reject_rate),
        "reject_cost": float(cfg.reject_cost),
        "seed": int(cfg.seed),
        "is_reject_action": bool(cfg.is_reject_action),
        "is_reject": bool(cfg.is_reject),
        "reward_rejected_outcome": bool(cfg.reward_rejected_outcome),
        "is_active": bool(cfg.is_active),
        "is_fine_tuning": bool(cfg.is_fine_tuning),
        "select_reject_labeled": bool(cfg.select_reject_labeled),
        "posthoc_augmented_AL": int(cfg.posthoc_augmented_AL),
        "integrated_reject_volume": "policy_endogenous",
        "feedback_set_rule": (
            "Q_t=all_samples" if variant.get("full_feedback", False)
            else ("Q_t=R_t" if variant.get("selector_mode") == "iral"
                  else f"Q_t={variant.get('selector_mode')}_budget_{variant.get('feedback_budget')}")),
        "al_select_mode": str(cfg.al_select_mode),
        "training_window_mode": TRAINING_WINDOW_MODE,
        "temporal_reference_policy": FIXED_TEMPORAL_REFERENCE_POLICY,
        "training_window": TRAINING_WINDOW,
        "testing_window": TESTING_WINDOW,
        "granularity": GRANULARITY,
        "training_epochs": int(cfg.training_epochs),
        "update_epochs": int(cfg.update_epochs),
        "minibatch_size": int(cfg.minibatch_size),
        "layer_size": int(cfg.layer_size),
        "hidden_layers_parameter": int(cfg.hidden_layers),
        # PPO_Network tạo một lớp Linear đầu vào rồi lặp hidden_layers lần.
        "actual_hidden_linear_layers": int(cfg.hidden_layers) + 1,
        "expected_periods": int(EXPECTED_PERIODS),
        "expected_test_samples": int(EXPECTED_TEST_SAMPLES),
        "bhr_error_partition_prediction": "binary_argmax_head",
        "policy_action_space": [0, 1, 2],
        "reject_action_index": 2,
        "feedback_selector_mode": str(variant.get("selector_mode", "iral")),
        "feedback_budget": int(variant.get("feedback_budget", 0)),
        "candidate_multiplier": int(variant.get("candidate_multiplier", 1)),
        "full_feedback": bool(variant.get("full_feedback", False)),
        "training_history_cap": getattr(cfg, "_training_history_cap", None),
        "training_history_policy": (
            RECENCY_POLICY if getattr(cfg, "_training_history_cap", None) is not None
            else None),
        "feedback_timing": (
            "predict_t_then_fit_t_plus_1"
            if variant.get("full_feedback", False)
            else "query_after_predict_t_then_fit_t_plus_1"),
        "single_classifier_initialization": True,
    }

SIGNATURE_SCHEMA_VERSION = 7
def _signature_for_components(spec, dataset_fingerprint, split_fingerprint,
                              source_fingerprints, parameter_provenance):
    payload = {
        "protocol_version": PROTOCOL_VERSION,
        "dataset_source_fingerprint": dataset_fingerprint,
        "split_fingerprint": split_fingerprint,
        "parameter_provenance": parameter_provenance,
        "source_code_fingerprints": source_fingerprints,
        "run_spec": spec,
    }
    return hashlib.sha256(canonical_json(payload).encode()).hexdigest()

def signature_for_spec(spec):
    return _signature_for_components(
        spec, DATASET_SOURCE_FINGERPRINT, SPLIT_FINGERPRINT,
        SOURCE_CODE_FINGERPRINTS, PARAMETER_PROVENANCE)

def _signature_from_sidecar_snapshot(metadata):
    """Tái tạo chữ ký gốc mà không thay dấu vết đã lưu trong sidecar."""
    return _signature_for_components(
        metadata["run_spec"], metadata["dataset_source_fingerprint"],
        metadata["split_fingerprint"], metadata["source_code_fingerprints"],
        metadata.get("parameter_provenance", PARAMETER_PROVENANCE))

def _as_index_vector(values, period, name):
    raw = np.asarray(values)
    if raw.ndim != 1:
        raise ValueError(f"Chu kỳ {period}: {name} phải là vector một chiều.")
    if raw.size and not np.all(np.equal(raw, raw.astype(np.int64))):
        raise ValueError(f"Chu kỳ {period}: {name} chứa chỉ số không nguyên.")
    indexes = raw.astype(np.int64, copy=False)
    if np.unique(indexes).size != indexes.size:
        raise ValueError(f"Chu kỳ {period}: {name} chứa chỉ số trùng.")
    return indexes

def _as_time_vector(values, period, name):
    try:
        return np.asarray(values).reshape(-1).astype("datetime64[ns]")
    except Exception as exc:
        raise ValueError(
            f"Chu kỳ {period}: {name} không chuyển được sang thời gian chuẩn.") from exc

def _same_float(observed, expected, atol=1e-10):
    try:
        left, right = float(observed), float(expected)
    except (TypeError, ValueError):
        return False
    if np.isnan(left) and np.isnan(right):
        return True
    return bool(np.isclose(left, right, rtol=0.0, atol=atol))

def _confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.int64).reshape(-1)
    return {
        "tp": int(np.sum((y_true == 1) & (y_pred == 1))),
        "tn": int(np.sum((y_true == 0) & (y_pred == 0))),
        "fp": int(np.sum((y_true == 0) & (y_pred == 1))),
        "fn": int(np.sum((y_true == 1) & (y_pred == 0))),
    }

def validate_result_object(result, expected_periods, policy=None, variant=None):
    if not isinstance(result, dict):
        raise TypeError("Artifact DRMD phải là dict.")
    missing = sorted(REQUIRED_RESULT_SERIES - set(result))
    if missing:
        raise ValueError(f"Artifact thiếu khóa: {missing}")
    wrong_lengths = {}
    for key in REQUIRED_RESULT_SERIES:
        if not hasattr(result[key], "__len__"):
            wrong_lengths[key] = "không có độ dài"
        elif len(result[key]) != expected_periods:
            wrong_lengths[key] = len(result[key])
    if wrong_lengths:
        raise ValueError(
            f"Số chu kỳ không đúng: {wrong_lengths}; kỳ vọng {expected_periods}")
    for key in ("selection_history", "temporal_reference_history",
                "strategy_runtime_config"):
        if key not in result:
            raise ValueError(f"Artifact thiếu khóa cấp lượt chạy: {key}")
    if len(result["selection_history"]) != expected_periods:
        raise ValueError("selection_history sai số chu kỳ.")
    if len(result["temporal_reference_history"]) != expected_periods:
        raise ValueError("temporal_reference_history sai số lần fit.")

    runtime = result["strategy_runtime_config"]
    if not isinstance(runtime, dict):
        raise TypeError("strategy_runtime_config phải là dict.")
    expected_runtime = {
        "feedback_timing": "predict_t_then_fit_t_plus_1",
        "single_classifier_initialization": True,
        "temporal_reference_policy": FIXED_TEMPORAL_REFERENCE_POLICY,
        "temporal_reference_period": TRAINING_WINDOW,
        "training_history_cap": STRATEGY_FINETUNING_SIZE,
        "bhr_error_partition_prediction": "binary_argmax_head",
        "policy_action_space": [0, 1, 2],
        "reject_action_index": 2,
    }
    if variant is not None:
        expected_runtime.update({
            "feedback_selector_mode": str(variant["selector_mode"]),
            "feedback_budget": int(variant["feedback_budget"]),
            "candidate_multiplier": int(variant["candidate_multiplier"]),
            "adaptive_fn_penalty": bool(
                variant.get("adaptive_fn_penalty", False)),
            "audit_sampling": bool(variant.get("audit_sampling", False)),
            "balanced_hard_replay": bool(
                variant.get("balanced_hard_replay", False)),
            "full_feedback": bool(variant.get("full_feedback", False)),
            "bhr_fn_fraction": float(BHR_FN_FRACTION),
            "bhr_fp_fraction": float(BHR_FP_FRACTION),
            "bhr_fn_max_repeat": int(BHR_FN_MAX_REPEAT),
            "bhr_fp_max_repeat": int(BHR_FP_MAX_REPEAT),
        })
    for key, expected in expected_runtime.items():
        if runtime.get(key) != expected:
            raise ValueError(
                f"strategy_runtime_config sai {key}: "
                f"{runtime.get(key)!r} != {expected!r}")

    adaptive = bool(runtime.get("adaptive_fn_penalty", False))
    audit_sampling = bool(runtime.get("audit_sampling", False))
    bhr_enabled = bool(runtime.get("balanced_hard_replay", False))
    feedback_budget = int(runtime.get("feedback_budget", 0))
    if adaptive != audit_sampling:
        raise ValueError("V13 yêu cầu bộ điều khiển FN và mẫu kiểm toán cùng bật/tắt.")

    total_test_samples = 0
    for period in range(expected_periods):
        canonical_y = np.asarray(y_tests[period], dtype=np.int64).reshape(-1)
        canonical_t = _as_time_vector(t_tests[period], period, "t_tests chuẩn")
        canonical_total = int(X_tests[period].shape[0])
        if not (canonical_total == canonical_y.size == canonical_t.size):
            raise ValueError(f"Chu kỳ {period}: split chuẩn X/y/t lệch số hàng.")
        total_test_samples += canonical_total

        values = {
            key: int(result[key][period])
            for key in ("tp", "tn", "fp", "fn", "p", "n", "tot",
                        "selected", "rejected")
        }
        if min(values.values()) < 0:
            raise ValueError(f"Chu kỳ {period}: số đếm âm {values}")
        if values["tp"] + values["fn"] != values["p"]:
            raise ValueError(f"Chu kỳ {period}: TP + FN != P")
        if values["tn"] + values["fp"] != values["n"]:
            raise ValueError(f"Chu kỳ {period}: TN + FP != N")
        if (values["tp"] + values["tn"] + values["fp"] + values["fn"]
                != values["tot"]):
            raise ValueError(f"Chu kỳ {period}: ma trận nhầm lẫn != total accepted")
        if values["tot"] + values["rejected"] != canonical_total:
            raise ValueError(
                f"Chu kỳ {period}: tổng trước từ chối "
                f"{values['tot'] + values['rejected']} != {canonical_total}")

        selected = _as_index_vector(
            result["selected_indexes_all"][period], period, "selected indexes")
        rejected = _as_index_vector(
            result["rejected_indexes_all"][period], period, "rejected indexes")
        audit = _as_index_vector(
            result["audit_indexes_all"][period], period, "audit indexes")
        if selected.size != values["selected"] or rejected.size != values["rejected"]:
            raise ValueError(f"Chu kỳ {period}: số đếm chỉ số selected/rejected sai.")
        for name, indexes in (("selected", selected), ("rejected", rejected),
                              ("audit", audit)):
            if np.any((indexes < 0) | (indexes >= canonical_total)):
                raise ValueError(
                    f"Chu kỳ {period}: {name} vượt miền [0, {canonical_total}).")
        if np.setdiff1d(audit, selected).size:
            raise ValueError(f"Chu kỳ {period}: audit không phải tập con của selected.")

        binary_y = np.asarray(
            result["binary_y_true_all"][period], dtype=np.int64).reshape(-1)
        binary_pred_raw = np.asarray(result["binary_y_preds_all"][period]).reshape(-1)
        binary_t = _as_time_vector(
            result["binary_t_all"][period], period, "binary_t_all")
        if not (binary_y.size == binary_pred_raw.size == binary_t.size
                == canonical_total):
            raise ValueError(f"Chu kỳ {period}: dãy nhị phân sai số hàng.")
        if not np.array_equal(binary_y, canonical_y):
            raise ValueError(f"Chu kỳ {period}: binary_y_true lệch nhãn split chuẩn.")
        if not np.array_equal(binary_t, canonical_t):
            raise ValueError(f"Chu kỳ {period}: binary_t lệch thời gian split chuẩn.")
        if not np.isin(binary_pred_raw, [0, 1]).all():
            raise ValueError(f"Chu kỳ {period}: dự đoán nhị phân ngoài miền 0/1.")
        binary_pred = binary_pred_raw.astype(np.int64, copy=False)

        policy_action_raw = np.asarray(
            result["policy_action_all"][period]).reshape(-1)
        if policy_action_raw.size != canonical_total:
            raise ValueError(f"Chu kỳ {period}: dãy hành động chính sách sai số hàng.")
        if not np.isin(policy_action_raw, [0, 1, 2]).all():
            raise ValueError(f"Chu kỳ {period}: hành động chính sách ngoài miền 0/1/2.")
        policy_action = policy_action_raw.astype(np.int64, copy=False)
        expected_rejected = np.flatnonzero(policy_action == 2).astype(np.int64)
        if not np.array_equal(expected_rejected, rejected):
            raise ValueError(
                f"Chu kỳ {period}: chỉ số từ chối không khớp hành động 2.")

        kept = np.setdiff1d(
            np.arange(canonical_total, dtype=np.int64), rejected,
            assume_unique=True)
        accepted_y = np.asarray(result["y_tests"][period], dtype=np.int64).reshape(-1)
        accepted_pred_raw = np.asarray(result["y_preds"][period]).reshape(-1)
        accepted_t = _as_time_vector(
            result["t_tests"][period], period, "accepted t_tests")
        if not (accepted_y.size == accepted_pred_raw.size == accepted_t.size
                == values["tot"] == kept.size):
            raise ValueError(f"Chu kỳ {period}: dãy accepted sai số hàng.")
        if not np.isin(accepted_pred_raw, [0, 1]).all():
            raise ValueError(f"Chu kỳ {period}: dự đoán accepted ngoài miền 0/1.")
        accepted_pred = accepted_pred_raw.astype(np.int64, copy=False)
        if not np.array_equal(accepted_y, canonical_y[kept]):
            raise ValueError(f"Chu kỳ {period}: nhãn accepted lệch split chuẩn.")
        if not np.array_equal(accepted_t, canonical_t[kept]):
            raise ValueError(f"Chu kỳ {period}: thời gian accepted lệch split chuẩn.")
        if not np.array_equal(accepted_pred, binary_pred[kept]):
            raise ValueError(
                f"Chu kỳ {period}: dự đoán nhị phân và nhánh accepted không khớp.")
        recomputed = _confusion_counts(accepted_y, accepted_pred)
        if any(recomputed[key] != values[key] for key in recomputed):
            raise ValueError(
                f"Chu kỳ {period}: ma trận nhầm lẫn lưu {values} "
                f"khác ma trận tính lại {recomputed}.")

        expected_rejected_y = canonical_y[rejected]
        stored_rejected_y = np.asarray(
            result["rejected_y_true"][period], dtype=np.int64).reshape(-1)
        if not np.array_equal(stored_rejected_y, expected_rejected_y):
            raise ValueError(f"Chu kỳ {period}: nhãn rejected sai.")
        rejected_malware = int(np.sum(expected_rejected_y == 1))
        rejected_benign = int(np.sum(expected_rejected_y == 0))
        if (int(result["rejected_malware_count"][period]) != rejected_malware
                or int(result["rejected_benign_count"][period]) != rejected_benign):
            raise ValueError(f"Chu kỳ {period}: số lớp rejected sai.")

        expected_selected_y = canonical_y[selected]
        stored_selected_y = np.asarray(
            result["selected_y_true"][period], dtype=np.int64).reshape(-1)
        if not np.array_equal(stored_selected_y, expected_selected_y):
            raise ValueError(f"Chu kỳ {period}: nhãn selected sai.")
        selected_malware = int(np.sum(expected_selected_y == 1))
        selected_benign = int(np.sum(expected_selected_y == 0))
        if (int(result["selected_malware_count"][period]) != selected_malware
                or int(result["selected_benign_count"][period]) != selected_benign):
            raise ValueError(f"Chu kỳ {period}: số lớp selected sai.")

        expected_audit_positive = int(np.sum(canonical_y[audit] == 1))
        expected_audit_fn = int(np.sum(
            (canonical_y[audit] == 1) & (binary_pred[audit] == 0)))
        if (int(result["audit_count"][period]) != audit.size
                or int(result["audit_positive_count"][period])
                != expected_audit_positive
                or int(result["audit_fn_count"][period]) != expected_audit_fn):
            raise ValueError(f"Chu kỳ {period}: thống kê mẫu kiểm toán sai.")
        total_budget = min(feedback_budget, canonical_total)
        expected_audit_count = (
            min(total_budget, max(1, int(np.ceil(np.sqrt(total_budget)))))
            if audit_sampling and total_budget > 0 else 0)
        if audit.size != expected_audit_count:
            raise ValueError(
                f"Chu kỳ {period}: audit={audit.size}, "
                f"kỳ vọng {expected_audit_count}.")

        if policy is not None:
            name = policy["name"]
            if name == "IR" and selected.size:
                raise ValueError(f"Chu kỳ {period}: IR không được chọn mẫu huấn luyện")
            if name == "IRAL" and not np.array_equal(selected, rejected):
                raise ValueError(f"Chu kỳ {period}: IRAL phải dùng đúng tập bị từ chối")
            if name == "IRAAL" and selected.size != min(
                    int(policy["budget"]), canonical_total):
                raise ValueError(
                    f"Chu kỳ {period}: IRAAL chọn {selected.size}, "
                    f"kỳ vọng {min(int(policy['budget']), canonical_total)}")
            if name == "FFCR":
                expected_all = np.arange(canonical_total, dtype=np.int64)
                if not np.array_equal(selected, expected_all):
                    raise ValueError(
                        f"Chu kỳ {period}: FFCR không phủ toàn bộ mẫu.")

        diagnostic = result["selection_diagnostics"][period]
        if not isinstance(diagnostic, dict):
            raise TypeError(f"Chu kỳ {period}: selection diagnostic không phải dict.")
        if (int(diagnostic.get("n_rows", -1)) != canonical_total
                or int(diagnostic.get("n_selected", -1)) != selected.size
                or int(diagnostic.get("n_rejected_total", -1)) != rejected.size
                or bool(diagnostic.get("selection_used_labels", True))):
            raise ValueError(f"Chu kỳ {period}: selection diagnostic sai.")
        history = result["selection_history"][period]
        if (not isinstance(history, dict)
                or int(history.get("period_index", -1)) != period
                or int(history.get("n_selected", -1)) != selected.size
                or int(history.get("n_rejected_total", -1)) != rejected.size):
            raise ValueError(f"Chu kỳ {period}: selection_history sai.")

        telemetry = result["bhr_telemetry"][period]
        if not isinstance(telemetry, dict):
            raise TypeError(f"Chu kỳ {period}: BHR telemetry không phải dict.")
        if (bool(telemetry.get("enabled", False)) != bhr_enabled
                or int(telemetry.get("fit_period", -1)) != period + 1
                or not bool(telemetry.get("causal_lag_ok", False))):
            raise ValueError(f"Chu kỳ {period}: cờ BHR/nhân quả sai.")
        memory_rows = int(telemetry.get("memory_rows", -1))
        if memory_rows != STRATEGY_FINETUNING_SIZE:
            raise ValueError(
                f"Chu kỳ {period}: bộ nhớ {memory_rows}, "
                f"kỳ vọng {STRATEGY_FINETUNING_SIZE} hàng.")
        if bhr_enabled:
            if int(telemetry.get("memory_cap", -1)) != STRATEGY_FINETUNING_SIZE:
                raise ValueError(f"Chu kỳ {period}: memory_cap BHR sai.")
            component_rows = sum(int(telemetry.get(key, -1)) for key in (
                "fn_rows", "fp_rows", "background_rows"))
            if component_rows != memory_rows:
                raise ValueError(f"Chu kỳ {period}: thành phần BHR không cộng đủ.")
            if int(telemetry.get("newest_source_period", -1)) != period:
                raise ValueError(f"Chu kỳ {period}: BHR dùng phản hồi cùng/tương lai.")
            fn_replay = int(telemetry.get("fn_replay_rows", -1))
            fp_replay = int(telemetry.get("fp_replay_rows", -1))
            unique_rows = int(telemetry.get("unique_rows_used", -1))
            if (fn_replay < 0 or fp_replay < 0
                    or not 0 < unique_rows <= memory_rows):
                raise ValueError(f"Chu kỳ {period}: số hàng replay/duy nhất sai.")
            replay_fraction = float(telemetry.get("replay_fraction", np.nan))
            expected_replay_fraction = (fn_replay + fp_replay) / memory_rows
            if (not np.isfinite(replay_fraction)
                    or not _same_float(replay_fraction, expected_replay_fraction)):
                raise ValueError(f"Chu kỳ {period}: replay_fraction không hợp lệ.")
            mean_age = float(telemetry.get("mean_labeled_age_periods", np.nan))
            max_age = float(telemetry.get("max_labeled_age_periods", np.nan))
            if (not np.isfinite(mean_age) or not np.isfinite(max_age)
                    or mean_age < 1.0 or max_age < mean_age):
                raise ValueError(f"Chu kỳ {period}: tuổi phản hồi không hợp lệ.")

        temporal = result["temporal_reference_history"][period]
        test_month = canonical_t[0].astype("datetime64[M]")
        year = int(test_month.astype("datetime64[Y]").astype(int) + 1970)
        month = int(test_month.astype(int) % 12 + 1)
        test_absolute_month = year * 12 + month
        if (int(temporal.get("fit_index", -1)) != period
                or int(temporal.get("reference_period", -1)) != TRAINING_WINDOW
                or int(temporal.get("memory_period", 0)) <= 0
                or int(temporal.get("memory_end", test_absolute_month))
                >= test_absolute_month):
            raise ValueError(f"Chu kỳ {period}: dấu vết thời gian vi phạm nhân quả.")
        factors = [float(temporal.get(key, np.nan)) for key in (
            "factor_min", "factor_max", "factor_mean")]
        if not np.isfinite(factors).all() or factors[0] > factors[1]:
            raise ValueError(f"Chu kỳ {period}: hệ số thời gian không hợp lệ.")

    if total_test_samples != EXPECTED_TEST_SAMPLES:
        raise ValueError(
            f"Mỗi lượt có {total_test_samples} mẫu, "
            f"kỳ vọng {EXPECTED_TEST_SAMPLES}.")

    # Tái dựng chính xác bộ điều khiển FN từ thống kê audit để kiểm tra rằng
    # hệ số của tháng t chỉ phản ánh nhãn đã mở đến tháng t-1.
    dual, target, controller_step = 0.0, None, 0
    audit_history = []
    for period in range(expected_periods):
        expected_used = 1.0 + dual
        if not _same_float(result["fn_penalty_used"][period], expected_used):
            raise ValueError(f"Chu kỳ {period}: fn_penalty_used sai độ trễ.")
        audit_fn = int(result["audit_fn_count"][period])
        audit_positive = int(result["audit_positive_count"][period])
        audit_history.append((audit_fn, audit_positive))
        estimated = float("nan")
        if adaptive:
            total_fn = int(sum(item[0] for item in audit_history))
            total_positive = int(sum(item[1] for item in audit_history))
            minimum_support = max(2, int(np.ceil(np.sqrt(max(1, feedback_budget)))))
            if target is None and total_positive >= minimum_support:
                target = (total_fn + 0.5) / (total_positive + 1.0)
            elif target is not None:
                recent = audit_history[-max(1, TRAINING_WINDOW):]
                recent_fn = int(sum(item[0] for item in recent))
                recent_positive = int(sum(item[1] for item in recent))
                if recent_positive > 0:
                    estimated = (recent_fn + 0.5) / (recent_positive + 1.0)
                    controller_step += 1
                    dual = max(
                        0.0,
                        dual + (estimated - target) / np.sqrt(controller_step))
        expected_target = float(target) if target is not None else float("nan")
        if not _same_float(result["afnp_target_fnr"][period], expected_target):
            raise ValueError(f"Chu kỳ {period}: FNR mục tiêu lưu sai.")
        if not _same_float(result["afnp_estimated_fnr"][period], estimated):
            raise ValueError(f"Chu kỳ {period}: FNR audit ước lượng lưu sai.")
        if not _same_float(result["fn_penalty_next"][period], 1.0 + dual):
            raise ValueError(f"Chu kỳ {period}: fn_penalty_next sai.")

    return {
        "periods": int(expected_periods),
        "test_samples": int(total_test_samples),
        "causal_lag_violations": 0,
        "memory_cap_violations": 0,
        "canonical_split_alignment": True,
        "confusion_counts_recomputed": True,
    }

def validate_result_file(path, expected_periods, policy=None, variant=None):
    with open(path, "rb") as handle:
        result = pickle.load(handle)
    return validate_result_object(
        result, expected_periods, policy=policy, variant=variant)

def _non_patch_fingerprint_differences(stored, current):
    ignored = {"notebook_runtime_patch"}
    keys = (set(stored) | set(current)) - ignored
    return sorted(key for key in keys if stored.get(key) != current.get(key))

def validate_sidecar(artifact_path, signature):
    metadata_path = sidecar_path(artifact_path)
    if not metadata_path.exists():
        raise FileNotFoundError("Thiếu sidecar của artifact giao thức V13")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("status") != "complete":
        raise ValueError("Sidecar không có trạng thái complete")
    if metadata.get("protocol_version") != PROTOCOL_VERSION:
        raise ValueError("Sai protocol_version")
    if int(metadata.get("signature_schema_version", -1)) != SIGNATURE_SCHEMA_VERSION:
        raise ValueError("Sai signature_schema_version")
    if metadata.get("dataset_source_fingerprint") != DATASET_SOURCE_FINGERPRINT:
        raise ValueError("Sai dataset_source_fingerprint")
    if metadata.get("split_fingerprint") != SPLIT_FINGERPRINT:
        raise ValueError("Sai split_fingerprint")
    if metadata.get("source_code_fingerprints") != SOURCE_CODE_FINGERPRINTS:
        raise ValueError("Sai source_code_fingerprints")
    if metadata.get("parameter_provenance") != PARAMETER_PROVENANCE:
        raise ValueError("Sai parameter_provenance")
    if metadata.get("run_signature") != signature:
        raise ValueError("Sai run_signature; V13 không nhận cache giao thức cũ")
    if metadata.get("artifact_sha256") != sha256_file(artifact_path):
        raise ValueError("Sai SHA-256 của artifact")
    if metadata.get("run_signature") != signature_for_spec(metadata["run_spec"]):
        raise ValueError("run_spec của sidecar không khớp cấu hình đang yêu cầu")
    if int(metadata["run_spec"].get("expected_periods", -1)) != EXPECTED_TEST_PERIODS:
        raise ValueError("run_spec sai số chu kỳ kiểm thử")
    if int(metadata["run_spec"].get("expected_test_samples", -1)) != EXPECTED_TEST_SAMPLES:
        raise ValueError("run_spec sai số mẫu kiểm thử")
    validation = metadata.get("validation", {})
    if (int(validation.get("periods", -1)) != EXPECTED_TEST_PERIODS
            or int(validation.get("test_samples", -1)) != EXPECTED_TEST_SAMPLES
            or not bool(validation.get("canonical_split_alignment", False))
            or int(validation.get("causal_lag_violations", -1)) != 0
            or int(validation.get("memory_cap_violations", -1)) != 0):
        raise ValueError("Sidecar thiếu bằng chứng toàn vẹn V13")
    return metadata

def quarantine_path(path, reason):
    if not path.exists():
        return None
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    try:
        relative = path.relative_to(OUT)
    except ValueError:
        relative = Path(path.name)
    target = QUARANTINE_OUT / timestamp / uuid.uuid4().hex[:8] / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(target))
    append_jsonl(QUARANTINE_LOG, {
        "time_utc": datetime.now(timezone.utc).isoformat(),
        "source": str(path), "target": str(target), "reason": str(reason),
    })
    return target

def quarantine_artifact(artifact_path, reason):
    moved = quarantine_path(artifact_path, reason)
    metadata_path = sidecar_path(artifact_path)
    if metadata_path.exists():
        quarantine_path(metadata_path, reason)
    return moved

def cache_is_valid(artifact_path, signature, policy, variant, require_sidecar):
    if not artifact_path.exists():
        return False
    try:
        validate_result_file(
            artifact_path, EXPECTED_PERIODS, policy=policy, variant=variant)
        if require_sidecar:
            validate_sidecar(artifact_path, signature)
        elif sidecar_path(artifact_path).exists():
            validate_sidecar(artifact_path, signature)
        return True
    except Exception as exc:
        print(f"⚠️ Cache không hợp lệ, chuyển vùng cách ly: {artifact_path.name}: {exc}")
        quarantine_artifact(artifact_path, f"{type(exc).__name__}: {exc}")
        return False

def audit_reward_weights(variant):
    """Chỉ kiểm tra nhãn huấn luyện ban đầu; không đọc nhãn kiểm thử tương lai."""
    benign, malware = int(np.sum(y_train == 0)), int(np.sum(y_train == 1))
    ratio = benign / malware if malware else float("inf")
    effective_fn_weight = variant["minority_priority"] * variant["fn_penalty"]
    print({"variant": variant["name"], "initial_train_benign": benign,
           "initial_train_malware": malware, "benign_to_malware_ratio": ratio,
           "mp": variant["minority_priority"], "fn_penalty": variant["fn_penalty"],
           "effective_fn_reward_weight": effective_fn_weight})

def write_variant_metadata(raw_dir, variant, operating_values, track, policy=None):
    raw_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "experiment": EXPERIMENT_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "strategy_run_tag": STRATEGY_RUN_TAG if track == "strategy" else None,
                "dataset_years": DATASET_YEARS,
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "training_window_mode": TRAINING_WINDOW_MODE,
        "temporal_reference_policy": FIXED_TEMPORAL_REFERENCE_POLICY,
        "training_window": TRAINING_WINDOW,
        "testing_window": TESTING_WINDOW,
        "granularity": GRANULARITY,
        "feature_dim": int(X.shape[1]),
        "initial_train_benign": int(np.sum(y_train == 0)),
        "initial_train_malware": int(np.sum(y_train == 1)),
        "raw_artifact_directory": str(raw_dir),
        "track": track,
        "variant": variant,
        "policy": policy,
        "operating_values": operating_values,
        "seeds": SEEDS,
        "mp_interpretation": "hệ số phần thưởng của nhãn mã độc",
        "effective_fn_weight": variant["minority_priority"] * variant["fn_penalty"],
        "source_code_fingerprints": SOURCE_CODE_FINGERPRINTS,
    }
    atomic_json(raw_dir / "variant_metadata.json", payload)

def build_strategy_plan():
    """Cấu hình chiến lược và kiểm tra chi phí IRAL, ghép theo cùng seed."""
    records = []
    dataset_path = (
        f"{DATA_OUT},{DATASET_ARTIFACT_TAG}-X.p,{DATASET_ARTIFACT_TAG}-y.p,"
        f"{DATASET_ARTIFACT_TAG}-t.p,{X.shape[1]}")
    for variant in STRATEGY_VARIANTS:
        policy = variant["policy"]
        raw_dir = STRATEGY_RAW_OUT / variant["name"]
        audit_reward_weights(variant)
        write_variant_metadata(
            raw_dir, variant,
            [{"reject_cost": variant["reject_cost"],
              "selector_mode": variant["selector_mode"],
              "feedback_budget": variant["feedback_budget"],
              "feedback_timing": "after-prediction; update-from-next-month"}],
            "strategy", policy=policy)
        settings = Settings(
            dataset=[dataset_path], model=[variant["model"]], seed=SEEDS,
            is_active=[True], al_select_mode=[AL_SELECTION_MODE],
            is_reject=[True], is_reject_action=[True],
            al_rate=[0], reject_rate=[0], reject_cost=[variant["reject_cost"]],
            reward_rejected_outcome=[True], select_reject_labeled=[True],
            posthoc_augmented_AL=[0],
            minority_priority=[MP_FIXED], majority_priority=[1.0],
            temporal_rewards=[True], temporal_scaling=[6.0],
            is_fine_tuning=[True], training_epochs=[DRMD_TRAINING_EPOCHS],
            finetuning_size=[STRATEGY_FINETUNING_SIZE])
        for cfg in generate_experiment_configs(settings):
            cfg.results_save_location = str(raw_dir) + "/"
            cfg.training_window, cfg.testing_window = TRAINING_WINDOW, TESTING_WINDOW
            cfg.granularity, cfg.cuda, cfg.mps = GRANULARITY, True, False
            cfg._fn_penalty = 1.0
            cfg._adaptive_fn_penalty = bool(
                variant.get('adaptive_fn_penalty', False))
            cfg._afnp_audit_sampling = bool(
                variant.get('audit_sampling', False))
            cfg._afnp_budget = int(variant.get('feedback_budget', 0))
            cfg._afnp_seed = int(cfg.seed)
            cfg._afnp_window = int(TRAINING_WINDOW)
            cfg._feedback_selector_mode = variant["selector_mode"]
            cfg._feedback_budget = variant["feedback_budget"]
            cfg._candidate_multiplier = variant["candidate_multiplier"]
            cfg._full_feedback = variant["full_feedback"]
            cfg._training_history_cap = STRATEGY_FINETUNING_SIZE
            cfg._balanced_hard_replay = bool(variant.get('balanced_hard_replay', False))
            cfg._bhr_fn_fraction = BHR_FN_FRACTION
            cfg._bhr_fp_fraction = BHR_FP_FRACTION
            cfg._bhr_fn_max_repeat = BHR_FN_MAX_REPEAT
            cfg._bhr_fp_max_repeat = BHR_FP_MAX_REPEAT
            artifact_path = expected_result_path(cfg)
            artifact_name_bytes = len((artifact_path.name + ".meta.json").encode("utf-8"))
            if artifact_name_bytes > ARTIFACT_FILENAME_MAX_BYTES:
                raise OSError(
                    f"Tên artifact {artifact_path.name!r} dài {artifact_name_bytes} byte.")
            label = f"strategy::{variant['name']}::{policy['name']}::Seed{cfg.seed}"
            records.append({
                "cfg": cfg, "label": label, "track": "strategy",
                "variant": variant, "policy": policy, "require_sidecar": True,
            })
    order = {variant["name"]: index for index, variant in enumerate(STRATEGY_VARIANTS)}
    return sorted(records, key=lambda row: (
        order[row["variant"]["name"]], int(row["cfg"].seed)))

def build_session_plan():
    if not RUN_STRATEGY_EXPERIMENT or KAGGLE_RUN_PHASE == "aggregate_only":
        return []
    return build_strategy_plan()

def stage_and_promote(record):
    global CURRENT_RUN_TAG, CURRENT_PERIOD_IDX
    cfg, label = record["cfg"], record["label"]
    policy = record["policy"]
    spec = run_spec(cfg, label, record["track"], record["variant"], policy=policy)
    signature = signature_for_spec(spec)
    final_path = expected_result_path(cfg)

    for cached_path in compatible_cached_paths(cfg):
        if cache_is_valid(
                cached_path, signature, policy, record["variant"],
                record["require_sidecar"]):
            write_progress_event("skip_validated_cache", cfg, {
                "label": label, "path": str(cached_path), "run_signature": signature})
            print("⏩ Cache hợp lệ:", cached_path.name)
            return "cached"

    stage_dir = STAGING_OUT / signature
    if stage_dir.exists():
        quarantine_path(stage_dir, "staging dở dang từ phiên trước")
    stage_dir.mkdir(parents=True, exist_ok=False)
    stage_cfg = deepcopy(cfg)
    stage_cfg.results_save_location = str(stage_dir) + "/"
    staged_path = expected_result_path(stage_cfg)

    co_dinh_hat_giong(int(cfg.seed))
    CURRENT_RUN_TAG, CURRENT_PERIOD_IDX = label, -1
    write_progress_event("start_run", cfg, {
        "label": label, "staging_path": str(staged_path),
        "final_path": str(final_path), "run_signature": signature})
    gpu_available_for_run = bool(torch.cuda.is_available())
    gpu_device_name = (torch.cuda.get_device_name(0)
                       if gpu_available_for_run else None)
    if gpu_available_for_run:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    run_started_monotonic = time.perf_counter()
    run_exception = None
    try:
        run(stage_cfg)
    except Exception as exc:
        # DRMD gốc có thể ném lỗi ở bước in AUT sau khi đã ghi pickle. Chỉ chấp
        # nhận trường hợp đó khi artifact đã vượt toàn bộ kiểm tra cấu trúc.
        run_exception = exc

    try:
        validation = validate_result_file(
            staged_path, EXPECTED_PERIODS, policy=policy,
            variant=record["variant"])
    except Exception:
        if run_exception is not None:
            raise run_exception
        raise

    if run_exception is not None:
        write_progress_event("post_save_exception_ignored", cfg, {
            "label": label, "error_type": type(run_exception).__name__,
            "message": str(run_exception)[:500]})

    if gpu_available_for_run:
        torch.cuda.synchronize()
    run_wall_seconds = time.perf_counter() - run_started_monotonic
    gpu_peak_allocated_bytes = (
        int(torch.cuda.max_memory_allocated()) if gpu_available_for_run else None)
    gpu_peak_reserved_bytes = (
        int(torch.cuda.max_memory_reserved()) if gpu_available_for_run else None)
    artifact_bytes = int(staged_path.stat().st_size)
    metadata = {
        "status": "complete",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "protocol_version": PROTOCOL_VERSION,
        "run_signature": signature,
        "artifact_sha256": sha256_file(staged_path),
        "artifact_naming": {
            "scheme": "sha256_20_of_legacy_drmd_stem",
            "stored_stem": final_path.stem,
            "legacy_stem": legacy_result_stem(cfg),
        },
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "signature_schema_version": SIGNATURE_SCHEMA_VERSION,
        "parameter_provenance": PARAMETER_PROVENANCE,
        "source_code_fingerprints": SOURCE_CODE_FINGERPRINTS,
        "run_spec": spec,
        "raw_artifact": str(final_path.relative_to(OUT)),
        "environment": ENVIRONMENT_STAMP,
        "run_wall_seconds": float(run_wall_seconds),
        "gpu_available": gpu_available_for_run,
        "gpu_device_name": gpu_device_name,
        "gpu_peak_allocated_bytes": gpu_peak_allocated_bytes,
        "gpu_peak_reserved_bytes": gpu_peak_reserved_bytes,
        "artifact_bytes": artifact_bytes,
        "validation": validation,
    }
    atomic_json(sidecar_path(staged_path), metadata)

    final_path.parent.mkdir(parents=True, exist_ok=True)
    os.replace(staged_path, final_path)
    os.replace(sidecar_path(staged_path), sidecar_path(final_path))
    try:
        stage_dir.rmdir()
    except OSError:
        pass
    validate_result_file(
        final_path, EXPECTED_PERIODS, policy=policy,
        variant=record["variant"])
    validate_sidecar(final_path, signature)
    write_progress_event("finish_run", cfg, {
        "label": label, "path": str(final_path),
        "run_signature": signature, "artifact_sha256": metadata["artifact_sha256"],
        "run_wall_seconds": float(run_wall_seconds),
        "gpu_peak_allocated_bytes": gpu_peak_allocated_bytes,
        "gpu_peak_reserved_bytes": gpu_peak_reserved_bytes,
        "artifact_bytes": artifact_bytes})
    return "completed"

def record_failure(record, exc):
    cfg = record["cfg"]
    payload = {
        "time_utc": datetime.now(timezone.utc).isoformat(),
        "label": record["label"], "track": record["track"],
        "policy": record["policy"]["name"] if record["policy"] else "external",
        "seed": int(cfg.seed), "al": int(cfg.al_rate), "rej": int(cfg.reject_rate),
        "reject_cost": float(cfg.reject_cost),
        "fn_penalty": float(record["variant"]["fn_penalty"]),
        "error_type": type(exc).__name__, "message": str(exc)[:1000],
        "traceback": traceback.format_exc()[-6000:],
    }
    FAILED_RUNS.append(payload)
    append_jsonl(FAILURE_LOG, payload)
    write_progress_event("run_failed", cfg, payload)
    print(f"❌ {record['label']} — {type(exc).__name__}: {exc}")
    if payload["traceback"]:
        print(payload["traceback"])

SESSION_PLAN = build_session_plan()
print({"session_phase": KAGGLE_RUN_PHASE, "planned_drmd_runs": len(SESSION_PLAN),
       "launch_cutoff_hours": SESSION_LAUNCH_CUTOFF_HOURS,
       "strategy_root": str(STRATEGY_RAW_OUT),

       "policies": [IRAL_POLICY["name"], IRAAL_POLICY["name"],
                    FFCR_POLICY["name"]],
       "balanced_hard_replay_arms": ["DRMD-BHR", "DRMD-FN-BHR"]})

consecutive_failures = 0
for record in SESSION_PLAN:
    if (STOP_AFTER_DRMD_RUNS is not None
            and DRMD_RUNS_STARTED_THIS_SESSION >= STOP_AFTER_DRMD_RUNS):
        DRMD_SESSION_STOPPED = True
        SESSION_STOP_REASON = "session_run_cap"
        break
    if session_elapsed_hours() >= SESSION_LAUNCH_CUTOFF_HOURS:
        DRMD_SESSION_STOPPED = True
        SESSION_STOP_REASON = "launch_cutoff"
        break

    final_path = expected_result_path(record["cfg"])
    # Chỉ tăng bộ đếm khi artifact chưa được xác nhận là cache hợp lệ. Hàm
    # stage_and_promote vẫn tự kiểm tra lần nữa để tránh race.
    try:
        result_state = stage_and_promote(record)
        if result_state == "completed":
            DRMD_RUNS_STARTED_THIS_SESSION += 1
            DRMD_RUNS_COMPLETED_THIS_SESSION += 1
        consecutive_failures = 0
    except (KeyboardInterrupt, SystemExit):
        raise
    except Exception as exc:
        DRMD_RUNS_STARTED_THIS_SESSION += 1
        consecutive_failures += 1
        record_failure(record, exc)
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            DRMD_SESSION_STOPPED = True
            SESSION_STOP_REASON = "maximum_consecutive_failures"
            break

if FAILED_RUNS and not DRMD_SESSION_STOPPED:
    SESSION_STOP_REASON = "completed_plan_with_failed_runs"

status_payload = {
    "time_utc": datetime.now(timezone.utc).isoformat(),
    "protocol_version": PROTOCOL_VERSION,
    "phase": KAGGLE_RUN_PHASE,
    "planned_drmd_runs_this_phase": len(SESSION_PLAN),
    "runs_started_this_session": DRMD_RUNS_STARTED_THIS_SESSION,
    "runs_completed_this_session": DRMD_RUNS_COMPLETED_THIS_SESSION,
    "failures_this_session": len(FAILED_RUNS),
    "session_stopped": DRMD_SESSION_STOPPED,
    "stop_reason": SESSION_STOP_REASON,
    "elapsed_hours": round(session_elapsed_hours(), 4),
    "runtime_prior": RUNTIME_PRIOR,
}
atomic_json(RUN_STATUS_PATH, status_payload)
write_progress_event("session_training_cell_done", extra=status_payload)

if FAILED_RUNS:
    keys = sorted({key for row in FAILED_RUNS for key in row})
    with open(TABLE_OUT / "failed_runs.csv", "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader()
        writer.writerows(FAILED_RUNS)

# Xuất checkpoint ngay tại cuối ô huấn luyện, trước mọi phép tổng hợp. Vì vậy một
# bảng/hình chưa đủ dữ liệu hoặc module báo cáo lỗi không thể làm mất artifact của phiên.
import zipfile

def export_session_checkpoint_atomic():
    checkpoint_path = OUT / f"{EXPERIMENT_NAME}_checkpoint_raw.zip"
    temporary = checkpoint_path.with_name(checkpoint_path.name + ".tmp")
    roots = [STRATEGY_RAW_OUT, RAW_OUT / "static_baselines",
             RUN_PROGRESS_LOG, RUN_STATUS_PATH, FAILURE_LOG]
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=1) as archive:
        for root in roots:
            if not root.exists():
                continue
            paths = [root] if root.is_file() else sorted(root.rglob("*"))
            for path in paths:
                if path.is_file():
                    archive.write(path, path.relative_to(OUT))
    os.replace(temporary, checkpoint_path)
    print(f"Checkpoint nguyên tử: {checkpoint_path.name} "
          f"({checkpoint_path.stat().st_size / 1024**2:.2f} MiB)")
    return checkpoint_path

export_session_checkpoint_atomic()
print(status_payload)

## Ô 11 — Tổng hợp ghép cặp và cổng thành công V13


In [ ]:
# Tổng hợp V13: bảy phương pháp × mười seed và kiểm định ghép cặp.
import itertools
import json
import math
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

period_months = []
for split_row in split_rows[1:]:
    start_month, end_month = str(split_row["start"])[:7], str(split_row["end"])[:7]
    if start_month != end_month:
        raise ValueError(f"Chu kỳ {split_row['period']} đi qua hai tháng.")
    period_months.append(start_month)
if len(period_months) != EXPECTED_PERIODS:
    raise ValueError("Số tháng kiểm thử không khớp EXPECTED_PERIODS.")

def _safe_rate(numerator, denominator):
    return float(numerator / denominator) if denominator else float("nan")

RATE_SUPPORT_DIAGNOSTIC_THRESHOLD = 30

def _counts(y_true, y_pred):
    tn, fp, fn, tp = skmetrics.confusion_matrix(
        np.asarray(y_true), np.asarray(y_pred), labels=[0, 1]).ravel()
    positive_support, negative_support = int(tp + fn), int(tn + fp)
    return {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
            "positive_support": positive_support,
            "negative_support": negative_support,
            "precision": _safe_rate(int(tp), int(tp) + int(fp)),
            "recall": _safe_rate(int(tp), positive_support),
            "f1": _safe_rate(2 * int(tp), 2 * int(tp) + int(fp) + int(fn)),
            "fnr": _safe_rate(int(fn), positive_support),
            "fpr": _safe_rate(int(fp), negative_support),
            "fnr_defined": bool(positive_support > 0),
            "fpr_defined": bool(negative_support > 0),
            "low_positive_support": bool(
                positive_support < RATE_SUPPORT_DIAGNOSTIC_THRESHOLD),
            "low_negative_support": bool(
                negative_support < RATE_SUPPORT_DIAGNOSTIC_THRESHOLD),
            "single_class_period": bool(
                positive_support == 0 or negative_support == 0)}

def _aut(values, months):
    values = np.asarray(values, dtype=float)
    month_number = np.asarray([
        int(month[:4]) * 12 + int(month[5:7]) for month in months], dtype=int)
    valid = (np.isfinite(values[:-1]) & np.isfinite(values[1:])
             & (np.diff(month_number) == 1))
    return ({"value": float(np.mean((values[:-1][valid] + values[1:][valid]) / 2)),
             "valid_edges": int(valid.sum())}
            if valid.any() else {"value": float("nan"), "valid_edges": 0})

def _write_csv(path, rows):
    if not rows:
        return
    keys = []
    for row in rows:
        for key in row:
            if key not in keys:
                keys.append(key)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys, extrasaction="ignore")
        writer.writeheader(); writer.writerows(rows)

DISPLAY_NAMES = {
    "Static-MLP": "MLP tĩnh",
    "MLP-CFJR": "CFJR (MLP)",
    "DRMD-IRAL": "DRMD (IRAL)",
    "DRMD-IRAAL": "DRMD (IRAAL)",
    "DRMD-FN": "DRMD-FN",
    "DRMD-BHR": "BHR-only",
    "DRMD-FN-BHR": "DRMD-FN–BHR",
    "DRMD-FFCR": "FFCR",
}
for _budget in SENSITIVITY_BUDGETS:
    if _budget == FEEDBACK_UPDATE_BUDGET:
        continue
    DISPLAY_NAMES[f"DRMD-IRAAL-B{_budget}"] = f"DRMD (IRAAL), B={_budget}"
    DISPLAY_NAMES[f"DRMD-FN-BHR-B{_budget}"] = (
        f"DRMD-FN–BHR, B={_budget}")
for _reject_cost in SENSITIVITY_REJECT_COSTS:
    if abs(float(_reject_cost) - float(STRATEGY_REJECT_COST)) < 1e-12:
        continue
    _suffix = _cost_suffix(_reject_cost)
    DISPLAY_NAMES[f"DRMD-IRAL-{_suffix}"] = (
        f"DRMD (IRAL), R_cost={_reject_cost:g}")
    DISPLAY_NAMES[f"DRMD-IRAAL-{_suffix}"] = (
        f"DRMD (IRAAL), R_cost={_reject_cost:g}")
EXPECTED_KEYS = {(variant["name"], int(seed))
                 for variant in STRATEGY_VARIANTS for seed in SEEDS}
variant_map = {variant["name"]: variant for variant in STRATEGY_VARIANTS}
artifacts, errors, duplicates = {}, [], []
for metadata_path in sorted(STRATEGY_RAW_OUT.rglob("*.p.meta.json")):
    artifact_path = Path(str(metadata_path).removesuffix(".meta.json"))
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        spec = metadata["run_spec"]
        if metadata.get("protocol_version") != PROTOCOL_VERSION or spec.get("track") != "strategy":
            continue
        variant = variant_map.get(spec.get("variant_name"))
        if variant is None:
            continue
        for field in ("audit_sampling", "adaptive_fn_penalty",
                      "balanced_hard_replay", "full_feedback"):
            if bool(spec.get(field, False)) != bool(variant.get(field, False)):
                raise ValueError(f"Artifact sai cờ {field}.")
        if (spec.get("feedback_selector_mode") != variant["selector_mode"]
                or int(spec.get("feedback_budget", -1)) != int(variant["feedback_budget"])
                or not np.isclose(
                    float(spec.get("reject_cost")),
                    float(variant["reject_cost"]))):
            raise ValueError("Artifact sai điểm vận hành V13.")
        validate_sidecar(artifact_path, signature_for_spec(spec))
        result = pickle.load(open(artifact_path, "rb"))
        validation = validate_result_object(
            result, EXPECTED_PERIODS, policy=variant["policy"], variant=variant)
        key = (variant["name"], int(spec["seed"]))
        if key in artifacts:
            duplicates.append(key); continue
        artifacts[key] = {
            "result": result, "metadata": metadata, "validation": validation}
    except Exception as exc:
        errors.append({"artifact": str(artifact_path),
                       "error": f"{type(exc).__name__}: {exc}"})

missing = sorted(EXPECTED_KEYS - set(artifacts))
STRATEGY_REPORT_READY = (
    set(artifacts) == EXPECTED_KEYS and not errors and not duplicates)
STATIC_EXPECTED_KEYS = {("Static-MLP", int(seed)) for seed in SEEDS}
static_errors = []
static_artifacts = {}
for key in sorted(STATIC_EXPECTED_KEYS):
    method, seed = key
    artifact_path = _static_path(seed)
    if not artifact_path.exists():
        continue
    try:
        static_artifact = _validate_static_artifact(
            artifact_path, seed, require_sidecar=True)
        static_artifacts[key] = static_artifact["period_rows"]
    except Exception as exc:
        static_errors.append({"key": key, "error": f"{type(exc).__name__}: {exc}"})
static_missing = sorted(STATIC_EXPECTED_KEYS - set(static_artifacts))
STATIC_REPORT_READY = (
    set(static_artifacts) == STATIC_EXPECTED_KEYS and not static_errors)
COMBINED_REPORT_READY = STRATEGY_REPORT_READY and STATIC_REPORT_READY
strategy_sample_counts = sorted({
    int(payload["validation"]["test_samples"])
    for payload in artifacts.values()})
static_sample_counts = sorted({
    int(sum(int(row["tot"]) for row in rows))
    for rows in static_artifacts.values()})
integrity = {
    "protocol_version": PROTOCOL_VERSION,
    "signature_schema_version": SIGNATURE_SCHEMA_VERSION,
    "expected_runs": len(EXPECTED_KEYS) + len(STATIC_EXPECTED_KEYS),
    "valid_runs": len(artifacts) + len(static_artifacts),
    "expected_strategy_runs": len(EXPECTED_KEYS),
    "valid_strategy_runs": len(artifacts),
    "expected_static_runs": len(STATIC_EXPECTED_KEYS),
    "valid_static_runs": len(static_artifacts),
    "missing_runs": missing, "missing_static_runs": static_missing,
    "errors": errors, "static_errors": static_errors,
    "duplicates": duplicates,
    "expected_periods_per_run": EXPECTED_TEST_PERIODS,
    "expected_test_samples_per_run": EXPECTED_TEST_SAMPLES,
    "strategy_test_sample_counts": strategy_sample_counts,
    "static_test_sample_counts": static_sample_counts,
    "all_runs_match_canonical_split": bool(
        strategy_sample_counts in ([], [EXPECTED_TEST_SAMPLES])
        and static_sample_counts in ([], [EXPECTED_TEST_SAMPLES])),
    "causal_lag_violations": int(sum(
        int(payload["validation"]["causal_lag_violations"])
        for payload in artifacts.values())),
    "memory_cap_violations": int(sum(
        int(payload["validation"]["memory_cap_violations"])
        for payload in artifacts.values())),
    "report_ready": COMBINED_REPORT_READY,
}
(TABLE_OUT / "v13_integrity_summary.json").write_text(
    json.dumps(integrity, indent=2, ensure_ascii=False), encoding="utf-8")
print(integrity)
if STRICT_INVARIANTS and not COMBINED_REPORT_READY:
    raise RuntimeError("V13 chưa đủ 210/210 hiện vật hợp lệ; không tổng hợp kết luận.")

monthly, telemetry_rows = [], []
for (method, seed), payload in sorted(artifacts.items()):
    result = payload["result"]
    for index, month in enumerate(period_months):
        y_true = np.asarray(result["binary_y_true_all"][index], dtype=int)
        y_pred = np.asarray(result["binary_y_preds_all"][index], dtype=int)
        rejected = np.unique(np.asarray(result["rejected_indexes_all"][index], dtype=int))
        selected = np.unique(np.asarray(result["selected_indexes_all"][index], dtype=int))
        counts = _counts(y_true, y_pred)
        # Chỉ các mẫu bị từ chối được xem là đã được chuyên gia xử lý đúng ở
        # đầu ra hiện tại. Nhãn active-learning ngoài tập từ chối chỉ phục vụ
        # cập nhật từ chu kỳ kế tiếp, không được sửa hồi tố dự đoán tháng này.
        system_prediction = y_pred.copy()
        system_prediction[rejected] = y_true[rejected]
        system_counts = _counts(y_true, system_prediction)
        unique_workload = int(np.union1d(selected, rejected).size)
        monthly.append({
            "method": method, "display_name": DISPLAY_NAMES[method],
            "seed": int(seed), "month": month, "year": int(month[:4]),
            "n": int(y_true.size), **counts,
            **{f"system_{key}": value for key, value in system_counts.items()},
            "label_count": int(selected.size), "reject_count": int(rejected.size),
            "unique_expert_workload": unique_workload,
            "automatic_coverage": _safe_rate(
                int(y_true.size) - int(rejected.size), int(y_true.size)),
            "labels_per_1000": 1000 * _safe_rate(selected.size, y_true.size),
            "rejects_per_1000": 1000 * _safe_rate(rejected.size, y_true.size),
            "expert_workload_per_1000": 1000 * _safe_rate(
                unique_workload, y_true.size),
            "selected_malware": int(np.sum(y_true[selected] == 1)),
            "selected_benign": int(np.sum(y_true[selected] == 0)),
        })
        telemetry = dict(result["bhr_telemetry"][index])
        telemetry_rows.append({
            "method": method, "seed": int(seed), "month": month,
            **telemetry,
        })
strategy_monthly = list(monthly)
static_monthly = []
for (method, seed), rows in sorted(static_artifacts.items()):
    for index, row in enumerate(rows):
        month = period_months[index]
        tp, tn = int(row["tp"]), int(row["tn"])
        fp, fn = int(row["fp"]), int(row["fn"])
        positive_support, negative_support = tp + fn, tn + fp
        static_row = {
            "method": method, "display_name": DISPLAY_NAMES[method],
            "seed": int(seed), "month": month, "year": int(month[:4]),
            "n": int(row["tot"]),
            "tp": int(row["tp"]), "tn": int(row["tn"]),
            "fp": int(row["fp"]), "fn": int(row["fn"]),
            "positive_support": positive_support,
            "negative_support": negative_support,
            "precision": _safe_rate(tp, tp + fp),
            "recall": _safe_rate(tp, positive_support),
            "f1": float(row["f1"]), "fnr": float(row["fnr"]),
            "fpr": float(row["fpr"]),
            "fnr_defined": bool(positive_support > 0),
            "fpr_defined": bool(negative_support > 0),
            "low_positive_support": bool(
                positive_support < RATE_SUPPORT_DIAGNOSTIC_THRESHOLD),
            "low_negative_support": bool(
                negative_support < RATE_SUPPORT_DIAGNOSTIC_THRESHOLD),
            "single_class_period": bool(
                positive_support == 0 or negative_support == 0),
            "label_count": 0, "reject_count": 0,
            "unique_expert_workload": 0,
            "automatic_coverage": 1.0,
            "labels_per_1000": 0.0, "rejects_per_1000": 0.0,
            "expert_workload_per_1000": 0.0,
            "selected_malware": 0, "selected_benign": 0,
        }
        # Mô hình tĩnh không có từ chối, nên đầu ra hệ thống trùng hoàn toàn
        # với đầu ra tự động. Giữ cả hai phạm vi để bảng có schema đồng nhất.
        for key in (
                "tp", "tn", "fp", "fn", "positive_support",
                "negative_support", "precision", "recall", "f1", "fnr",
                "fpr", "fnr_defined", "fpr_defined",
                "low_positive_support", "low_negative_support",
                "single_class_period"):
            static_row[f"system_{key}"] = static_row[key]
        static_monthly.append(static_row)
monthly.extend(static_monthly)
_write_csv(TABLE_OUT / "v13_strategy_monthly.csv", strategy_monthly)
_write_csv(TABLE_OUT / "v13_static_mlp_monthly.csv", static_monthly)
_write_csv(TABLE_OUT / "v13_all_methods_monthly.csv", monthly)
_write_csv(TABLE_OUT / "v13_bhr_telemetry_monthly.csv", telemetry_rows)
_write_csv(
    TABLE_OUT / "v13_rate_support_audit.csv",
    [{key: row[key] for key in (
        "method", "seed", "month", "n", "positive_support",
        "negative_support", "fnr_defined", "fpr_defined",
        "low_positive_support", "low_negative_support",
        "single_class_period")}
     for row in monthly])

def _summarize(method, seed, rows, expected_n_test_samples=None):
    rows = sorted(rows, key=lambda row: row["month"])
    record = {"method": method, "display_name": DISPLAY_NAMES[method],
              "seed": int(seed), "n_test_months": len(rows),
              "n_test_samples": int(sum(row["n"] for row in rows)),
              "total_labels": int(sum(row["label_count"] for row in rows)),
              "total_rejects": int(sum(row["reject_count"] for row in rows)),
              "total_unique_expert_workload": int(sum(row["unique_expert_workload"] for row in rows)),
              "peak_monthly_labels": int(max(row["label_count"] for row in rows)),
              "peak_monthly_rejects": int(max(row["reject_count"] for row in rows)),
              "peak_monthly_unique_expert_workload": int(max(
                  row["unique_expert_workload"] for row in rows)),
              "low_positive_support_periods": int(sum(
                  bool(row["low_positive_support"]) for row in rows)),
              "low_negative_support_periods": int(sum(
                  bool(row["low_negative_support"]) for row in rows)),
              "undefined_fnr_periods": int(sum(
                  not bool(row["fnr_defined"]) for row in rows)),
              "undefined_fpr_periods": int(sum(
                  not bool(row["fpr_defined"]) for row in rows))}
    if (expected_n_test_samples is not None
            and record["n_test_samples"] != int(expected_n_test_samples)):
        raise ValueError(
            f"{method}/seed={seed}: có {record['n_test_samples']} mẫu, "
            f"kỳ vọng {int(expected_n_test_samples)}.")
    for metric in ("f1", "fnr", "fpr",
                   "system_f1", "system_fnr", "system_fpr"):
        aut = _aut([row[metric] for row in rows], [row["month"] for row in rows])
        record[f"aut_{metric}"] = aut["value"]
        record[f"aut_{metric}_valid_edges"] = aut["valid_edges"]
    for count in ("tp", "tn", "fp", "fn"):
        record[f"total_{count}"] = int(sum(row[count] for row in rows))
        record[f"total_system_{count}"] = int(sum(
            row[f"system_{count}"] for row in rows))
    record["pooled_f1"] = _safe_rate(
        2 * record["total_tp"],
        2 * record["total_tp"] + record["total_fp"] + record["total_fn"])
    record["pooled_fnr"] = _safe_rate(
        record["total_fn"], record["total_fn"] + record["total_tp"])
    record["pooled_fpr"] = _safe_rate(
        record["total_fp"], record["total_fp"] + record["total_tn"])
    record["pooled_precision"] = _safe_rate(
        record["total_tp"], record["total_tp"] + record["total_fp"])
    record["pooled_recall"] = _safe_rate(
        record["total_tp"], record["total_tp"] + record["total_fn"])
    record["pooled_system_f1"] = _safe_rate(
        2 * record["total_system_tp"],
        2 * record["total_system_tp"]
        + record["total_system_fp"] + record["total_system_fn"])
    record["pooled_system_fnr"] = _safe_rate(
        record["total_system_fn"],
        record["total_system_fn"] + record["total_system_tp"])
    record["pooled_system_fpr"] = _safe_rate(
        record["total_system_fp"],
        record["total_system_fp"] + record["total_system_tn"])
    record["automatic_coverage"] = _safe_rate(
        record["n_test_samples"] - record["total_rejects"],
        record["n_test_samples"])
    record["labels_per_1000"] = 1000 * _safe_rate(
        record["total_labels"], record["n_test_samples"])
    record["rejects_per_1000"] = 1000 * _safe_rate(
        record["total_rejects"], record["n_test_samples"])
    record["expert_workload_per_1000"] = 1000 * _safe_rate(
        record["total_unique_expert_workload"], record["n_test_samples"])
    return record

ALL_METHOD_NAMES = ["Static-MLP"] + list(PRIMARY_STRATEGY_NAMES)
EVALUATED_METHOD_NAMES = ["Static-MLP"] + [
    variant["name"] for variant in STRATEGY_VARIANTS]
summary = []
for method in EVALUATED_METHOD_NAMES:
    for seed in SEEDS:
        rows = [row for row in monthly if row["method"] == method and row["seed"] == seed]
        if rows:
            summary.append(_summarize(
                method, seed, rows,
                expected_n_test_samples=EXPECTED_TEST_SAMPLES))
_write_csv(
    TABLE_OUT / "v13_strategy_summary_by_seed.csv",
    [row for row in summary if row["method"] != "Static-MLP"])
_write_csv(
    TABLE_OUT / "v13_static_mlp_summary_by_seed.csv",
    [row for row in summary if row["method"] == "Static-MLP"])
_write_csv(TABLE_OUT / "v13_all_methods_summary_by_seed.csv", summary)
summary_map = {(row["method"], row["seed"]): row for row in summary}

method_overview = []
for method in EVALUATED_METHOD_NAMES:
    rows = [row for row in summary if row["method"] == method]
    record = {"method": method, "display_name": DISPLAY_NAMES[method],
              "n_seeds": len(rows)}
    for metric in ("aut_f1", "aut_fnr", "aut_fpr",
                   "aut_system_f1", "aut_system_fnr", "aut_system_fpr"):
        values = np.asarray([row[metric] for row in rows], dtype=float)
        finite = values[np.isfinite(values)]
        record[f"mean_{metric}_pct"] = (
            100 * float(np.mean(finite)) if finite.size else float("nan"))
        record[f"sd_{metric}_pct"] = (
            100 * float(np.std(finite, ddof=1)) if finite.size > 1 else float("nan"))
    for metric in ("pooled_f1", "pooled_fnr", "pooled_fpr",
                   "pooled_precision", "pooled_recall",
                   "pooled_system_f1", "pooled_system_fnr",
                   "pooled_system_fpr", "automatic_coverage"):
        values = np.asarray([row[metric] for row in rows], dtype=float)
        finite = values[np.isfinite(values)]
        record[f"mean_{metric}_pct"] = (
            100 * float(np.mean(finite)) if finite.size else float("nan"))
        record[f"sd_{metric}_pct"] = (
            100 * float(np.std(finite, ddof=1)) if finite.size > 1 else float("nan"))
    for metric in ("labels_per_1000", "rejects_per_1000",
                   "expert_workload_per_1000",
                   "peak_monthly_unique_expert_workload"):
        values = np.asarray([row[metric] for row in rows], dtype=float)
        record[f"mean_{metric}"] = float(np.mean(values))
        record[f"sd_{metric}"] = (
            float(np.std(values, ddof=1)) if values.size > 1 else float("nan"))
    method_overview.append(record)
_write_csv(TABLE_OUT / "v13_all_methods_overview.csv", method_overview)

def _budget_method(base_method, budget):
    if int(budget) == FEEDBACK_UPDATE_BUDGET:
        return base_method
    return f"{base_method}-B{int(budget)}"

def _reject_cost_method(base_method, reject_cost):
    if abs(float(reject_cost) - float(STRATEGY_REJECT_COST)) < 1e-12:
        return base_method
    return f"{base_method}-{_cost_suffix(reject_cost)}"

def _overview_rows(rows, grouping_keys, metric_specs):
    grouped = {}
    for row in rows:
        key = tuple(row[field] for field in grouping_keys)
        grouped.setdefault(key, []).append(row)
    overview = []
    for key, items in sorted(grouped.items(), key=lambda pair: pair[0]):
        record = {field: value for field, value in zip(grouping_keys, key)}
        record["n_seeds"] = len(items)
        for source, output, scale in metric_specs:
            values = np.asarray([item[source] for item in items], dtype=float)
            finite = values[np.isfinite(values)]
            record[f"mean_{output}"] = (
                scale * float(np.mean(finite)) if finite.size else float("nan"))
            record[f"sd_{output}"] = (
                scale * float(np.std(finite, ddof=1))
                if finite.size > 1 else float("nan"))
        overview.append(record)
    return overview

# Độ nhạy một yếu tố theo ngân sách phản hồi. B=15 là điểm vận hành chính;
# mọi hiệu đều ghép đúng cùng seed và cùng cơ chế, không lấy trung bình qua B.
budget_rows = []
for base_method in ("DRMD-IRAAL", "DRMD-FN-BHR"):
    canonical_method = _budget_method(base_method, FEEDBACK_UPDATE_BUDGET)
    for budget in SENSITIVITY_BUDGETS:
        method = _budget_method(base_method, budget)
        for seed in SEEDS:
            row = summary_map.get((method, seed))
            canonical = summary_map.get((canonical_method, seed))
            if row is None:
                continue
            budget_rows.append({
                "family": base_method, "method": method,
                "display_name": DISPLAY_NAMES[method],
                "budget": int(budget), "seed": int(seed),
                "aut_f1": row["aut_f1"], "aut_fnr": row["aut_fnr"],
                "aut_fpr": row["aut_fpr"],
                "delta_aut_f1_vs_B15_pp": (
                    100 * (row["aut_f1"] - canonical["aut_f1"])
                    if canonical is not None else float("nan")),
                "delta_aut_fnr_vs_B15_pp": (
                    100 * (row["aut_fnr"] - canonical["aut_fnr"])
                    if canonical is not None else float("nan")),
                "delta_aut_fpr_vs_B15_pp": (
                    100 * (row["aut_fpr"] - canonical["aut_fpr"])
                    if canonical is not None else float("nan")),
                "labels_per_1000": row["labels_per_1000"],
                "rejects_per_1000": row["rejects_per_1000"],
                "expert_workload_per_1000": row["expert_workload_per_1000"],
            })
_write_csv(TABLE_OUT / "v13_budget_sensitivity_by_seed.csv", budget_rows)
budget_overview = _overview_rows(
    budget_rows, ("family", "budget"),
    (("aut_f1", "aut_f1_pct", 100.0),
     ("aut_fnr", "aut_fnr_pct", 100.0),
     ("aut_fpr", "aut_fpr_pct", 100.0),
     ("delta_aut_f1_vs_B15_pp", "delta_aut_f1_vs_B15_pp", 1.0),
     ("delta_aut_fnr_vs_B15_pp", "delta_aut_fnr_vs_B15_pp", 1.0),
     ("delta_aut_fpr_vs_B15_pp", "delta_aut_fpr_vs_B15_pp", 1.0),
     ("labels_per_1000", "labels_per_1000", 1.0),
     ("rejects_per_1000", "rejects_per_1000", 1.0),
     ("expert_workload_per_1000", "expert_workload_per_1000", 1.0)))
_write_csv(TABLE_OUT / "v13_budget_sensitivity_overview.csv", budget_overview)

# Độ nhạy chi phí từ chối được tách riêng cho IRAL và IRAAL để trả lời liệu
# suy giảm của IRAL do điểm R_cost hay do vòng phản hồi nội sinh Q_t=R_t.
reject_cost_rows = []
for base_method in ("DRMD-IRAL", "DRMD-IRAAL"):
    canonical_method = _reject_cost_method(base_method, STRATEGY_REJECT_COST)
    for reject_cost in SENSITIVITY_REJECT_COSTS:
        method = _reject_cost_method(base_method, reject_cost)
        for seed in SEEDS:
            row = summary_map.get((method, seed))
            canonical = summary_map.get((canonical_method, seed))
            if row is None:
                continue
            reject_cost_rows.append({
                "family": base_method, "method": method,
                "display_name": DISPLAY_NAMES[method],
                "reject_cost": float(reject_cost), "seed": int(seed),
                "aut_f1": row["aut_f1"], "aut_fnr": row["aut_fnr"],
                "aut_fpr": row["aut_fpr"],
                "delta_aut_f1_vs_Cm0_1_pp": (
                    100 * (row["aut_f1"] - canonical["aut_f1"])
                    if canonical is not None else float("nan")),
                "delta_aut_fnr_vs_Cm0_1_pp": (
                    100 * (row["aut_fnr"] - canonical["aut_fnr"])
                    if canonical is not None else float("nan")),
                "delta_aut_fpr_vs_Cm0_1_pp": (
                    100 * (row["aut_fpr"] - canonical["aut_fpr"])
                    if canonical is not None else float("nan")),
                "labels_per_1000": row["labels_per_1000"],
                "rejects_per_1000": row["rejects_per_1000"],
                "expert_workload_per_1000": row["expert_workload_per_1000"],
            })
_write_csv(TABLE_OUT / "v13_reject_cost_sensitivity_by_seed.csv", reject_cost_rows)
reject_cost_overview = _overview_rows(
    reject_cost_rows, ("family", "reject_cost"),
    (("aut_f1", "aut_f1_pct", 100.0),
     ("aut_fnr", "aut_fnr_pct", 100.0),
     ("aut_fpr", "aut_fpr_pct", 100.0),
     ("delta_aut_f1_vs_Cm0_1_pp", "delta_aut_f1_vs_Cm0_1_pp", 1.0),
     ("delta_aut_fnr_vs_Cm0_1_pp", "delta_aut_fnr_vs_Cm0_1_pp", 1.0),
     ("delta_aut_fpr_vs_Cm0_1_pp", "delta_aut_fpr_vs_Cm0_1_pp", 1.0),
     ("labels_per_1000", "labels_per_1000", 1.0),
     ("rejects_per_1000", "rejects_per_1000", 1.0),
     ("expert_workload_per_1000", "expert_workload_per_1000", 1.0)))
_write_csv(
    TABLE_OUT / "v13_reject_cost_sensitivity_overview.csv",
    reject_cost_overview)

# Ba chân trời thời gian dùng để mô tả suy giảm dưới concept drift. Khoảng
# đầu chỉ là IID-proxy theo thời gian (cùng giai đoạn ngay sau huấn luyện),
# không được diễn giải thành kiểm định cùng phân phối nghiêm ngặt.
TEMPORAL_HORIZONS = (
    ("IID-proxy", lambda year: year == 2014),
    ("NEAR", lambda year: year in (2016, 2017)),
    ("FAR", lambda year: year >= 2018),
)
horizon_expected_samples = {
    name: int(sum(
        int(y_tests[index].size)
        for index, month in enumerate(period_months)
        if predicate(int(month[:4]))))
    for name, predicate in TEMPORAL_HORIZONS
}
horizon_by_seed = []
for method in ALL_METHOD_NAMES:
    for seed in SEEDS:
        for horizon, predicate in TEMPORAL_HORIZONS:
            rows = [
                row for row in monthly
                if row["method"] == method and row["seed"] == seed
                and predicate(int(row["year"]))]
            if not rows:
                continue
            summarized = _summarize(
                method, seed, rows,
                expected_n_test_samples=horizon_expected_samples[horizon])
            summarized["horizon"] = horizon
            summarized["horizon_definition"] = (
                "2014-06_to_2014-08_same_era_proxy"
                if horizon == "IID-proxy" else
                "2016_to_2017_near_temporal"
                if horizon == "NEAR" else "2018_to_2025_far_temporal")
            horizon_by_seed.append(summarized)
_write_csv(TABLE_OUT / "v13_temporal_horizons_by_seed.csv", horizon_by_seed)
horizon_overview = _overview_rows(
    horizon_by_seed, ("method", "display_name", "horizon"),
    (("aut_f1", "aut_f1_pct", 100.0),
     ("aut_fnr", "aut_fnr_pct", 100.0),
     ("aut_fpr", "aut_fpr_pct", 100.0),
     ("labels_per_1000", "labels_per_1000", 1.0),
     ("rejects_per_1000", "rejects_per_1000", 1.0),
     ("expert_workload_per_1000", "expert_workload_per_1000", 1.0)))
_write_csv(TABLE_OUT / "v13_temporal_horizons_overview.csv", horizon_overview)
if COMBINED_REPORT_READY:
    expected_horizon_rows = (
        len(ALL_METHOD_NAMES) * len(SEEDS) * len(TEMPORAL_HORIZONS))
    if len(horizon_by_seed) != expected_horizon_rows:
        raise RuntimeError(
            f"Bảng IID-proxy/NEAR/FAR có {len(horizon_by_seed)} hàng, "
            f"kỳ vọng {expected_horizon_rows} hàng.")

COMPARISONS = [
    ("IRAAL_minus_IRAL", "DRMD-IRAAL", "DRMD-IRAL"),
    ("FN_minus_IRAAL", "DRMD-FN", "DRMD-IRAAL"),
    ("BHR_minus_IRAAL", "DRMD-BHR", "DRMD-IRAAL"),
    ("FN_BHR_minus_FN", "DRMD-FN-BHR", "DRMD-FN"),
    ("FN_BHR_minus_BHR_only", "DRMD-FN-BHR", "DRMD-BHR"),
    ("FN_BHR_minus_IRAAL", "DRMD-FN-BHR", "DRMD-IRAAL"),
]
paired = []
for comparison, treatment, control in COMPARISONS:
    for seed in SEEDS:
        left = summary_map.get((treatment, seed)); right = summary_map.get((control, seed))
        if left is None or right is None:
            continue
        paired.append({
            "comparison": comparison, "treatment": treatment, "control": control,
            "seed": int(seed),
            "delta_aut_f1_pp": 100 * (left["aut_f1"] - right["aut_f1"]),
            "delta_aut_fnr_pp": 100 * (left["aut_fnr"] - right["aut_fnr"]),
            "delta_aut_fpr_pp": 100 * (left["aut_fpr"] - right["aut_fpr"]),
            "delta_aut_system_f1_pp": 100 * (
                left["aut_system_f1"] - right["aut_system_f1"]),
            "delta_aut_system_fnr_pp": 100 * (
                left["aut_system_fnr"] - right["aut_system_fnr"]),
            "delta_aut_system_fpr_pp": 100 * (
                left["aut_system_fpr"] - right["aut_system_fpr"]),
            "delta_pooled_f1_pp": 100 * (left["pooled_f1"] - right["pooled_f1"]),
            "delta_pooled_system_f1_pp": 100 * (
                left["pooled_system_f1"] - right["pooled_system_f1"]),
            "delta_tp": int(left["total_tp"] - right["total_tp"]),
            "delta_fn": int(left["total_fn"] - right["total_fn"]),
            "delta_fp": int(left["total_fp"] - right["total_fp"]),
            "additional_malware_detected": int(
                right["total_fn"] - left["total_fn"]),
            "additional_malware_missed": int(
                left["total_fn"] - right["total_fn"]),
            "additional_false_alerts": int(
                left["total_fp"] - right["total_fp"]),
            "delta_labels": int(left["total_labels"] - right["total_labels"]),
            "delta_unique_expert_workload": int(
                left["total_unique_expert_workload"] - right["total_unique_expert_workload"]),
            "relative_unique_workload_change_pct": (
                100 * (left["total_unique_expert_workload"]
                       - right["total_unique_expert_workload"])
                / right["total_unique_expert_workload"]
                if right["total_unique_expert_workload"] else float("nan")),
        })
_write_csv(TABLE_OUT / "v13_paired_effects_by_seed.csv", paired)

def _exact_sign_flip_p(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not values.size:
        return float("nan")
    observed = abs(float(np.mean(values)))
    permuted = [abs(float(np.mean(values * np.asarray(signs))))
                for signs in itertools.product((-1.0, 1.0), repeat=values.size)]
    return float(np.mean(np.asarray(permuted) >= observed - 1e-12))

effect_summary = []
metrics = ("delta_aut_f1_pp", "delta_aut_fnr_pp", "delta_aut_fpr_pp",
           "delta_aut_system_f1_pp", "delta_aut_system_fnr_pp",
           "delta_aut_system_fpr_pp", "delta_pooled_f1_pp",
           "delta_pooled_system_f1_pp", "delta_tp", "delta_fn", "delta_fp",
           "additional_malware_detected", "additional_malware_missed",
           "additional_false_alerts",
           "delta_labels", "delta_unique_expert_workload",
           "relative_unique_workload_change_pct")
for comparison, treatment, control in COMPARISONS:
    rows = [row for row in paired if row["comparison"] == comparison]
    record = {"comparison": comparison, "treatment": treatment, "control": control,
              "n_paired_seeds": len(rows)}
    for metric in metrics:
        values = np.asarray([row[metric] for row in rows], dtype=float)
        finite = values[np.isfinite(values)]
        mean = float(np.mean(finite)) if finite.size else float("nan")
        sd = float(np.std(finite, ddof=1)) if finite.size > 1 else float("nan")
        half = (float(stats.t.ppf(0.975, finite.size - 1)) * sd / np.sqrt(finite.size)
                if finite.size > 1 else float("nan"))
        record[f"mean_{metric}"] = mean
        record[f"sd_{metric}"] = sd
        record[f"ci95_low_{metric}"] = mean - half
        record[f"ci95_high_{metric}"] = mean + half
        record[f"paired_t_p_{metric}"] = (
            float(stats.ttest_1samp(finite, 0.0).pvalue) if finite.size > 1 else float("nan"))
        record[f"exact_sign_flip_p_{metric}"] = _exact_sign_flip_p(finite)
    effect_summary.append(record)

# Giữ nguyên họ 15 kiểm định BHR đã khóa trước: 5 tương phản × 3 AUT metrics.
# Tương phản giải thích IRAAL−IRAL được hiệu chỉnh riêng trên ba metric để việc
# bổ sung đối chứng không làm thay đổi hậu nghiệm ngưỡng của giả thuyết BHR.
holm_items = []
for record in effect_summary:
    if record["comparison"] == "IRAAL_minus_IRAL":
        continue
    for metric in ("delta_aut_f1_pp", "delta_aut_fnr_pp", "delta_aut_fpr_pp"):
        holm_items.append((record, metric, record[f"paired_t_p_{metric}"]))
finite_items = sorted(
    [item for item in holm_items if np.isfinite(item[2])], key=lambda item: item[2])
running = 0.0
for rank, (record, metric, p_value) in enumerate(finite_items):
    adjusted = min(1.0, (len(finite_items) - rank) * p_value)
    running = max(running, adjusted)
    record[f"holm_p_{metric}"] = running

mechanism_record = next(
    (row for row in effect_summary
     if row["comparison"] == "IRAAL_minus_IRAL"), None)
if mechanism_record is not None:
    mechanism_items = sorted(
        [(metric, mechanism_record[f"paired_t_p_{metric}"])
         for metric in ("delta_aut_f1_pp", "delta_aut_fnr_pp",
                        "delta_aut_fpr_pp")
         if np.isfinite(mechanism_record[f"paired_t_p_{metric}"])],
        key=lambda item: item[1])
    running = 0.0
    for rank, (metric, p_value) in enumerate(mechanism_items):
        adjusted = min(1.0, (len(mechanism_items) - rank) * p_value)
        running = max(running, adjusted)
        mechanism_record[f"mechanism_holm_p_{metric}"] = running
_write_csv(TABLE_OUT / "v13_paired_effects_summary.csv", effect_summary)

# Quy chiếu đồng bộ 10 seed: lợi ích so với MLP tĩnh và khoảng cách tới FFCR.
# IRAL được đưa vào đây như cơ chế thích nghi có phản hồi nội sinh, không được
# diễn giải là một nhánh cùng ngân sách B=15.
ADAPTIVE_METHODS = [
    "DRMD-IRAL", "DRMD-IRAAL", "DRMD-FN", "DRMD-BHR", "DRMD-FN-BHR"]
REFERENCE_COMPARISONS = [
    (f"{method}_minus_Static", method, "Static-MLP")
    for method in ADAPTIVE_METHODS
] + [
    (f"FFCR_minus_{method}", "DRMD-FFCR", method)
    for method in ADAPTIVE_METHODS
] + [("FFCR_minus_Static", "DRMD-FFCR", "Static-MLP")]
reference_paired = []
for comparison, treatment, control in REFERENCE_COMPARISONS:
    for seed in SEEDS:
        left = summary_map.get((treatment, seed)); right = summary_map.get((control, seed))
        if left is None or right is None:
            continue
        reference_paired.append({
            "comparison": comparison, "treatment": treatment, "control": control,
            "seed": int(seed),
            "delta_aut_f1_pp": 100 * (left["aut_f1"] - right["aut_f1"]),
            "delta_aut_fnr_pp": 100 * (left["aut_fnr"] - right["aut_fnr"]),
            "delta_aut_fpr_pp": 100 * (left["aut_fpr"] - right["aut_fpr"]),
            "delta_aut_system_f1_pp": 100 * (
                left["aut_system_f1"] - right["aut_system_f1"]),
            "delta_aut_system_fnr_pp": 100 * (
                left["aut_system_fnr"] - right["aut_system_fnr"]),
            "delta_aut_system_fpr_pp": 100 * (
                left["aut_system_fpr"] - right["aut_system_fpr"]),
        })
_write_csv(TABLE_OUT / "v13_reference_gaps_by_seed.csv", reference_paired)

reference_summary = []
for comparison, treatment, control in REFERENCE_COMPARISONS:
    rows = [row for row in reference_paired if row["comparison"] == comparison]
    record = {"comparison": comparison, "treatment": treatment, "control": control,
              "n_paired_seeds": len(rows)}
    for metric in ("delta_aut_f1_pp", "delta_aut_fnr_pp", "delta_aut_fpr_pp",
                   "delta_aut_system_f1_pp", "delta_aut_system_fnr_pp",
                   "delta_aut_system_fpr_pp"):
        values = np.asarray([row[metric] for row in rows], dtype=float)
        finite = values[np.isfinite(values)]
        mean = float(np.mean(finite)) if finite.size else float("nan")
        sd = float(np.std(finite, ddof=1)) if finite.size > 1 else float("nan")
        half = (float(stats.t.ppf(0.975, finite.size - 1)) * sd / np.sqrt(finite.size)
                if finite.size > 1 else float("nan"))
        record[f"mean_{metric}"] = mean
        record[f"sd_{metric}"] = sd
        record[f"ci95_low_{metric}"] = mean - half
        record[f"ci95_high_{metric}"] = mean + half
        record[f"paired_t_p_{metric}"] = (
            float(stats.ttest_1samp(finite, 0.0).pvalue)
            if finite.size > 1 else float("nan"))
        record[f"exact_sign_flip_p_{metric}"] = _exact_sign_flip_p(finite)
    reference_summary.append(record)
reference_holm = []
for record in reference_summary:
    for metric in ("delta_aut_f1_pp", "delta_aut_fnr_pp", "delta_aut_fpr_pp"):
        reference_holm.append((record, metric, record[f"paired_t_p_{metric}"]))
finite_reference_holm = sorted(
    [item for item in reference_holm if np.isfinite(item[2])], key=lambda item: item[2])
running = 0.0
for rank, (record, metric, p_value) in enumerate(finite_reference_holm):
    adjusted = min(1.0, (len(finite_reference_holm) - rank) * p_value)
    running = max(running, adjusted)
    record[f"holm_p_{metric}"] = running
_write_csv(TABLE_OUT / "v13_reference_gaps_summary.csv", reference_summary)

primary = next((row for row in effect_summary
                if row["comparison"] == "FN_BHR_minus_FN"), None)
success_rows = []
if primary is not None and primary["n_paired_seeds"] == len(SEEDS):
    criteria = {
        "fnr_improvement": primary["mean_delta_aut_fnr_pp"] <= -1.0,
        "fpr_control": primary["mean_delta_aut_fpr_pp"] <= 0.5,
        "f1_noninferiority": primary["mean_delta_aut_f1_pp"] >= -0.5,
        "workload_control": primary["mean_relative_unique_workload_change_pct"] <= 5.0,
        "same_label_budget": abs(primary["mean_delta_labels"]) < 1e-12,
    }
    for criterion, passed in criteria.items():
        success_rows.append({"primary_comparison": "FN_BHR_minus_FN",
                             "criterion": criterion, "passed": bool(passed)})
    success_rows.append({"primary_comparison": "FN_BHR_minus_FN",
                         "criterion": "overall_controlled_sensitivity_improvement",
                         "passed": bool(all(criteria.values()))})
_write_csv(TABLE_OUT / "v13_success_gate.csv", success_rows)

# Bảng theo năm 2016–2019 và lát cắt 2016 cho đủ bảy phương pháp, cùng seed.
expected_samples_by_year = {
    year: int(sum(
        int(X_tests[index].shape[0])
        for index, month in enumerate(period_months)
        if int(month[:4]) == year))
    for year in (2016, 2017, 2018, 2019)
}
yearly = []
for method in ALL_METHOD_NAMES:
    for seed in SEEDS:
        for year in (2016, 2017, 2018, 2019):
            rows = [row for row in monthly if row["method"] == method
                    and row["seed"] == seed and row["year"] == year]
            if rows:
                yearly.append(_summarize(
                    method, seed, rows,
                    expected_n_test_samples=expected_samples_by_year[year]
                ) | {"year": year})
_write_csv(TABLE_OUT / "v13_yearly_2016_2019_by_seed.csv", yearly)
slice_2016 = []
for method in ALL_METHOD_NAMES:
    rows = [row for row in yearly if row["method"] == method and row["year"] == 2016]
    if rows:
        slice_2016.append({
            "method": method, "display_name": DISPLAY_NAMES[method],
            "n_seeds": len(rows),
            "mean_f1_pct": 100 * float(np.mean([row["aut_f1"] for row in rows])),
            "sd_f1_pct": 100 * float(np.std([row["aut_f1"] for row in rows], ddof=1)),
            "mean_fnr_pct": 100 * float(np.mean([row["aut_fnr"] for row in rows])),
            "sd_fnr_pct": 100 * float(np.std([row["aut_fnr"] for row in rows], ddof=1)),
            "mean_fpr_pct": 100 * float(np.mean([row["aut_fpr"] for row in rows])),
            "sd_fpr_pct": 100 * float(np.std([row["aut_fpr"] for row in rows], ddof=1)),
            "mean_system_f1_pct": 100 * float(np.mean(
                [row["aut_system_f1"] for row in rows])),
            "sd_system_f1_pct": 100 * float(np.std(
                [row["aut_system_f1"] for row in rows], ddof=1)),
            "mean_system_fnr_pct": 100 * float(np.mean(
                [row["aut_system_fnr"] for row in rows])),
            "sd_system_fnr_pct": 100 * float(np.std(
                [row["aut_system_fnr"] for row in rows], ddof=1)),
            "mean_system_fpr_pct": 100 * float(np.mean(
                [row["aut_system_fpr"] for row in rows])),
            "sd_system_fpr_pct": 100 * float(np.std(
                [row["aut_system_fpr"] for row in rows], ddof=1)),
        })
if COMBINED_REPORT_READY:
    expected_yearly_rows = len(ALL_METHOD_NAMES) * len(SEEDS) * 4
    if len(yearly) != expected_yearly_rows:
        raise RuntimeError(
            f"Bảng theo năm có {len(yearly)} hàng, kỳ vọng "
            f"{expected_yearly_rows} hàng.")
    if (len(slice_2016) != len(ALL_METHOD_NAMES)
            or any(row["n_seeds"] != len(SEEDS) for row in slice_2016)):
        raise RuntimeError("Lát cắt 2016 chưa đủ bảy phương pháp × mười seed.")
_write_csv(TABLE_OUT / "v13_slice_2016_summary.csv", slice_2016)

if COMBINED_REPORT_READY:
    # Hình 1: hiệu ghép cặp chính theo ba AUT metrics.
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.7))
    for ax, metric, label, good in zip(
            axes,
            ("delta_aut_f1_pp", "delta_aut_fnr_pp", "delta_aut_fpr_pp"),
            ("ΔF1 (điểm %)", "ΔFNR (điểm %)", "ΔFPR (điểm %)"),
            ("cao hơn tốt", "thấp hơn tốt", "thấp hơn tốt")):
        rows = [row for row in paired if row["comparison"] == "FN_BHR_minus_FN"]
        values = [row[metric] for row in rows]
        ax.scatter(range(len(values)), values, facecolors="white",
                   edgecolors="black", linewidths=0.8, zorder=3)
        ax.axhline(0, color="black", linewidth=0.9)
        ax.axhline(np.mean(values), color="black", linestyle="--", linewidth=1.3)
        positions = np.arange(len(values))
        ax.set_xticks(positions, [f"S{{index + 1}}" for index in positions])
        ax.set_xlabel("Cặp hạt giống (S1–S10)")
        ax.set_ylabel(label); ax.set_title(good)
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Hiệu ghép cặp theo AUT: DRMD-FN–BHR trừ DRMD-FN")
    fig.tight_layout(); fig.savefig(
        FIG_OUT / "v13_primary_paired_effects.png", dpi=220,
        bbox_inches="tight")
    plt.close(fig)

    # Hình 2: lý do chọn IRAAL làm nền, ghép đúng cùng hạt giống với IRAL.
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.7))
    iral_rows = [
        row for row in paired if row["comparison"] == "IRAAL_minus_IRAL"]
    for ax, metric, label, good in zip(
            axes,
            ("delta_aut_f1_pp", "delta_aut_fnr_pp", "delta_aut_fpr_pp"),
            ("ΔF1 (điểm %)", "ΔFNR (điểm %)", "ΔFPR (điểm %)"),
            ("cao hơn tốt", "thấp hơn tốt", "thấp hơn tốt")):
        values = [row[metric] for row in iral_rows]
        positions = np.arange(len(values))
        ax.scatter(positions, values, facecolors="white", edgecolors="black",
                   linewidths=0.8, zorder=3)
        ax.axhline(0, color="black", linewidth=0.9)
        ax.axhline(np.mean(values), color="black", linestyle="--", linewidth=1.3)
        ax.set_xticks(positions, [f"S{index + 1}" for index in positions])
        ax.set_xlabel("Cặp hạt giống (S1–S10)")
        ax.set_ylabel(label); ax.set_title(good); ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Hiệu ghép cặp theo AUT: DRMD (IRAAL) trừ DRMD (IRAL)")
    fig.tight_layout(); fig.savefig(
        FIG_OUT / "v13_iraal_minus_iral_paired.png", dpi=220,
        bbox_inches="tight")
    plt.close(fig)

    # Hình 3: lát cắt 2016 đồng bộ Static/IRAL/B=15/FFCR trên ba chỉ số.
    colors = [
        "#f7f7f7", "#e5e5e5", "#cccccc", "#b2b2b2",
        "#969696", "#636363", "#252525"]
    hatches = ["", "///", "\\\\", "...", "xx", "++", "||"]
    positions = np.arange(len(slice_2016))
    fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.8))
    for ax, mean_key, sd_key, label in zip(
            axes,
            ("mean_f1_pct", "mean_fnr_pct", "mean_fpr_pct"),
            ("sd_f1_pct", "sd_fnr_pct", "sd_fpr_pct"),
            ("F1 (%)", "FNR (%)", "FPR (%)")):
        bars = ax.bar(
            positions, [row[mean_key] for row in slice_2016],
            yerr=[row[sd_key] for row in slice_2016], capsize=3,
            color=colors, edgecolor="black", linewidth=0.6)
        for bar, hatch in zip(bars, hatches):
            bar.set_hatch(hatch)
        ax.set_xticks(positions, [row["display_name"] for row in slice_2016],
                      rotation=22, ha="right")
        ax.set_ylabel(label); ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Lát cắt 2016 (trung bình theo thời gian): bảy phương pháp")
    fig.tight_layout(); fig.savefig(
        FIG_OUT / "v13_2016_slice.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    # Hình 4: đối chiếu toàn chuỗi của đủ bảy phương pháp.
    positions = np.arange(len(method_overview))
    fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.8))
    for ax, mean_key, sd_key, label in zip(
            axes,
            ("mean_aut_f1_pct", "mean_aut_fnr_pct", "mean_aut_fpr_pct"),
            ("sd_aut_f1_pct", "sd_aut_fnr_pct", "sd_aut_fpr_pct"),
            ("F1 (%)", "FNR (%)", "FPR (%)")):
        bars = ax.bar(
            positions, [row[mean_key] for row in method_overview],
            yerr=[row[sd_key] for row in method_overview], capsize=3,
            color=colors, edgecolor="black", linewidth=0.6)
        for bar, hatch in zip(bars, hatches):
            bar.set_hatch(hatch)
        ax.set_xticks(positions, [row["display_name"] for row in method_overview],
                      rotation=22, ha="right")
        ax.set_ylabel(label); ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Toàn chuỗi theo AUT: bảy phương pháp, 10 seed")
    fig.tight_layout(); fig.savefig(
        FIG_OUT / "v13_all_methods_aut.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    # Hình 5: lợi ích sau thẩm định phải luôn đi cùng tải chuyên gia.
    fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.8))
    for ax, mean_key, sd_key, label in (
            (axes[0], "mean_pooled_system_f1_pct",
             "sd_pooled_system_f1_pct", "F1 hệ thống sau thẩm định (%)"),
            (axes[1], "mean_expert_workload_per_1000",
             "sd_expert_workload_per_1000", "APK cần thẩm định / 1.000 APK")):
        bars = ax.bar(
            positions, [row[mean_key] for row in method_overview],
            yerr=[row[sd_key] for row in method_overview], capsize=3,
            color=colors, edgecolor="black", linewidth=0.6)
        for bar, hatch in zip(bars, hatches):
            bar.set_hatch(hatch)
        ax.set_xticks(
            positions, [row["display_name"] for row in method_overview],
            rotation=22, ha="right")
        ax.set_ylabel(label); ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Hiệu năng sau thẩm định và khối lượng chuyên gia")
    fig.tight_layout(); fig.savefig(
        FIG_OUT / "v13_system_f1_and_expert_workload.png", dpi=220,
        bbox_inches="tight")
    plt.close(fig)

    # Hình 6: thành phần bộ nhớ của hai nhánh BHR theo thời gian.
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 6.6), sharex=True)
    for ax, method in zip(axes, ("DRMD-BHR", "DRMD-FN-BHR")):
        for key, label, style, marker in (
                ("fn_rows", "FN replay", "-", "o"),
                ("fp_rows", "FP khó", "--", "s"),
                ("background_rows", "Nền gần đây", ":", "^")):
            means = []
            for month in period_months:
                values = [float(row[key]) for row in telemetry_rows
                          if row["method"] == method and row["month"] == month]
                means.append(np.mean(values))
            ax.plot(period_months, means, label=label, color="black",
                    linestyle=style, marker=marker, markersize=2.2,
                    markerfacecolor="white", markeredgewidth=0.6,
                    linewidth=1.0)
        ax.set_title(DISPLAY_NAMES[method]); ax.set_ylabel("Số hàng bộ nhớ")
        ax.grid(alpha=0.2); ax.legend(ncol=3, fontsize=8)
    axes[-1].set_xticks(range(0, len(period_months), 12), period_months[::12], rotation=45)
    fig.tight_layout(); fig.savefig(
        FIG_OUT / "v13_bhr_memory_telemetry.png", dpi=220,
        bbox_inches="tight")
    plt.close(fig)

xuat_nhat_ky_suy_bien()
print({"v13_report_ready": COMBINED_REPORT_READY,
       "valid_runs": len(artifacts) + len(static_artifacts),
       "expected_runs": len(EXPECTED_KEYS) + len(STATIC_EXPECTED_KEYS),
       "tables": str(TABLE_OUT), "figures": str(FIG_OUT)})


## Ô 12 — Đánh giá CFJR và khoảng cách đến các mốc hiện có

CFJR có bảng toàn vẹn riêng và chỉ được diễn giải khi đủ mười hạt giống. Đây là cận tham chiếu thực nghiệm nhân quả, không phải cận trên lý thuyết.


In [ ]:
# Tổng hợp riêng CFJR; không thay đổi cổng 210 lượt chính.
cfjr_monthly = []
for (method, seed), rows in sorted(CFJR_RESULTS.items()):
    for row in rows:
        period = int(row["period"])
        month = period_months[period - 1]
        tp, tn = int(row["tp"]), int(row["tn"])
        fp, fn = int(row["fp"]), int(row["fn"])
        positive_support, negative_support = tp + fn, tn + fp
        item = {
            "method": CFJR_METHOD, "display_name": DISPLAY_NAMES[CFJR_METHOD],
            "seed": int(seed), "month": month, "year": int(month[:4]),
            "period": period, "n": int(row["tot"]),
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "positive_support": positive_support,
            "negative_support": negative_support,
            "precision": _safe_rate(tp, tp + fp),
            "recall": _safe_rate(tp, positive_support),
            "f1": float(row["f1"]), "fnr": float(row["fnr"]),
            "fpr": float(row["fpr"]),
            "fnr_defined": bool(positive_support > 0),
            "fpr_defined": bool(negative_support > 0),
            "low_positive_support": bool(
                positive_support < RATE_SUPPORT_DIAGNOSTIC_THRESHOLD),
            "low_negative_support": bool(
                negative_support < RATE_SUPPORT_DIAGNOSTIC_THRESHOLD),
            "single_class_period": bool(
                positive_support == 0 or negative_support == 0),
            # Toàn bộ nhãn tháng t được mở sau dự đoán để phục vụ t+1;
            # không cộng lặp toàn bộ tập lịch sử như chi phí gán nhãn mới.
            "label_count": int(row["tot"]),
            "reject_count": 0,
            "unique_expert_workload": int(row["tot"]),
            "automatic_coverage": 1.0,
            "labels_per_1000": 1000.0,
            "rejects_per_1000": 0.0,
            "expert_workload_per_1000": 1000.0,
            "selected_malware": int(row["p"]),
            "selected_benign": int(row["n"]),
            "training_samples": int(row["training_samples"]),
            "training_malware": int(row["training_malware"]),
            "training_benign": int(row["training_benign"]),
            "history_end": row["history_end"],
            "fit_seconds": float(row["fit_seconds"]),
            "n_iter": int(row["n_iter"]),
        }
        # CFJR không từ chối; phạm vi hệ thống trùng đầu ra tự động.
        for key in (
                "tp", "tn", "fp", "fn", "positive_support",
                "negative_support", "precision", "recall", "f1", "fnr",
                "fpr", "fnr_defined", "fpr_defined",
                "low_positive_support", "low_negative_support",
                "single_class_period"):
            item[f"system_{key}"] = item[key]
        cfjr_monthly.append(item)

_write_csv(TABLE_OUT / "v13_cfjr_monthly.csv", cfjr_monthly)
CFJR_EXPECTED_KEYS = {(CFJR_METHOD, int(seed)) for seed in SEEDS}
CFJR_VALID_KEYS = {
    (CFJR_METHOD, int(seed)) for seed in SEEDS
    if len(CFJR_RESULTS.get((CFJR_METHOD, int(seed)), [])) == EXPECTED_PERIODS}
CFJR_REPORT_READY = CFJR_VALID_KEYS == CFJR_EXPECTED_KEYS
cfjr_integrity = {
    "protocol_version": CFJR_PROTOCOL_VERSION,
    "method": CFJR_METHOD,
    "architecture_reference": "Static-MLP",
    "expected_seeds": SEEDS,
    "complete_seeds": sorted(seed for _, seed in CFJR_VALID_KEYS),
    "missing_complete_seeds": sorted(
        seed for _, seed in CFJR_EXPECTED_KEYS - CFJR_VALID_KEYS),
    "expected_periods_per_seed": EXPECTED_PERIODS,
    "expected_monthly_refits": CFJR_PLANNED_REFITS,
    "valid_period_artifacts": int(sum(
        len(rows) for rows in CFJR_RESULTS.values())),
    "causal_rule": "fit labels through t-1; predict t; release labels t",
    "report_ready": CFJR_REPORT_READY,
    "claim": "strong causal empirical reference; not a theoretical upper bound",
}
(TABLE_OUT / "v13_cfjr_integrity_summary.json").write_text(
    json.dumps(cfjr_integrity, indent=2, ensure_ascii=False), encoding="utf-8")

cfjr_summary = []
for seed in SEEDS:
    if (CFJR_METHOD, int(seed)) not in CFJR_VALID_KEYS:
        continue
    rows = [row for row in cfjr_monthly if row["seed"] == int(seed)]
    cfjr_summary.append(_summarize(
        CFJR_METHOD, seed, rows,
        expected_n_test_samples=EXPECTED_TEST_SAMPLES))
_write_csv(TABLE_OUT / "v13_cfjr_summary_by_seed.csv", cfjr_summary)

cfjr_overview = []
if cfjr_summary:
    record = {"method": CFJR_METHOD,
              "display_name": DISPLAY_NAMES[CFJR_METHOD],
              "n_seeds": len(cfjr_summary),
              "mean_total_refits": EXPECTED_PERIODS,
              "mean_total_fit_seconds": float(np.mean([
                  sum(row["fit_seconds"] for row in cfjr_monthly
                      if row["seed"] == summary_row["seed"])
                  for summary_row in cfjr_summary])),
              "mean_final_training_samples": float(np.mean([
                  max(row["training_samples"] for row in cfjr_monthly
                      if row["seed"] == summary_row["seed"])
                  for summary_row in cfjr_summary]))}
    for metric in ("aut_f1", "aut_fnr", "aut_fpr",
                   "pooled_f1", "pooled_fnr", "pooled_fpr"):
        values = np.asarray([row[metric] for row in cfjr_summary], dtype=float)
        finite = values[np.isfinite(values)]
        record[f"mean_{metric}_pct"] = (
            100 * float(np.mean(finite)) if finite.size else float("nan"))
        record[f"sd_{metric}_pct"] = (
            100 * float(np.std(finite, ddof=1))
            if finite.size > 1 else float("nan"))
    cfjr_overview.append(record)
_write_csv(TABLE_OUT / "v13_cfjr_overview.csv", cfjr_overview)

# Hiệu ghép cặp với MLP tĩnh, FFCR và điểm vận hành IRAAL nếu các nhánh đó có.
cfjr_summary_map = {int(row["seed"]): row for row in cfjr_summary}
cfjr_effects = []
for reference in ("Static-MLP", "DRMD-FFCR", "DRMD-IRAAL", "DRMD-FN"):
    for seed in SEEDS:
        cfjr = cfjr_summary_map.get(int(seed))
        baseline = summary_map.get((reference, int(seed)))
        if cfjr is None or baseline is None:
            continue
        static = summary_map.get(("Static-MLP", int(seed)))
        denominator = (
            cfjr["aut_f1"] - static["aut_f1"] if static is not None else float("nan"))
        recovered = (
            (baseline["aut_f1"] - static["aut_f1"]) / denominator
            if static is not None and np.isfinite(denominator)
            and denominator > 0 else float("nan"))
        cfjr_effects.append({
            "reference": reference, "seed": int(seed),
            "delta_aut_f1_cfjr_minus_reference_pp": 100 * (
                cfjr["aut_f1"] - baseline["aut_f1"]),
            "delta_aut_fnr_cfjr_minus_reference_pp": 100 * (
                cfjr["aut_fnr"] - baseline["aut_fnr"]),
            "delta_aut_fpr_cfjr_minus_reference_pp": 100 * (
                cfjr["aut_fpr"] - baseline["aut_fpr"]),
            "delta_pooled_f1_cfjr_minus_reference_pp": 100 * (
                cfjr["pooled_f1"] - baseline["pooled_f1"]),
            "fraction_static_to_cfjr_gap_recovered": recovered,
        })
_write_csv(TABLE_OUT / "v13_cfjr_reference_effects_by_seed.csv", cfjr_effects)

cfjr_effect_summary = []
for reference in ("Static-MLP", "DRMD-FFCR", "DRMD-IRAAL", "DRMD-FN"):
    rows = [row for row in cfjr_effects if row["reference"] == reference]
    if not rows:
        continue
    record = {"reference": reference, "n_paired_seeds": len(rows)}
    for metric in (
            "delta_aut_f1_cfjr_minus_reference_pp",
            "delta_aut_fnr_cfjr_minus_reference_pp",
            "delta_aut_fpr_cfjr_minus_reference_pp",
            "delta_pooled_f1_cfjr_minus_reference_pp",
            "fraction_static_to_cfjr_gap_recovered"):
        values = np.asarray([row[metric] for row in rows], dtype=float)
        finite = values[np.isfinite(values)]
        record[f"mean_{metric}"] = (
            float(np.mean(finite)) if finite.size else float("nan"))
        record[f"sd_{metric}"] = (
            float(np.std(finite, ddof=1)) if finite.size > 1 else float("nan"))
        record[f"exact_sign_flip_p_{metric}"] = _exact_sign_flip_p(finite)
    cfjr_effect_summary.append(record)
_write_csv(TABLE_OUT / "v13_cfjr_reference_effects_summary.csv",
           cfjr_effect_summary)

# Hình tham chiếu dùng trung bình ghép theo tháng; dải xám là ±1 độ lệch chuẩn.
plot_sources = {
    CFJR_METHOD: cfjr_monthly,
    "DRMD-FFCR": [row for row in monthly if row["method"] == "DRMD-FFCR"],
    "Static-MLP": [row for row in monthly if row["method"] == "Static-MLP"],
}
available_plot_methods = [
    method for method, rows in plot_sources.items() if rows]
if available_plot_methods:
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 6.6), sharex=True)
    styles = {
        CFJR_METHOD: ("-", "o", 1.8),
        "DRMD-FFCR": ("--", "s", 1.4),
        "Static-MLP": (":", "^", 1.3),
    }
    x = np.arange(len(period_months))
    for metric, axis, ylabel in (
            ("f1", axes[0], "F1 (%)"),
            ("fnr", axes[1], "FNR (%)")):
        for method in available_plot_methods:
            means, stds = [], []
            rows = plot_sources[method]
            for month in period_months:
                values = np.asarray([
                    float(row[metric]) for row in rows
                    if row["month"] == month and np.isfinite(float(row[metric]))
                ], dtype=float)
                means.append(100 * float(np.mean(values)) if values.size else np.nan)
                stds.append(100 * float(np.std(values, ddof=1))
                            if values.size > 1 else 0.0)
            means, stds = np.asarray(means), np.asarray(stds)
            linestyle, marker, linewidth = styles[method]
            axis.plot(x, means, color="black", linestyle=linestyle,
                      marker=marker, markevery=max(1, len(x) // 18),
                      markersize=3.0, markerfacecolor="white",
                      linewidth=linewidth, label=DISPLAY_NAMES[method])
            axis.fill_between(x, means - stds, means + stds,
                              color="0.80", alpha=0.16, linewidth=0)
        axis.set_ylabel(ylabel); axis.set_ylim(0, 100)
        axis.grid(axis="y", color="0.85", linewidth=0.6)
    year_ticks, year_labels, seen = [], [], set()
    for index, month in enumerate(period_months):
        year = month[:4]
        if year not in seen:
            seen.add(year); year_ticks.append(index); year_labels.append(year)
    axes[-1].set_xticks(year_ticks, year_labels, rotation=45)
    axes[-1].set_xlabel("Năm kiểm thử (chỉ các tháng có quan sát)")
    axes[0].legend(ncol=min(3, len(available_plot_methods)), frameon=False)
    fig.tight_layout()
    fig.savefig(FIG_OUT / "v13_cfjr_reference_full_timeline.png",
                dpi=220, bbox_inches="tight")
    plt.close(fig)

print({"cfjr_report_ready": CFJR_REPORT_READY,
       "cfjr_complete_seeds": len(CFJR_VALID_KEYS),
       "cfjr_expected_seeds": len(CFJR_EXPECTED_KEYS),
       "cfjr_period_artifacts": cfjr_integrity["valid_period_artifacts"],
       "cfjr_tables": str(TABLE_OUT), "cfjr_figures": str(FIG_OUT)})


## Ô 13 — Hiện vật cho báo cáo

Chỉ diễn giải đầy đủ hiệu năng và độ nhạy khi `v13_report_ready=True` và đủ 210/210 hiện vật phương pháp–seed (10 Static-MLP + 200 DRMD). CFJR có cổng độc lập `cfjr_report_ready=True` khi đủ 10 chuỗi × 110 chu kỳ; không dùng kết quả CFJR dở dang để kết luận.


## Ô 14 — Đóng gói kết quả và checkpoint V13


In [ ]:
# Đóng gói nguyên tử: bảng, hình và checkpoint V13.
import os
import zipfile

ARCHIVE_RAW_RESULTS = True

def zip_tree_atomic(zip_path, roots, compresslevel=6):
    temporary = zip_path.with_name(zip_path.name + ".tmp")
    with zipfile.ZipFile(
            temporary, "w", compression=zipfile.ZIP_DEFLATED,
            compresslevel=compresslevel) as archive:
        for root in roots:
            if not root.exists():
                continue
            paths = [root] if root.is_file() else sorted(root.rglob("*"))
            for path in paths:
                if path.is_file() and path != temporary:
                    archive.write(path, path.relative_to(OUT))
    os.replace(temporary, zip_path)
    print(f"Đã tạo {zip_path.name}: {zip_path.stat().st_size / 1024**2:.2f} MiB")

bundle_kind = ("full_test_report_bundle" if COMBINED_REPORT_READY
               else "provisional_audit_bundle")
report_bundle = OUT / f"{EXPERIMENT_NAME}_{bundle_kind}.zip"
zip_tree_atomic(report_bundle, [TABLE_OUT, FIG_OUT, RUN_STATUS_PATH,
                                RUN_PROGRESS_LOG, FAILURE_LOG, QUARANTINE_LOG])

if ARCHIVE_RAW_RESULTS:
    checkpoint_bundle = OUT / f"{EXPERIMENT_NAME}_checkpoint_raw.zip"
    zip_tree_atomic(checkpoint_bundle,
                    [STRATEGY_RAW_OUT, RAW_OUT / "static_baselines", CFJR_CACHE_DIR,
                     RUN_PROGRESS_LOG, RUN_STATUS_PATH, FAILURE_LOG],
                    compresslevel=1)


# Trên Colab, sao lưu không ghi đè nếu Google Drive đã được mount trước đó.
def _backup_colab_archives_to_mounted_drive(archives):
    if not IS_GOOGLE_COLAB or not COLAB_BACKUP_TO_MOUNTED_DRIVE:
        return []
    mounted_my_drive = Path("/content/drive/MyDrive")
    if not mounted_my_drive.is_dir():
        print(
            "ℹ️ Google Drive chưa được mount; checkpoint vẫn nằm trong "
            f"{OUT}. Mount Drive trước khi chạy nếu muốn tự động sao lưu."
        )
        return []

    import shutil

    backup_root = Path(COLAB_DRIVE_BACKUP_DIR)
    backup_root.mkdir(parents=True, exist_ok=True)
    timestamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    saved = []
    for source_path in archives:
        source_path = Path(source_path)
        if not source_path.is_file():
            continue
        destination = backup_root / (
            f"{source_path.stem}_{KAGGLE_RUN_PHASE}_{timestamp}"
            f"{source_path.suffix}"
        )
        duplicate_index = 1
        while destination.exists():
            destination = backup_root / (
                f"{source_path.stem}_{KAGGLE_RUN_PHASE}_{timestamp}_"
                f"{duplicate_index}{source_path.suffix}"
            )
            duplicate_index += 1
        temporary = destination.with_name(destination.name + ".tmp")
        try:
            shutil.copy2(source_path, temporary)
            os.replace(temporary, destination)
        except Exception as exc:
            if temporary.exists():
                temporary.unlink()
            print({"colab_drive_backup": "failed",
                   "source": str(source_path),
                   "error_type": type(exc).__name__})
            continue
        saved.append(str(destination))
    print({"colab_drive_backup": "complete", "saved": saved})
    return saved


_colab_archives = [report_bundle]
if ARCHIVE_RAW_RESULTS:
    _colab_archives.append(checkpoint_bundle)
COLAB_SAVED_ARCHIVES = _backup_colab_archives_to_mounted_drive(_colab_archives)
